<a href="https://colab.research.google.com/github/123qazplm77/MacBERT-zime/blob/main/XIME_MacBERT_Phase2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving xime_macbert_colab_ready.zip to xime_macbert_colab_ready (1).zip


In [ ]:
from pathlib import Path
import hashlib
import zipfile
import shutil

zip_path = Path("/content/xime_macbert_colab_ready.zip")
expected_sha256 = (
    "a375a52542c48a6e9fa816b32f7bb645"
    "c35de3a7d1620c4c79e6ea1bdf133924"
)

if not zip_path.exists():
    raise FileNotFoundError(
        "没有找到 ZIP。请先运行上传单元格，并上传 xime_macbert_colab_ready.zip"
    )

actual_sha256 = hashlib.sha256(zip_path.read_bytes()).hexdigest()
print("文件大小：", zip_path.stat().st_size, "字节")
print("SHA-256：", actual_sha256)
print("哈希一致：", actual_sha256 == expected_sha256)

if actual_sha256 != expected_sha256:
    raise RuntimeError(
        "ZIP 与本次归档报告中的 SHA-256 不一致，请检查是否上传了旧版本。"
    )

with zipfile.ZipFile(zip_path, "r") as zf:
    bad_file = zf.testzip()
    if bad_file:
        raise RuntimeError(f"ZIP 完整性检查失败：{bad_file}")
    print("ZIP CRC 检查通过")
    print("ZIP 条目数：", len(zf.namelist()))

work_dir = Path("/content/xime_work")
if work_dir.exists():
    shutil.rmtree(work_dir)
work_dir.mkdir(parents=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(work_dir)

project_dir = work_dir / "xime_macbert_colab_ready"
if not project_dir.is_dir():
    raise FileNotFoundError(
        f"没有找到预期目录：{project_dir}。请检查 ZIP 内部目录结构。"
    )

print("工程目录：", project_dir)
print("README 存在：", (project_dir / "README_zh.md").exists())
print("桥接脚本存在：",
      (project_dir / "scripts/bridge_shuangpin_to_text.py").exists())
print("纠错训练脚本存在：",
      (project_dir / "scripts/train_correction.py").exists())

文件大小： 109769 字节
SHA-256： a375a52542c48a6e9fa816b32f7bb645c35de3a7d1620c4c79e6ea1bdf133924
哈希一致： True
ZIP CRC 检查通过
ZIP 条目数： 57
工程目录： /content/xime_work/xime_macbert_colab_ready
README 存在： True
桥接脚本存在： True
纠错训练脚本存在： True


In [ ]:
%cd /content/xime_work/xime_macbert_colab_ready
!pwd
!ls

/content/xime_work/xime_macbert_colab_ready
/content/xime_work/xime_macbert_colab_ready
configs  docs	 logs	 README.md     requirements-colab.txt  scripts
data	 export  models  README_zh.md  requirements.txt        tests


In [11]:
!python scripts/xiaohe_codec.py selftest

[PASS] 自检通过：19 个例词的编码与解码往返一致。
       音节表：409 个音节


In [12]:
!python tests/test_xiaohe_codec.py

[PASS] 配置文件存在
[PASS] 所有键位是单个大写字母
[PASS] 声母键唯一（zh/ch/sh 各自独占）
[PASS] 韵母键可一键多韵母（如 X=ia/ua、S=ong/iong、K=ing/uai）
[PASS] 音节表已加载
[PASS] 已知例词编码正确
[PASS] 中国 = vsgo
[PASS] 双拼 = ulpb
[PASS] 小鹤 = xnhe
[PASS] 零声母规则（啊=aa 爱=ai 昂=ah en/ou/ee/eg/er）
[PASS] 全音节往返（409 个）
[PASS] 歧义键码 lo 返回全部候选
[PASS] 歧义判定为 ambiguous
[PASS] 键码 do 由音节表消歧为 duo
[PASS] 唯一键码可唯一解码
[PASS] 非法音节 'zzz' 抛错
[PASS] 非法音节 'abc' 抛错
[PASS] 非法音节 '' 抛错
[PASS] 非法音节 'xyz' 抛错
[PASS] 非法键码 'a' 抛错
[PASS] 非法键码 'abc' 抛错
[PASS] 非法键码 'z1' 抛错
[PASS] 非法键码 '' 抛错
[PASS] 爱 的等价写法为 ad
[PASS] 默认不把 ad 解为爱
[PASS] 显式开启后 ad -> ai
[PASS] xiaohe_codec.py 代码部分不引用邻键表
[PASS] 邻键表仍是 26 条且未被双拼映射污染

[PASS] 小鹤双拼测试全部通过。


In [16]:
!python tests/test_shuangpin_errors.py

[FAIL] 缺少数据文件 /content/xime_work/xime_macbert_colab_ready/tests/_out/shuangpin_errors.jsonl，请先运行：
  python scripts/generate_shuangpin_errors.py --out tests/_out/shuangpin_errors.jsonl --force


In [17]:
!python scripts/generate_shuangpin_errors.py \
  --out tests/_out/shuangpin_errors.jsonl \
  --force

已写出：/content/xime_work/xime_macbert_colab_ready/tests/_out/shuangpin_errors.jsonl（3143 条）
统计  ：/content/xime_work/xime_macbert_colab_ready/tests/_out/shuangpin_errors.stats.json
  错误类型：{'zero_initial_lead': 43, 'final_letter': 37, 'final_key': 1571, 'initial_key': 1492}
  错误键码可解码 66.78%；IME 歧义 0.25%
  恢复唯一 1.02%；仅凭键码无法唯一恢复 98.98%（不同错误键码 671 个）
  最难样例：chan(ij) --I->J--> jj，可能来自 ['chan', 'han', 'ji', 'jiao', 'jing', 'ju', 'kan', 'nan', 'shan']


In [18]:
!python tests/test_shuangpin_errors.py

记录数：3143；音节数：409
[PASS] 结构：只在 error_pos 处不同
[PASS] 方向：只使用 neighbors.json 的正向邻键
[PASS] 分类：error_type 与音节结构一致
[PASS] 标记：decodable/ambiguous 与候选数一致
[PASS] 标记：recovery_ambiguity 与反向分组一致
[PASS] 完整性：音节表全部覆盖
[PASS] 反向关系未被自动补全（不存在 A->Z）
[PASS] 邻键表仍与项目约定完全一致

[PASS] 双拼误触数据生成测试全部通过。


In [19]:
!python tests/test_correction_alignment.py

[PASS] 等长序列可对齐
[PASS] 错字位置识别正确
[PASS] 长度不一致被拒绝
[PASS] 长度不一致时 build_example 返回 None
[PASS] 标签只出现在错字位置
[PASS] 错字位置标签 = clean 的字
[PASS] 错字位置输入被替换为 [MASK]
[PASS] [CLS]/[SEP] 不参与损失
[PASS] mask_typo_prob=0 时仍以 clean 为标签
[PASS] mask_typo_prob=0 时输入保留噪声 token
[PASS] 正确位置可被抽查参与监督
[PASS] 抽查位置的标签 = 原字（正确字）
[PASS] 标签中无特殊 token
[PASS] 同随机种子结果可复现
[PASS] padding 后等长
[PASS] attention_mask 正确
[PASS] 标签 padding 为 -100
[PASS] 超长截断到 max_length
[PASS] 截断后 attention_mask 同步
[PASS] 无错字样本不产生无标签样本
[PASS] 无错字样本仍有'保持原文'标签

[PASS] 纠错标签对齐测试全部通过。


In [20]:
!python tests/test_bridge_shuangpin_to_text.py

[PASS] 主用例退出码为 0
[PASS] 四个输出文件都生成了
[PASS] 接受样本集合符合预期
[PASS] TSV 只有两列
[PASS] 每条样本等长且只有一处不同
[PASS] 元数据里的 replace_pos 就是差异位置
[PASS] 元数据标明 noisy 来源
[PASS] 唯一候选：比的 candidate_count=1
[PASS] 多个候选：我色的 candidate_count=2
[PASS] 零声母音节可桥接（爱→气）
[PASS] 零候选 → undecodable_wrong_code
[PASS] 非法键码 → invalid_code
[PASS] 非法音节 → invalid_syllable
[PASS] 码音不自洽 → code_mismatch
[PASS] 损坏/缺字段记录 → invalid_record
[PASS] 歧义记录被单独标记
[PASS] 歧义样本没有进入默认训练集
[PASS] 桥接保持了原始邻键方向（未对称化）
[PASS] 同种子重复运行结果一致
[PASS] train_correction.read_pairs 能读取输出
[PASS] --allow-ambiguous 时歧义样本才出现
[PASS] 歧义样本的元数据带有标记
[PASS] 无词库时生成单字样本
[PASS] 空输入被明确拒绝（退出码 2）
[PASS] 全是损坏记录时退出码为 1 且不产生伪样本
[PASS] 输出已存在时拒绝覆盖（退出码 2）

[PASS] 桥接测试全部通过（仅验证代码行为，不代表模型效果）。


In [21]:
%pip -q install pypinyin

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 21.4 MB/s eta 0:00:00


In [22]:
import pypinyin
print("pypinyin 安装成功")
print("示例：", pypinyin.lazy_pinyin("我们今天去公园"))

pypinyin 安装成功
示例： ['wo', 'men', 'jin', 'tian', 'qu', 'gong', 'yuan']


In [23]:
!python scripts/fetch_corpus.py \
  --source hf-wikipedia \
  --max-sentences 60000 \
  --out data/raw/wiki_zh.txt

正在流式读取 hf:wikimedia/wikipedia/20231101.zh#train ...
README.md: 100% 131k/131k [00:00<00:00, 190MB/s]
已写出 60000 句 -> /content/xime_work/xime_macbert_colab_ready/data/raw/wiki_zh.txt
溯源信息 -> /content/xime_work/xime_macbert_colab_ready/data/raw/wiki_zh.provenance.json
'[Errno 9] Bad file descriptor' thrown while requesting GET https://huggingface.co/datasets/wikimedia/wikipedia/resolve/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/20231101.zh/train-00000-of-00006.parquet
Retrying in 1s [Retry 1/5].
^C


In [24]:
from pathlib import Path
import json

root = Path("/content/xime_work/xime_macbert_colab_ready")
corpus = root / "data/raw/wiki_zh.txt"
meta = root / "data/raw/wiki_zh.provenance.json"

if not corpus.exists():
    print("❌ 语料文件不存在，需要重新下载")
else:
    print("文件大小：", round(corpus.stat().st_size / 1024 / 1024, 2), "MB")

    count = 0
    with corpus.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                count += 1

    print("非空句子数：", count)

    with corpus.open("r", encoding="utf-8") as f:
        print("\n语料示例：")
        for _, line in zip(range(3), f):
            print(line.strip()[:200])

    if count >= 60000:
        print("\n✅ 已达到 60000 句目标")
    else:
        print("\n⚠️ 语料数量不足，需要检查下载过程")

if meta.exists():
    print("\n溯源文件：")
    print(meta.read_text(encoding="utf-8")[:1500])

文件大小： 5.85 MB
非空句子数： 60000

语料示例：
数学，是研究數量、结构以及空间等概念及其变化的一門学科，屬於形式科學的一種
數學利用抽象化和邏輯推理，從計數、計算、量度、對物體形狀及運動的觀察發展而成
數學家們拓展這些概念，以公式化新的猜想，以及從選定的公理及定義出發，嚴謹地推導出一些定理

✅ 已达到 60000 句目标

溯源文件：
{
  "created_at": "2026-10-17T14:27:49",
  "source": "hf:wikimedia/wikipedia/20231101.zh#train",
  "license": "CC BY-SA 4.0（以数据集卡片为准，需自行核实）",
  "records_scanned": 417,
  "sentences_written": 60000,
  "out": "/content/xime_work/xime_macbert_colab_ready/data/raw/wiki_zh.txt",
  "sha256": "182b09268efbaba9c92c8e1ba9d791f023dd9e618d847fca19e7a6326d15aa44",
  "filters": {
    "min_cjk": 8,
    "max_chars": 80,
    "cjk_ratio_min": 0.5,
    "dedup": "全文件去重"
  },
  "notice": "本文件只做下载与清洗；使用/分发请遵守原数据集许可证。"
}


In [25]:
from pathlib import Path
from collections import Counter
import re

p = Path("data/raw/wiki_zh.txt")

total = 0
empty = 0
short = 0
long = 0
traditional_hint = 0
unique = set()
lengths = []

with p.open("r", encoding="utf-8") as f:
    for line in f:
        s = line.strip()

        if not s:
            empty += 1
            continue

        total += 1
        unique.add(s)
        lengths.append(len(s))

        if len(s) < 8:
            short += 1
        if len(s) > 80:
            long += 1

        if re.search(r"[數學門屬於體國會發展學]", s):
            traditional_hint += 1

print("非空句子数：", total)
print("不同句子数：", len(unique))
print("少于8字符：", short)
print("超过80字符：", long)
print("疑似含繁体字的句子数：", traditional_hint)
print("平均字符数：", round(sum(lengths) / max(len(lengths), 1), 2))

非空句子数： 60000
不同句子数： 60000
少于8字符： 0
超过80字符： 0
疑似含繁体字的句子数： 17399
平均字符数： 35.19


In [26]:
%pip -q install opencc-python-reimplemented

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.8/481.8 kB 14.7 MB/s eta 0:00:00


In [27]:
from pathlib import Path
from opencc import OpenCC
import hashlib
import json

src = Path("data/raw/wiki_zh.txt")
dst = Path("data/raw/wiki_zh_simplified.txt")
cc = OpenCC("t2s")

seen = set()
written = 0
duplicates = 0

dst.parent.mkdir(parents=True, exist_ok=True)

with src.open("r", encoding="utf-8") as fin, \
     dst.open("w", encoding="utf-8") as fout:

    for line in fin:
        text = cc.convert(line.strip())

        if not text:
            continue

        if text in seen:
            duplicates += 1
            continue

        seen.add(text)
        fout.write(text + "\n")
        written += 1

sha256 = hashlib.sha256(dst.read_bytes()).hexdigest()

print("原始句子数：", sum(
    1 for line in src.open("r", encoding="utf-8")
    if line.strip()
))
print("简体语料句子数：", written)
print("转换后重复并删除：", duplicates)
print("文件大小：", round(dst.stat().st_size / 1024 / 1024, 2), "MB")
print("SHA-256：", sha256)
print("\n转换示例：")

with dst.open("r", encoding="utf-8") as f:
    for _, line in zip(range(3), f):
        print(line.strip())

原始句子数： 60000
简体语料句子数： 59910
转换后重复并删除： 90
文件大小： 5.84 MB
SHA-256： eea3271c3fa60c206458eea8989eda5c5ef944db3f0048b95f51134a4095e8fc

转换示例：
数学，是研究数量、结构以及空间等概念及其变化的一门学科，属于形式科学的一种
数学利用抽象化和逻辑推理，从计数、计算、量度、对物体形状及运动的观察发展而成
数学家们拓展这些概念，以公式化新的猜想，以及从选定的公理及定义出发，严谨地推导出一些定理


In [28]:
!python scripts/build_pinyin_inventory.py \
  --input data/raw/wiki_zh_simplified.txt \
  --out configs/pinyin_inventory.json

语料：/content/xime_work/xime_macbert_colab_ready/data/raw/wiki_zh_simplified.txt
不同汉字：5308
已写出：/content/xime_work/xime_macbert_colab_ready/configs/pinyin_inventory.json
收录汉字：5307；读音：409
同音字最多的读音：yi(120)，ji(105)，yu(99)，qi(85)，zhi(83)


In [29]:
import json
from pathlib import Path

p = Path("configs/pinyin_inventory.json")

print("候选表存在：", p.exists())

if p.exists():
    data = json.loads(p.read_text(encoding="utf-8"))
    print("顶层字段：", list(data.keys()))

    by_syllable = data.get("by_syllable", {})
    print("候选音节数：", len(by_syllable))

    for syllable, chars in list(by_syllable.items())[:15]:
        print(syllable, "->", chars[:10])

候选表存在： True
顶层字段： ['meta', 'chars', 'by_syllable', 'syllables']
候选音节数： 409
de -> ['的', '地', '德', '得', '底', '登', '陟', '锝']
di -> ['的', '地', '第', '提', '帝', '逐', '低', '底', '隶', '弟']
guo -> ['国', '过', '活', '划', '果', '郭', '锅', '裹', '涡', '虢']
zhong -> ['中', '重', '种', '众', '终', '忠', '童', '钟', '董', '仲']
nian -> ['年', '念', '趁', '廿', '黏', '粘', '辗', '捻', '碾', '辇']
ning -> ['年', '宁', '疑', '冰', '泥', '凝', '柠', '攘', '佞', '泞']
wei -> ['为', '有', '于', '立', '位', '机', '维', '委', '遗', '未']
zai -> ['在', '再', '才', '载', '灾', '宰', '仔', '栽', '哉', '甾']
yi -> ['一', '以', '也', '台', '义', '已', '意', '议', '它', '医']
shi -> ['是', '时', '市', '世', '使', '斯', '事', '式', '十', '史']
ti -> ['是', '体', '提', '达', '题', '弟', '替', '折', '梯', '狄']
ren -> ['人', '认', '任', '儿', '仁', '忍', '韧', '壬', '饪', '刃']
da -> ['大', '达', '打', '塔', '答', '搭', '胆', '迭', '鞑', '靼']
dai -> ['大', '代', '带', '毒', '隶', '待', '戴', '傣', '逮', '贷']
tai -> ['大', '台', '能', '太', '态', '泰', '胎', '苔', '汰', '肽']


In [30]:
import json
from pathlib import Path
from pypinyin import pinyin, Style

path = Path("configs/pinyin_inventory.json")
data = json.loads(path.read_text(encoding="utf-8"))
by_syllable = data["by_syllable"]

# 每个汉字的所有常见读音
def get_readings(ch):
    result = pinyin(
        ch,
        style=Style.NORMAL,
        heteronym=True,
        errors=lambda x: []
    )
    if not result:
        return set()
    return {x.lower() for x in result[0]}

checked = 0
mismatches = []
empty_candidates = []

for syllable, chars in by_syllable.items():
    if not chars:
        empty_candidates.append(syllable)
        continue

    for ch in chars:
        readings = get_readings(ch)
        checked += 1

        if syllable not in readings:
            mismatches.append((syllable, ch, sorted(readings)))

print("候选表读音数：", len(by_syllable))
print("核对候选项数：", checked)
print("空候选读音数：", len(empty_candidates))
print("疑似读音不匹配项数：", len(mismatches))

print("\n疑似错误示例（最多30项）：")
for syllable, ch, readings in mismatches[:30]:
    print(f"{syllable} -> {ch}，pypinyin读音：{readings}")

print("\n空候选读音：", empty_candidates[:30])

候选表读音数： 409
核对候选项数： 7259
空候选读音数： 0
疑似读音不匹配项数： 0

疑似错误示例（最多30项）：

空候选读音： []


In [31]:
from pathlib import Path
import json

script = Path("scripts/build_pinyin_inventory.py")
print("=" * 25, "候选表构建脚本", "=" * 25)
print(script.read_text(encoding="utf-8"))

p = Path("configs/pinyin_inventory.json")
data = json.loads(p.read_text(encoding="utf-8"))

print("\n", "=" * 20, "候选表统计", "=" * 20)
print("元数据：")
print(json.dumps(data.get("meta", {}), ensure_ascii=False, indent=2))

by_syllable = data["by_syllable"]
counts = sorted(
    [(syllable, len(chars)) for syllable, chars in by_syllable.items()],
    key=lambda x: x[1],
    reverse=True
)

print("\n候选数量最多的 20 个音节：")
for syllable, count in counts[:20]:
    print(f"{syllable}: {count} 个")

print("\n候选数量最少的 20 个音节：")
for syllable, count in sorted(counts, key=lambda x: x[1])[:20]:
    print(f"{syllable}: {count} 个")

========================= 候选表构建脚本 =========================
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""从语料构建拼音表 configs/pinyin_inventory.json（供邻键纠错数据生成使用）。

为什么需要单独一步：
    "把 pinyin 拼写里某个字母换成键盘邻键" 这件事必须先知道「每个汉字怎么读」和
    「某个拼写对应哪些汉字」。这两张表由 pypinyin 在 Colab 上一次性算好并落盘，
    后续的数据生成与数据检查就只依赖这份 JSON，不再依赖 pypinyin，
    因此可以在没有 pypinyin 的机器上复现检查过程。

依赖：pypinyin（pip install pypinyin）。本机若没有 pypinyin，脚本会给出明确提示并退出。

用法：
    python scripts/build_pinyin_inventory.py \
        --input data/raw/wiki_zh.txt \
        --out configs/pinyin_inventory.json

输出结构：
    {
      "meta": {...},
      "chars": {"中": ["zhong"], ...},              # 汉字 -> 去重后的读音（不含声调）
      "by_syllable": {"zhong": ["中", "钟", ...]},   # 读音 -> 汉字（按语料词频降序）
      "syllables": ["zhong", ...]                   # 语料中实际出现过的读音
    }
"""

from __future__ import annotations

import argparse
import collections
import datetime as _dt
import json
import re
import sys
from pathlib import Path

CJK_RE = re.compile(r"[\u4e00-\u9fff]")
# pyp

In [32]:
import json
from pathlib import Path

# 读取候选表
data = json.loads(
    Path("configs/pinyin_inventory.json").read_text(encoding="utf-8")
)
inventory_syllables = set(data["by_syllable"].keys())

# 读取项目原始音节表
syllable_file = Path("configs/pinyin_syllables.txt")
codec_syllables = {
    line.strip().lower()
    for line in syllable_file.read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.strip().startswith("#")
}

print("候选表音节数：", len(inventory_syllables))
print("项目音节表音节数：", len(codec_syllables))

only_inventory = sorted(inventory_syllables - codec_syllables)
only_codec = sorted(codec_syllables - inventory_syllables)

print("\n候选表有、项目音节表没有的音节：", len(only_inventory))
print(only_inventory[:100])

print("\n项目音节表有、候选表没有的音节：", len(only_codec))
print(only_codec[:100])

# 检查几个特殊条目
for s in ["m", "n", "ng", "hng", "lo", "zhei", "dei"]:
    print(
        f"{s}: 候选表={s in inventory_syllables}, "
        f"项目音节表={s in codec_syllables}"
    )

候选表音节数： 409
项目音节表音节数： 409

候选表有、项目音节表没有的音节： 5
['hng', 'm', 'n', 'ng', 'yi']

项目音节表有、候选表没有的音节： 5
['den', 'eng', 'nou', 'o', 'rua']
m: 候选表=True, 项目音节表=False
n: 候选表=True, 项目音节表=False
ng: 候选表=True, 项目音节表=False
hng: 候选表=True, 项目音节表=False
lo: 候选表=True, 项目音节表=True
zhei: 候选表=True, 项目音节表=True
dei: 候选表=True, 项目音节表=True


In [33]:

import json
from pathlib import Path

root = Path.cwd()

inventory = json.loads(
    (root / "configs/pinyin_inventory.json").read_text(encoding="utf-8")
)
config_path = root / "configs/xiaohe_shuangpin.json"
config = json.loads(config_path.read_text(encoding="utf-8"))

inventory_syllables = set(inventory["by_syllable"])
codec_syllables = {
    line.strip().lower()
    for line in (root / "configs/pinyin_syllables.txt")
    .read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.strip().startswith("#")
}

only_inventory = sorted(inventory_syllables - codec_syllables)
only_codec = sorted(codec_syllables - inventory_syllables)

print("=== 差异音节的候选汉字 ===")
for s in only_inventory:
    print(f"{s}: {inventory['by_syllable'].get(s, [])[:20]}")

print("\n=== 项目独有音节 ===")
for s in only_codec:
    print(f"{s}: 候选表中存在={s in inventory_syllables}")

print("\n=== 双拼配置文件结构 ===")
print("顶层字段：", list(config.keys()))
for key, value in config.items():
    if isinstance(value, dict):
        print(f"{key}: {len(value)} 项，示例={list(value.items())[:5]}")
    elif isinstance(value, list):
        print(f"{key}: {len(value)} 项，示例={value[:10]}")
    else:
        print(f"{key}: {value}")

=== 差异音节的候选汉字 ===
hng: ['哼']
m: ['呒']
n: ['嗯']
ng: ['嗯']
yi: ['一', '以', '也', '台', '义', '已', '意', '议', '它', '医', '易', '亦', '遗', '艺', '失', '伊', '移', '依', '印', '施']

=== 项目独有音节 ===
den: 候选表中存在=False
eng: 候选表中存在=False
nou: 候选表中存在=False
o: 候选表中存在=False
rua: 候选表中存在=False

=== 双拼配置文件结构 ===
顶层字段： ['meta', 'initials', 'finals', 'zero_initial', 'syllable_file', 'neighbors_note']
meta: 12 项，示例=[('name', '小鹤双拼'), ('scheme_id', 'double_pinyin_flypy'), ('version', '0.1'), ('purpose', 'XIME 相位一：为「小鹤双拼输入」建立可校验的编码/解码表，供误触数据生成使用。'), ('source_primary', 'Rime 官方方案 rime-double-pinyin / double_pinyin_flypy.schema.yaml（小鶴雙拼 version 0.18，layout by 鶴）')]
initials: 23 项，示例=[('zh', 'V'), ('ch', 'I'), ('sh', 'U'), ('b', 'B'), ('p', 'P')]
finals: 33 项，示例=[('a', 'A'), ('o', 'O'), ('e', 'E'), ('i', 'I'), ('u', 'U')]
zero_initial: 5 项，示例=[('description', '零声母音节（a/o/e/ai/ei/ao/ou/an/en/ang/eng/er）用「首字母 + 韵母」占满两键。'), ('single_letter_finals_doubled', ['a', 'o', 'e']), ('two_letter_finals_spelled_out', ['ai', 'ei', 'ao',

In [34]:

from pathlib import Path

p = Path("scripts/bridge_shuangpin_to_text.py")
source = p.read_text(encoding="utf-8")
lines = source.splitlines()

print("=== 命令行参数 ===")
for i, line in enumerate(lines):
    if "add_argument(" in line or "ArgumentParser(" in line:
        print(f"{i + 1}: {line.strip()}")

print("\n=== 音节表 / 候选表 / 输出相关代码 ===")
keywords = (
    "pinyin_inventory", "by_syllable", "syllable",
    "candidate", "output", "accepted", "undecodable",
    "invalid_syllable", "write_text", "open("
)
shown = set()

for i, line in enumerate(lines):
    if any(k in line.lower() for k in keywords):
        start = max(0, i - 2)
        end = min(len(lines), i + 3)
        for j in range(start, end):
            if j not in shown:
                print(f"{j + 1:4}: {lines[j]}")
                shown.add(j)
        print("---")

print("\n检查完成：只读取脚本，没有修改文件。")

=== 命令行参数 ===
179: ap = argparse.ArgumentParser(description="双拼键码误触 → 汉字级 clean/noisy 样本（可审计）")
180: ap.add_argument("--errors", required=True, help="第一阶段 shuangpin_errors.jsonl")
181: ap.add_argument("--candidates", required=True, help="音节→汉字候选表（json 或 tsv）")
182: ap.add_argument("--lexicon", default=None, help="可选：词库（音节序列<TAB>词）")
183: ap.add_argument("--out-tsv", default="data/processed/shuangpin_train_pairs.tsv")
184: ap.add_argument("--out-jsonl", default=None, help="接受样本的元数据，默认与 tsv 同名 .jsonl")
185: ap.add_argument("--out-rejected", default=None, help="被过滤记录，默认与 tsv 同名 .rejected.jsonl")
186: ap.add_argument("--report", default=None, help="统计报告，默认与 tsv 同名 .report.json")
187: ap.add_argument("--strategy", choices=["first", "seeded"], default="first",
189: ap.add_argument("--seed", type=int, default=42)
190: ap.add_argument("--max-samples", type=int, default=0, help="接受样本上限，0=不限")
191: ap.add_argument("--max-words-per-record", type=int, default=3,
193: ap.add_argument("--max-chars",

In [35]:
from pathlib import Path
import json
import hashlib

root = Path.cwd()

checks = [
    "configs/xiaohe_shuangpin.json",
    "configs/pinyin_syllables.txt",
    "configs/neighbors.json",
    "configs/pinyin_inventory.json",
    "tests/_out/shuangpin_errors.jsonl",
    "scripts/bridge_shuangpin_to_text.py",
    "scripts/train_correction.py",
]

print("=== 输入文件检查 ===")
for name in checks:
    p = root / name
    print(f"{'存在' if p.is_file() else '缺失'}：{name}")
    if p.is_file():
        print(f"  大小：{p.stat().st_size:,} 字节")

print("\n=== 误触数据统计 ===")
errors = root / "tests/_out/shuangpin_errors.jsonl"
if errors.is_file():
    count = sum(1 for line in errors.open(encoding="utf-8") if line.strip())
    digest = hashlib.sha256(errors.read_bytes()).hexdigest()
    print("记录数：", count)
    print("SHA-256：", digest)

print("\n=== 目标输出是否已存在 ===")
outputs = [
    "data/processed/shuangpin_train_pairs.tsv",
    "data/processed/shuangpin_train_pairs.jsonl",
    "data/processed/shuangpin_train_pairs.rejected.jsonl",
    "data/processed/shuangpin_train_pairs.report.json",
]
for name in outputs:
    print(f"{'已存在' if (root / name).exists() else '不存在'}：{name}")

print("\n检查结束：只读，没有修改文件。")

=== 输入文件检查 ===
存在：configs/xiaohe_shuangpin.json
  大小：3,122 字节
存在：configs/pinyin_syllables.txt
  大小：2,110 字节
存在：configs/neighbors.json
  大小：386 字节
存在：configs/pinyin_inventory.json
  大小：235,499 字节
存在：tests/_out/shuangpin_errors.jsonl
  大小：1,281,587 字节
存在：scripts/bridge_shuangpin_to_text.py
  大小：23,908 字节
存在：scripts/train_correction.py
  大小：11,737 字节

=== 误触数据统计 ===
记录数： 3143
SHA-256： 834387971bd396728a329ae34279b53fd341c198a8fa7f03959985071dee3204

=== 目标输出是否已存在 ===
不存在：data/processed/shuangpin_train_pairs.tsv
不存在：data/processed/shuangpin_train_pairs.jsonl
不存在：data/processed/shuangpin_train_pairs.rejected.jsonl
不存在：data/processed/shuangpin_train_pairs.report.json

检查结束：只读，没有修改文件。


In [36]:
!python scripts/bridge_shuangpin_to_text.py \
  --errors tests/_out/shuangpin_errors.jsonl \
  --candidates configs/pinyin_inventory.json \
  --out-tsv data/processed/shuangpin_train_pairs_v1.tsv \
  --out-jsonl data/processed/shuangpin_train_pairs_v1.jsonl \
  --out-rejected data/processed/shuangpin_train_pairs_v1.rejected.jsonl \
  --report data/processed/shuangpin_train_pairs_v1.report.json \
  --strategy seeded \
  --seed 42

[INFO] 未提供 --lexicon：只生成单字样本（无上下文，仅适合验证管线）
候选表：pinyin_inventory.json 格式=pinyin_inventory(json):by_syllable，409 个音节 / 7259 个汉字
读取 3143 条误触记录；接受 2048 条（词级 0，字级 2051），去重移除 3，过滤 1092
过滤原因：ambiguous_wrong_code=8，no_candidates=40，undecodable_wrong_code=1044
训练表：/content/xime_work/xime_macbert_colab_ready/data/processed/shuangpin_train_pairs_v1.tsv
元数据：/content/xime_work/xime_macbert_colab_ready/data/processed/shuangpin_train_pairs_v1.jsonl
过滤表：/content/xime_work/xime_macbert_colab_ready/data/processed/shuangpin_train_pairs_v1.rejected.jsonl
报告  ：/content/xime_work/xime_macbert_colab_ready/data/processed/shuangpin_train_pairs_v1.report.json
[PASS] 桥接完成：所有接受样本均通过等长/单点差异/键码自洽校验。


In [37]:
from pathlib import Path
import json
import hashlib
import statistics

root = Path.cwd()
base = root / "data/processed/shuangpin_train_pairs_v1"

tsv_path = Path(str(base) + ".tsv")
jsonl_path = Path(str(base) + ".jsonl")
rejected_path = Path(str(base) + ".rejected.jsonl")
report_path = Path(str(base) + ".report.json")

for p in [tsv_path, jsonl_path, rejected_path, report_path]:
    print(f"{'存在' if p.is_file() else '缺失'}：{p.name}")

if not all(p.is_file() for p in [
    tsv_path, jsonl_path, rejected_path, report_path
]):
    raise FileNotFoundError("输出文件不完整，请先检查第 36 步的运行结果。")

report = json.loads(report_path.read_text(encoding="utf-8"))
pairs = []

with tsv_path.open("r", encoding="utf-8") as f:
    header = f.readline().rstrip("\n")
    for line_no, line in enumerate(f, start=2):
        cols = line.rstrip("\n").split("\t")
        if len(cols) != 2:
            raise ValueError(f"TSV 第 {line_no} 行不是两列")
        pairs.append(tuple(cols))

metadata = [
    json.loads(line)
    for line in jsonl_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

rejected_count = sum(
    1 for line in rejected_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
)

print("\n=== 数据量 ===")
print("TSV 表头：", header)
print("TSV 样本数：", len(pairs))
print("JSONL 元数据数：", len(metadata))
print("过滤记录数：", rejected_count)
print("报告中的接受数：", report.get("accepted"))
print("去重前接受数：", report.get("accepted_before_dedupe"))
print("去重移除数：", report.get("duplicates_removed"))
print("字级样本数：", report.get("char_level"))
print("词级样本数：", report.get("word_level"))
print("按错误类型统计：", report.get("accepted_by_error_type"))
print("过滤原因：", report.get("rejected_by_reason"))

print("\n=== 样本完整性 ===")
print("TSV 与 JSONL 数量一致：", len(pairs) == len(metadata))
print("clean/noisy 长度一致：", all(len(a) == len(b) for a, b in pairs))
print("每条样本恰好一处不同：",
      all(sum(x != y for x, y in zip(a, b)) == 1 for a, b in pairs))
print("clean/noisy 对重复数：", len(pairs) - len(set(pairs)))

print("\n=== 样本示例（最多 20 条）===")
for clean, noisy in pairs[:20]:
    print(f"{clean} -> {noisy}")

print("\n=== 文件 SHA-256 ===")
for p in [tsv_path, jsonl_path, rejected_path, report_path]:
    print(p.name, hashlib.sha256(p.read_bytes()).hexdigest())

存在：shuangpin_train_pairs_v1.tsv
存在：shuangpin_train_pairs_v1.jsonl
存在：shuangpin_train_pairs_v1.rejected.jsonl
存在：shuangpin_train_pairs_v1.report.json

=== 数据量 ===
TSV 表头： clean	noisy
TSV 样本数： 2048
JSONL 元数据数： 2048
过滤记录数： 1092
报告中的接受数： 2048
去重前接受数： 2051
去重移除数： 3
字级样本数： 2051
词级样本数： 0
按错误类型统计： {'zero_initial_lead': 23, 'final_letter': 4, 'initial_key': 928, 'final_key': 1093}
过滤原因： {'undecodable_wrong_code': 1044, 'no_candidates': 40, 'ambiguous_wrong_code': 8}

=== 样本完整性 ===
TSV 与 JSONL 数量一致： True
clean/noisy 长度一致： True
每条样本恰好一处不同： True
clean/noisy 对重复数： 0

=== 样本示例（最多 20 条）===
阿 -> 蛙
阿 -> 萨
爱 -> 气
爱 -> 嗣
爱 -> 鿫
广 -> 樵
仰 -> 皇
仰 -> 嗓
奥 -> 窝
奥 -> 挲
奥 -> 奇
巴 -> 喋
巴 -> 呵
巴 -> 南
巴 -> 被
百 -> 择
百 -> 氦
百 -> 奶
百 -> 畚
版 -> 绽

=== 文件 SHA-256 ===
shuangpin_train_pairs_v1.tsv 48205a06293b637da9e4319ae4561c8b632d144cd3f8e843499078c2e83b2933
shuangpin_train_pairs_v1.jsonl 6228484279f8761aa70d79c0fe6e3614d9435971ac2af6a244f0f84070849f16
shuangpin_train_pairs_v1.rejected.jsonl 91bb235ded55b2e08c5b67ea8d3b

In [38]:
from pathlib import Path
from collections import Counter
import json

corpus_path = Path("data/raw/wiki_zh_simplified.txt")
tsv_path = Path("data/processed/shuangpin_train_pairs_v1.tsv")

# 统计简体语料中的汉字出现次数
char_freq = Counter()

with corpus_path.open("r", encoding="utf-8") as f:
    for line in f:
        char_freq.update(
            ch for ch in line.strip()
            if "\u4e00" <= ch <= "\u9fff"
        )

pairs = []
with tsv_path.open("r", encoding="utf-8") as f:
    header = f.readline().rstrip("\n")
    for line in f:
        cols = line.rstrip("\n").split("\t")
        if len(cols) == 2:
            pairs.append(tuple(cols))

clean_freq = Counter()
noisy_freq = Counter()

for clean, noisy in pairs:
    for ch in clean:
        clean_freq[ch] += 1
    for ch in noisy:
        noisy_freq[ch] += 1

# 按样本条数统计：替换后的错字在原始语料中出现多少次
noisy_corpus_freq = Counter()
for clean, noisy in pairs:
    for ch in noisy:
        noisy_corpus_freq[char_freq[ch]] += 1

total = len(pairs)

print("=== 基本信息 ===")
print("语料汉字种类数：", len(char_freq))
print("训练样本数：", total)
print("clean 中的不同汉字数：", len(clean_freq))
print("noisy 中的不同汉字数：", len(noisy_freq))

print("\n=== noisy 字的语料频率 ===")
for threshold in [0, 1, 5, 20, 100]:
    count = sum(
        1 for _, noisy in pairs
        if any(char_freq[ch] <= threshold for ch in noisy)
    )
    print(f"至少含一个语料频次 <= {threshold} 的 noisy 样本："
          f"{count} / {total} ({count / max(total, 1):.1%})")

print("\n=== 最低频的 30 个 noisy 字 ===")
for ch, sample_count in sorted(
    noisy_freq.items(),
    key=lambda item: (char_freq[item[0]], item[0])
)[:30]:
    print(
        f"{ch}：语料频次={char_freq[ch]}，"
        f"出现在 noisy 样本中={sample_count} 次"
    )

print("\n=== noisy 字中出现最多的 30 个字 ===")
for ch, sample_count in noisy_freq.most_common(30):
    print(
        f"{ch}：语料频次={char_freq[ch]}，"
        f"出现在 noisy 样本中={sample_count} 次"
    )

print("\n审计完成：只读，没有修改任何文件。")

=== 基本信息 ===
语料汉字种类数： 5308
训练样本数： 2048
clean 中的不同汉字数： 333
noisy 中的不同汉字数： 1473

=== noisy 字的语料频率 ===
至少含一个语料频次 <= 0 的 noisy 样本：0 / 2048 (0.0%)
至少含一个语料频次 <= 1 的 noisy 样本：270 / 2048 (13.2%)
至少含一个语料频次 <= 5 的 noisy 样本：638 / 2048 (31.2%)
至少含一个语料频次 <= 20 的 noisy 样本：984 / 2048 (48.0%)
至少含一个语料频次 <= 100 的 noisy 样本：1398 / 2048 (68.3%)

=== 最低频的 30 个 noisy 字 ===
丨：语料频次=1，出现在 noisy 样本中=2 次
亁：语料频次=1，出现在 noisy 样本中=1 次
佻：语料频次=1，出现在 noisy 样本中=1 次
偌：语料频次=1，出现在 noisy 样本中=3 次
兢：语料频次=1，出现在 noisy 样本中=1 次
冂：语料频次=1，出现在 noisy 样本中=1 次
刖：语料频次=1，出现在 noisy 样本中=1 次
剐：语料频次=1，出现在 noisy 样本中=1 次
勺：语料频次=1，出现在 noisy 样本中=5 次
叩：语料频次=1，出现在 noisy 样本中=1 次
吭：语料频次=1，出现在 noisy 样本中=2 次
咂：语料频次=1，出现在 noisy 样本中=1 次
咆：语料频次=1，出现在 noisy 样本中=1 次
咚：语料频次=1，出现在 noisy 样本中=1 次
咛：语料频次=1，出现在 noisy 样本中=1 次
哄：语料频次=1，出现在 noisy 样本中=1 次
哆：语料频次=1，出现在 noisy 样本中=2 次
哟：语料频次=1，出现在 noisy 样本中=1 次
哼：语料频次=1，出现在 noisy 样本中=1 次
唃：语料频次=1，出现在 noisy 样本中=1 次
唠：语料频次=1，出现在 noisy 样本中=1 次
啻：语料频次=1，出现在 noisy 样本中=1 次
喋：语料频次=1，出现在 noisy 样本中=3 次
嗔：语料频次=1，出现在 noisy 样本中=1 次

In [39]:

from pathlib import Path
import ast

bridge_path = Path("scripts/bridge_shuangpin_to_text.py")
codec_path = Path("scripts/xiaohe_codec.py")

assert bridge_path.exists(), f"找不到文件：{bridge_path}"
assert codec_path.exists(), f"找不到文件：{codec_path}"

bridge = bridge_path.read_text(encoding="utf-8")
codec = codec_path.read_text(encoding="utf-8")

def show_context(source, keyword, before=5, after=24):
    lines = source.splitlines()
    hits = [i for i, line in enumerate(lines) if keyword in line]
    print(f"\n{'=' * 20} {keyword} {'=' * 20}")
    if not hits:
        print("未找到")
        return
    for i in hits:
        start = max(0, i - before)
        end = min(len(lines), i + after)
        print(f"\n--- 第 {i + 1} 行附近 ---")
        for j in range(start, end):
            print(f"{j + 1:4}: {lines[j]}")

# 1. 查看桥接脚本如何读取词库
show_context(bridge, "def load_lexicon")
show_context(bridge, "lexicon")

# 2. 查看双拼编码器有哪些公开函数和类
print("\n========== 编码器接口 ==========")
tree = ast.parse(codec)
for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
        print(f"第 {node.lineno} 行：{type(node).__name__} {node.name}")
    elif isinstance(node, ast.Assign):
        names = [
            t.id for t in node.targets
            if isinstance(t, ast.Name)
        ]
        if names:
            print(f"第 {node.lineno} 行：变量 {', '.join(names)}")

# 3. 检查当前词库与原始语料是否存在
print("\n========== 输入文件检查 ==========")
for name in [
    "data/raw/wiki_zh_simplified.txt",
    "configs/pinyin_inventory.json",
    "configs/xiaohe_shuangpin.json",
    "configs/pinyin_syllables.txt",
]:
    p = Path(name)
    print(f"{name}: {'存在' if p.exists() else '不存在'}")


==================== def load_lexicon ====================

--- 第 143 行附近 ---
 138:     meta["entries"] = sum(len(v) for v in cleaned.values())
 139:     meta["syllables"] = len(cleaned)
 140:     return cleaned, meta
 141: 
 142: 
 143: def load_lexicon(path: Path) -> tuple[list[dict], dict]:
 144:     """最小词库格式：可选注释行；`音节序列<TAB>汉字词[<TAB>来源]`。"""
 145:     entries: list[dict] = []
 146:     stats = {"lines": 0, "accepted": 0, "bad_columns": 0, "length_mismatch": 0}
 147:     for lineno, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
 148:         raw = line.rstrip()
 149:         if not raw.strip() or raw.lstrip().startswith("#"):
 150:             continue
 151:         stats["lines"] += 1
 152:         parts = raw.split("\t")
 153:         if len(parts) < 2:
 154:             stats["bad_columns"] += 1
 155:             continue
 156:         sylls = parts[0].split()
 157:         word = parts[1].strip()
 158:         source = parts[2].strip() if len(parts) > 2 

In [40]:

from pathlib import Path

p = Path("scripts/xiaohe_codec.py")
lines = p.read_text(encoding="utf-8").splitlines()

# 打印 XiaoheCodec 类及 load() 函数的实现
for start, end in [(50, 165), (175, 195)]:
    print(f"\n{'=' * 18} 第 {start}-{end} 行 {'=' * 18}")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1:4}: {lines[i]}")

# 确认拼音库可用
try:
    import pypinyin
    print("\npypinyin 版本：", pypinyin.__version__)
except Exception as e:
    print("\npypinyin 检查失败：", repr(e))


================== 第 50-165 行 ==================
  50:     s = s.replace("ü", "v").replace("u:", "v").replace("v:", "v")
  51:     s = re.sub(r"[0-9]", "", s)          # 去掉声调数字（zhong1 -> zhong）
  52:     s = re.sub(r"\s+", "", s)
  53:     return s
  54: 
  55: 
  56: class XiaoheCodec:
  57:     def __init__(self, config: dict, syllables: set[str] | None = None) -> None:
  58:         self.meta = config.get("meta", {})
  59:         self.initials: dict[str, str] = {k.lower(): v.upper() for k, v in config["initials"].items()}
  60:         self.finals: dict[str, str] = {k.lower(): v.upper() for k, v in config["finals"].items()}
  61:         z = config.get("zero_initial", {})
  62:         self.zero_single: list[str] = [s.lower() for s in z.get("single_letter_finals_doubled", [])]
  63:         self.zero_spelled: list[str] = [s.lower() for s in z.get("two_letter_finals_spelled_out", [])]
  64:         self.zero_nasal: list[str] = [s.lower() for s in z.get("nasal_finals_first_letter_pl

In [41]:

%%writefile scripts/build_corpus_lexicon.py
from __future__ import annotations

import argparse
import hashlib
import json
import re
import sys
from collections import Counter
from datetime import datetime
from pathlib import Path

from pypinyin import Style, lazy_pinyin

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import xiaohe_codec as xc


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def is_cjk(ch: str) -> bool:
    return "\u4e00" <= ch <= "\u9fff"


def main() -> int:
    ap = argparse.ArgumentParser(
        description="从中文语料生成经过拼音与小鹤双拼校验的候选词库"
    )
    ap.add_argument(
        "--input", default="data/raw/wiki_zh_simplified.txt"
    )
    ap.add_argument(
        "--inventory", default="configs/pinyin_inventory.json"
    )
    ap.add_argument(
        "--out", default="data/processed/wiki_lexicon_v1.tsv"
    )
    ap.add_argument(
        "--report", default="data/processed/wiki_lexicon_v1.report.json"
    )
    ap.add_argument("--min-freq", type=int, default=3)
    ap.add_argument("--min-len", type=int, default=2)
    ap.add_argument("--max-len", type=int, default=4)
    ap.add_argument("--max-entries", type=int, default=100000)
    args = ap.parse_args()

    src = Path(args.input)
    inv_path = Path(args.inventory)
    out = Path(args.out)
    report_path = Path(args.report)

    for p in (src, inv_path):
        if not p.is_file():
            print(f"[FAIL] 输入文件不存在：{p}")
            return 2

    # 防止覆盖之前的结果；需要重做时请使用新文件名。
    for p in (out, report_path):
        if p.exists():
            print(f"[STOP] 输出文件已存在，为保护旧结果而停止：{p}")
            print("请指定新的 --out 和 --report 文件名。")
            return 2

    if args.min_freq < 1 or args.min_len < 2:
        print("[FAIL] min-freq 必须 >= 1，min-len 必须 >= 2")
        return 2
    if args.max_len < args.min_len or args.max_len > 8:
        print("[FAIL] max-len 必须 >= min-len 且 <= 8")
        return 2
    if args.max_entries < 1:
        print("[FAIL] max-entries 必须 >= 1")
        return 2

    out.parent.mkdir(parents=True, exist_ok=True)
    report_path.parent.mkdir(parents=True, exist_ok=True)

    inventory = json.loads(inv_path.read_text(encoding="utf-8"))
    char_readings = inventory.get("chars", {})
    if not char_readings:
        print("[FAIL] 拼音候选表缺少 chars 字段，停止构建")
        return 2

    codec = xc.load()
    print(f"[INFO] 语料：{src}")
    print(f"[INFO] 最低出现次数：{args.min_freq}")
    print(f"[INFO] 词长：{args.min_len}–{args.max_len} 字")
    print("[INFO] 正在统计连续汉字片段……")

    freq = Counter()
    lines_read = 0
    runs_read = 0
    chars_read = 0

    with src.open("r", encoding="utf-8") as f:
        for line in f:
            lines_read += 1
            run = []

            def consume_run():
                nonlocal runs_read, chars_read
                if not run:
                    return
                word_text = "".join(run)
                runs_read += 1
                chars_read += len(word_text)

                # 只统计连续汉字片段里的 2–max_len 字 n-gram。
                for n in range(args.min_len, args.max_len + 1):
                    if len(word_text) < n:
                        continue
                    freq.update(
                        word_text[i:i+n]
                        for i in range(len(word_text) - n + 1)
                    )

            for ch in line.strip():
                if is_cjk(ch):
                    run.append(ch)
                else:
                    consume_run()
                    run.clear()
            consume_run()

            if lines_read % 10000 == 0:
                print(
                    f"[进度] 已读 {lines_read} 行，"
                    f"不同候选片段 {len(freq)}"
                )

    print(f"[INFO] 语料行数：{lines_read}")
    print(f"[INFO] 连续汉字段数量：{runs_read}")
    print(f"[INFO] 不同候选片段：{len(freq)}")

    reasons = Counter()
    valid = []
    examined = 0

    # 先按语料出现次数排序；同频时按词语排序，确保结果可复现。
    candidates = sorted(
        (
            (word, count)
            for word, count in freq.items()
            if count >= args.min_freq
        ),
        key=lambda item: (-item[1], item[0]),
    )
    print(f"[INFO] 达到频次门槛的候选：{len(candidates)}")

    for word, count in candidates:
        examined += 1

        if not all(is_cjk(ch) for ch in word):
            reasons["non_cjk"] += 1
            continue

        # 利用词语上下文取得拼音，而非逐字单独猜读音。
        try:
            syllables = [
                s.lower()
                for s in lazy_pinyin(
                    word,
                    style=Style.NORMAL,
                    strict=False,
                )
            ]
        except Exception:
            reasons["pinyin_error"] += 1
            continue

        if len(syllables) != len(word):
            reasons["length_mismatch"] += 1
            continue

        bad = False
        for ch, syl in zip(word, syllables):
            # 双拼编码器必须接受该音节。
            if not syl or not codec.is_valid(syl):
                reasons["invalid_codec_syllable"] += 1
                bad = True
                break

            # 当前字的候选读音表必须包含该音节。
            readings = char_readings.get(ch, [])
            if syl not in readings:
                reasons["inventory_mismatch"] += 1
                bad = True
                break

        if bad:
            continue

        # 记录音节序列和语料中的出现次数，供后续审计。
        valid.append((word, syllables, count))

    valid.sort(key=lambda item: (-item[2], item[0]))
    total_valid = len(valid)
    selected = valid[:args.max_entries]

    # 使用桥接脚本支持的 TSV 格式。
    with out.open("x", encoding="utf-8", newline="") as f:
        f.write("# 来源：简体中文维基语料；频次为连续汉字片段出现次数\n")
        f.write("# 注意：n-gram 不等于经过人工确认的词语\n")
        for word, syllables, count in selected:
            f.write(
                f"{' '.join(syllables)}\t{word}\t"
                f"wiki_ngram_freq={count}\n"
            )

    report = {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "input": str(src),
        "input_sha256": sha256(src),
        "inventory": str(inv_path),
        "inventory_sha256": sha256(inv_path),
        "pypinyin_version": __import__("pypinyin").__version__,
        "config": {
            "min_freq": args.min_freq,
            "min_len": args.min_len,
            "max_len": args.max_len,
            "max_entries": args.max_entries,
        },
        "lines_read": lines_read,
        "continuous_han_runs": runs_read,
        "han_chars_in_runs": chars_read,
        "unique_ngrams": len(freq),
        "ngrams_meeting_frequency": len(candidates),
        "candidates_examined": examined,
        "valid_before_limit": total_valid,
        "written_entries": len(selected),
        "truncated_by_limit": max(0, total_valid - len(selected)),
        "filtered_by_reason": dict(reasons),
        "top_entries": [
            {"word": w, "syllables": s, "corpus_count": c}
            for w, s, c in selected[:50]
        ],
        "output": str(out),
        "output_sha256": sha256(out),
    }

    with report_path.open("x", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
        f.write("\n")

    print("\n[PASS] 候选词库构建完成")
    print(f"达到频次门槛：{len(candidates)}")
    print(f"通过拼音和双拼校验：{total_valid}")
    print(f"实际写入：{len(selected)}")
    print(f"校验过滤统计：{dict(reasons)}")
    print(f"词库文件：{out}")
    print(f"审计报告：{report_path}")
    print(f"词库 SHA-256：{sha256(out)}")
    print("\n[提示] 这是候选 n-gram 词库，不是人工确认的标准词典。")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing scripts/build_corpus_lexicon.py


In [42]:

!python scripts/build_corpus_lexicon.py \
  --input data/raw/wiki_zh_simplified.txt \
  --inventory configs/pinyin_inventory.json \
  --out data/processed/wiki_lexicon_v1.tsv \
  --report data/processed/wiki_lexicon_v1.report.json \
  --min-freq 3 \
  --min-len 2 \
  --max-len 4 \
  --max-entries 100000

[INFO] 语料：data/raw/wiki_zh_simplified.txt
[INFO] 最低出现次数：3
[INFO] 词长：2–4 字
[INFO] 正在统计连续汉字片段……
[进度] 已读 10000 行，不同候选片段 432592
[进度] 已读 20000 行，不同候选片段 775827
[进度] 已读 30000 行，不同候选片段 1108834
[进度] 已读 40000 行，不同候选片段 1404146
[进度] 已读 50000 行，不同候选片段 1719644
[INFO] 语料行数：59910
[INFO] 连续汉字段数量：225111
[INFO] 不同候选片段：2012533
[INFO] 达到频次门槛的候选：206244

[PASS] 候选词库构建完成
达到频次门槛：206244
通过拼音和双拼校验：192351
实际写入：100000
校验过滤统计：{'invalid_codec_syllable': 13893}
词库文件：data/processed/wiki_lexicon_v1.tsv
审计报告：data/processed/wiki_lexicon_v1.report.json
词库 SHA-256：01e296bd70963e7cf80957a2567c28859dca32bf9173dd27e744b74c8c3c130e

[提示] 这是候选 n-gram 词库，不是人工确认的标准词典。


In [43]:
from pathlib import Path
from collections import Counter
import hashlib
import json

lex_path = Path("data/processed/wiki_lexicon_v1.tsv")
report_path = Path("data/processed/wiki_lexicon_v1.report.json")

assert lex_path.exists(), f"词库不存在：{lex_path}"
assert report_path.exists(), f"审计报告不存在：{report_path}"

raw = lex_path.read_bytes()
print("文件大小：", len(raw), "bytes")
print("SHA-256：", hashlib.sha256(raw).hexdigest())

lines = raw.decode("utf-8").splitlines()
word_counts = Counter()
length_counts = Counter()
syllable_counts = Counter()
source_counts = Counter()

bad_format = []
bad_length = []
duplicate_pairs = 0
seen_pairs = set()
examples = []

for line_no, line in enumerate(lines, 1):
    parts = line.split("\t")

    if len(parts) != 3:
        bad_format.append((line_no, line[:100]))
        continue

    syllables_text, word, source = parts
    syllables = syllables_text.split()

    if len(syllables) != len(word):
        bad_length.append((line_no, syllables_text, word))
        continue

    pair = (syllables_text, word)
    if pair in seen_pairs:
        duplicate_pairs += 1
    seen_pairs.add(pair)

    word_counts[word] += 1
    length_counts[len(word)] += 1
    syllable_counts[len(syllables)] += 1
    source_counts[source.split("=")[0]] += 1

    if len(examples) < 20:
        examples.append((syllables_text, word, source))

report = json.loads(report_path.read_text(encoding="utf-8"))

print("\n===== 词库完整性审计 =====")
print("总行数：", len(lines))
print("格式错误：", len(bad_format))
print("音节数与汉字数不一致：", len(bad_length))
print("重复的音节序列+词语记录：", duplicate_pairs)
print("不同词语数：", len(word_counts))
print("词长分布：", dict(sorted(length_counts.items())))
print("音节数分布：", dict(sorted(syllable_counts.items())))
print("来源标记分布：", dict(source_counts))

print("\n===== 前 20 条记录 =====")
for item in examples:
    print("\t".join(item))

print("\n===== 报告中的关键字段 =====")
for key, value in report.items():
    if isinstance(value, (str, int, float, bool)) or value is None:
        print(f"{key}: {value}")

print("\n===== 审计结论 =====")
if not bad_format and not bad_length and duplicate_pairs == 0:
    print("PASS：基础格式、音节数量和重复记录检查通过。")
else:
    print("注意：存在需要进一步调查的问题。")
    print("格式错误示例：", bad_format[:5])
    print("长度错误示例：", bad_length[:5])

文件大小： 3802823 bytes
SHA-256： 01e296bd70963e7cf80957a2567c28859dca32bf9173dd27e744b74c8c3c130e

===== 词库完整性审计 =====
总行数： 100002
格式错误： 2
音节数与汉字数不一致： 0
重复的音节序列+词语记录： 0
不同词语数： 100000
词长分布： {2: 49482, 3: 36746, 4: 13772}
音节数分布： {2: 49482, 3: 36746, 4: 13772}
来源标记分布： {'wiki_ngram_freq': 100000}

===== 前 20 条记录 =====
zhong guo	中国	wiki_ngram_freq=5336
ren min	人民	wiki_ngram_freq=2253
guo jia	国家	wiki_ngram_freq=2179
zhong hua	中华	wiki_ngram_freq=2162
di qu	地区	wiki_ngram_freq=2137
zheng fu	政府	wiki_ngram_freq=2094
shi jie	世界	wiki_ngram_freq=2020
yan jiu	研究	wiki_ngram_freq=1889
jing ji	经济	wiki_ngram_freq=1874
zhu yao	主要	wiki_ngram_freq=1844
wen hua	文化	wiki_ngram_freq=1804
tai wan	台湾	wiki_ngram_freq=1744
fa zhan	发展	wiki_ngram_freq=1739
shi yong	使用	wiki_ngram_freq=1693
mei guo	美国	wiki_ngram_freq=1641
cheng wei	成为	wiki_ngram_freq=1612
xi tong	系统	wiki_ngram_freq=1515
kai shi	开始	wiki_ngram_freq=1488
she hui	社会	wiki_ngram_freq=1486
gong he	共和	wiki_ngram_freq=1430

===== 报告中的关键字段 =====
created_at: 2026-10-

In [44]:
from pathlib import Path

path = Path("scripts/bridge_shuangpin_to_text.py")
lines = path.read_text(encoding="utf-8").splitlines()

keywords = ("def load_lexicon", "wiki_ngram_freq", "split('\\t')", 'split("\\t")')

for i, line in enumerate(lines):
    if any(k in line for k in keywords):
        start = max(0, i - 5)
        end = min(len(lines), i + 24)
        print(f"\n===== 第 {start + 1}–{end} 行 =====")
        for j in range(start, end):
            print(f"{j + 1:4}: {lines[j]}")


===== 第 138–166 行 =====
 138:     meta["entries"] = sum(len(v) for v in cleaned.values())
 139:     meta["syllables"] = len(cleaned)
 140:     return cleaned, meta
 141: 
 142: 
 143: def load_lexicon(path: Path) -> tuple[list[dict], dict]:
 144:     """最小词库格式：可选注释行；`音节序列<TAB>汉字词[<TAB>来源]`。"""
 145:     entries: list[dict] = []
 146:     stats = {"lines": 0, "accepted": 0, "bad_columns": 0, "length_mismatch": 0}
 147:     for lineno, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
 148:         raw = line.rstrip()
 149:         if not raw.strip() or raw.lstrip().startswith("#"):
 150:             continue
 151:         stats["lines"] += 1
 152:         parts = raw.split("\t")
 153:         if len(parts) < 2:
 154:             stats["bad_columns"] += 1
 155:             continue
 156:         sylls = parts[0].split()
 157:         word = parts[1].strip()
 158:         source = parts[2].strip() if len(parts) > 2 else path.name
 159:         if len(sylls) != len(word

In [45]:
from pathlib import Path

targets = [
    Path("data/processed/shuangpin_train_pairs_v2.tsv"),
    Path("data/processed/shuangpin_train_pairs_v2.jsonl"),
    Path("data/processed/shuangpin_train_pairs_v2.rejected.jsonl"),
    Path("data/processed/shuangpin_train_pairs_v2.report.json"),
]

existing = [str(p) for p in targets if p.exists()]

if existing:
    print("为避免覆盖，停止运行。以下文件已存在：")
    print("\n".join(existing))
else:
    print("PASS：v2 输出文件均不存在，可以安全生成。")

PASS：v2 输出文件均不存在，可以安全生成。


In [46]:
!python scripts/bridge_shuangpin_to_text.py \
  --errors tests/_out/shuangpin_errors.jsonl \
  --candidates configs/pinyin_inventory.json \
  --lexicon data/processed/wiki_lexicon_v1.tsv \
  --out-tsv data/processed/shuangpin_train_pairs_v2.tsv \
  --out-jsonl data/processed/shuangpin_train_pairs_v2.jsonl \
  --out-rejected data/processed/shuangpin_train_pairs_v2.rejected.jsonl \
  --report data/processed/shuangpin_train_pairs_v2.report.json \
  --strategy seeded \
  --seed 42 \
  --max-words-per-record 3

词库：wiki_lexicon_v1.tsv，有效条目 100000（列数不足 0，音节数≠汉字数 0）
候选表：pinyin_inventory.json 格式=pinyin_inventory(json):by_syllable，409 个音节 / 7259 个汉字
读取 3143 条误触记录；接受 5924 条（词级 5843，字级 90），去重移除 9，过滤 1091
过滤原因：ambiguous_wrong_code=8，no_candidates=39，undecodable_wrong_code=1044
训练表：/content/xime_work/xime_macbert_colab_ready/data/processed/shuangpin_train_pairs_v2.tsv
元数据：/content/xime_work/xime_macbert_colab_ready/data/processed/shuangpin_train_pairs_v2.jsonl
过滤表：/content/xime_work/xime_macbert_colab_ready/data/processed/shuangpin_train_pairs_v2.rejected.jsonl
报告  ：/content/xime_work/xime_macbert_colab_ready/data/processed/shuangpin_train_pairs_v2.report.json
[PASS] 桥接完成：所有接受样本均通过等长/单点差异/键码自洽校验。


In [47]:
from pathlib import Path
from collections import Counter
import hashlib
import json

base = Path("data/processed")
v1_path = base / "shuangpin_train_pairs_v1.tsv"
v2_path = base / "shuangpin_train_pairs_v2.tsv"
jsonl_path = base / "shuangpin_train_pairs_v2.jsonl"
rejected_path = base / "shuangpin_train_pairs_v2.rejected.jsonl"
report_path = base / "shuangpin_train_pairs_v2.report.json"

for p in [v2_path, jsonl_path, rejected_path, report_path]:
    assert p.exists(), f"文件缺失：{p}"

# 1. 检查 TSV
raw = v2_path.read_bytes()
lines = raw.decode("utf-8").splitlines()

assert lines and lines[0] == "clean\tnoisy", "TSV 表头异常"

pairs = []
bad_columns = []
bad_length = []
bad_diff = []

for line_no, line in enumerate(lines[1:], 2):
    parts = line.split("\t")
    if len(parts) != 2:
        bad_columns.append((line_no, line[:100]))
        continue

    clean, noisy = parts
    if len(clean) != len(noisy):
        bad_length.append((line_no, clean, noisy))
        continue

    diff_positions = [
        i for i, (a, b) in enumerate(zip(clean, noisy)) if a != b
    ]
    if len(diff_positions) != 1:
        bad_diff.append((line_no, clean, noisy, diff_positions))

    pairs.append((clean, noisy))

# 2. 检查 JSONL 与重复记录
with jsonl_path.open(encoding="utf-8") as f:
    metadata = [json.loads(line) for line in f if line.strip()]

duplicates = len(pairs) - len(set(pairs))

# 3. 统计词长及字符数量
length_counts = Counter(len(clean) for clean, noisy in pairs)
clean_chars = {ch for clean, _ in pairs for ch in clean}
noisy_chars = {ch for _, noisy in pairs for ch in noisy}

print("===== v2 文件完整性 =====")
print("TSV 样本数：", len(pairs))
print("JSONL 元数据数：", len(metadata))
print("两者数量一致：", len(pairs) == len(metadata))
print("TSV 格式错误：", len(bad_columns))
print("长度不一致：", len(bad_length))
print("不是单字符差异：", len(bad_diff))
print("重复 clean/noisy 对：", duplicates)
print("clean 不同字符数：", len(clean_chars))
print("noisy 不同字符数：", len(noisy_chars))
print("样本长度分布：", dict(sorted(length_counts.items())))
print("TSV SHA-256：", hashlib.sha256(raw).hexdigest())

# 4. 查看词级样本
word_pairs = [(c, n) for c, n in pairs if len(c) > 1]
print("\n===== 词级样本示例（最多 30 条） =====")
for clean, noisy in word_pairs[:30]:
    print(f"{noisy} -> {clean}")

# 5. 查看报告中的关键统计
report = json.loads(report_path.read_text(encoding="utf-8"))
print("\n===== v2 报告 =====")
for key, value in report.items():
    if isinstance(value, (str, int, float, bool)) or value is None:
        print(f"{key}: {value}")

# 6. 对比 v1
if v1_path.exists():
    v1_lines = v1_path.read_text(encoding="utf-8").splitlines()
    v1_count = max(0, len(v1_lines) - 1)
    print("\n===== v1 / v2 对比 =====")
    print("v1 TSV 样本数：", v1_count)
    print("v2 TSV 样本数：", len(pairs))
    print("v2 相比 v1 增加：", len(pairs) - v1_count)

print("\n===== 初步结论 =====")
if (
    len(pairs) == len(metadata)
    and not bad_columns
    and not bad_length
    and not bad_diff
    and duplicates == 0
):
    print("PASS：v2 结构检查通过。")
else:
    print("注意：至少有一项检查未通过，需要调查。")
    print("格式错误示例：", bad_columns[:5])
    print("长度错误示例：", bad_length[:5])
    print("差异错误示例：", bad_diff[:5])

===== v2 文件完整性 =====
TSV 样本数： 5924
JSONL 元数据数： 5924
两者数量一致： True
TSV 格式错误： 0
长度不一致： 0
不是单字符差异： 0
重复 clean/noisy 对： 0
clean 不同字符数： 896
noisy 不同字符数： 3049
样本长度分布： {1: 90, 2: 5284, 3: 489, 4: 61}
TSV SHA-256： 15d5d7c28ce4bfa4ccc5b8a5560a7b4be99d4dc69293700be1361b371f668f73

===== 词级样本示例（最多 30 条） =====
蛙尔 -> 阿尔
挖尔巴 -> 阿尔巴
瓦尔巴尼 -> 阿尔巴尼
飒尔 -> 阿尔
蔡尔巴 -> 阿尔巴
撒尔巴尼 -> 阿尔巴尼
揭滋 -> 艾滋
抵滋病 -> 艾滋病
切及 -> 埃及
司滋 -> 艾滋
雉滋病 -> 艾滋病
斯及 -> 埃及
奥滋 -> 艾滋
敖滋病 -> 艾滋病
嚣及 -> 埃及
樵全 -> 安全
橇徽 -> 安徽
长桥 -> 长安
魍格 -> 盎格
望贵 -> 昂贵
魍格鲁 -> 盎格鲁
桑格 -> 盎格
丧贵 -> 昂贵
桑格鲁 -> 盎格鲁
李卧 -> 李敖
我门 -> 澳门
挝运 -> 奥运
李唆 -> 李敖
琐门 -> 澳门
锁运 -> 奥运

===== v2 报告 =====
created_at: 2026-10-17T15:36:51
records_read: 3143
bad_json_lines: 0
accepted_before_dedupe: 5933
accepted: 5924
duplicates_removed: 9
word_level: 5843
char_level: 90
ambiguous_excluded: 8
max_chars_filtered: 0
rejected: 1091
candidate_count_mean: 18.1
candidate_count_max: 105

===== v1 / v2 对比 =====
v1 TSV 样本数： 2048
v2 TSV 样本数： 5924
v2 相比 v1 增加： 3876

===== 初步结论 =====
PASS：v2 结构检查通过。


In [48]:
!python scripts/train_correction.py --help

usage: train_correction.py [-h] [--base BASE] --train-file TRAIN_FILE
                           --valid-file VALID_FILE [--output-dir OUTPUT_DIR]
                           [--epochs EPOCHS] [--batch-size BATCH_SIZE]
                           [--grad-accum GRAD_ACCUM] [--lr LR]
                           [--warmup-ratio WARMUP_RATIO]
                           [--max-length MAX_LENGTH]
                           [--max-train-samples MAX_TRAIN_SAMPLES]
                           [--mask-typo-prob MASK_TYPO_PROB]
                           [--mask-correct-prob MASK_CORRECT_PROB]
                           [--seed SEED] [--fp16] [--preflight-only]

中文纠错训练（clean 作为标签，固定 MacBERT MLM 头）

options:
  -h, --help            show this help message and exit
  --base BASE
  --train-file TRAIN_FILE
  --valid-file VALID_FILE
  --output-dir OUTPUT_DIR
  --epochs EPOCHS
  --batch-size BATCH_SIZE
  --grad-accum GRAD_ACCUM
  --lr LR
  --warmup-ratio WARMUP_RATIO
  --max-length MAX_LENGTH
  --max-train-

In [49]:
!sed -n '1,280p' scripts/train_correction.py

#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""中文纠错训练（对齐标签监督），**不是**普通掩码语言模型预训练。

与工程自带的 scripts/train.py 的区别（原文件保持不动）：
    train.py : 只编码 noisy 列，labels 就是 noisy 自己 -> 学的是"噪声文本的分布"，
               对纠错没有监督信号（详见 docs/PROJECT_STATUS.md §4）。
    本脚本   : input_ids = noisy（错字位置按概率替换为 [MASK]）
               labels    = **clean** 对应位置的字，其余位置 -100。
               因此梯度方向是"把这个错字改成正确的字"，并可同时监督
               "本来就正确的位置不要乱改"（--mask-correct-prob），用于压低误纠正率。

边界与诚实说明：
    * 这是"已知错字位置"的监督。真实输入时错字位置未知，还需要检测环节；
      本脚本不做检测，不能当作端到端产品。
    * 支持试训练（--max-train-samples / --epochs 1）与正式训练。
    * 本机无 torch/transformers，**本脚本尚未实际运行**，标记为待验证。

用法：
    # 小规模试训练
    python scripts/train_correction.py --train-file data/processed/train.tsv \
        --valid-file data/processed/valid.tsv --output-dir models/smoke \
        --max-train-samples 200 --epochs 1 --batch-size 8 --fp16

    # 正式训练
    python scripts/train_correction.py --train-file data/processed/train.tsv \
        --valid-file data/processed/valid.tsv

In [50]:
!sed -n '1,280p' scripts/correction_labels.py

#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""纠错监督的标签构造（纯 Python，不依赖 torch / transformers）。

这是 `scripts/train_correction.py` 的核心逻辑，单独拆出来是为了**能在没有 GPU、
没有 torch 的机器上用假 token 序列做单元测试**，从而真正验证：
    * clean / noisy 的 token 对齐
    * 特殊 token（[CLS]/[SEP]/[PAD]）不参与损失
    * 长度不一致时的处理
    * 标签只在错字位置取 clean，其余为 -100
    * 正确位置参与"保持原文"监督（降低误纠正率）

损失口径：
    labels[i] = clean_ids[i]  当 i 属于被监督的位置（错字位置 / 被抽查的正确位置）
    labels[i] = -100          其余位置（含 [CLS]、[SEP]、[PAD]）

用法（供测试与调试）：
    python scripts/correction_labels.py --demo
"""

from __future__ import annotations

import random
import sys

IGNORE_INDEX = -100


def align_pair(clean_ids: list[int], noisy_ids: list[int]) -> dict:
    """检查两个序列能否逐位对齐；不能时报出原因。"""
    if len(clean_ids) != len(noisy_ids):
        return {"ok": False, "reason": f"长度不一致：clean={len(clean_ids)} noisy={len(noisy_ids)}",
                "typo_positions": []}
    typo = [i for i, (c, n) in enumerate(zip(clean_ids, noisy_ids)) if c != n]
    return {"ok": True, "reason": 

In [51]:
!sed -n '1,300p' scripts/eval_dual.py

#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""双层评估：① 双拼键码恢复  ② 中文纠错（模型）。

模式 shuangpin（纯标准库，本机可跑）：
    读 scripts/generate_shuangpin_errors.py 产出的 JSONL，统计
        - 错误键码在 IME 中是否可解码（decodable）
        - IME 歧义率（同一键码对应多个音节）
        - 仅凭键码能否唯一恢复目标音节（recovery_unique）
        - 按错误类型的分组统计
    结论用于判断"是否需要上下文模型"，不涉及 torch。

模式 correction（需要 torch/transformers，标记为待验证）：
    对测试集逐句：错字位置全部替换为 [MASK]，并随机抽查一部分"本来就正确"的位置也替换为
    [MASK]，用模型还原，报告：
        - 纠正准确率（错字位置 top-1 / top-3 命中正确字）
        - 错误位置指标（把"改动了的位置"当作检测结果，与真实错字位置比 P/R/F1）
        - 误纠正率（本来正确的位置被改坏的比例）
        - 保持率（本来正确的位置保持不变的比例）
    这是"已知候选位置集合"的评估，不是端到端产品指标。

用法：
    python scripts/eval_dual.py --mode shuangpin --errors data/generated/shuangpin_errors.jsonl
    python scripts/eval_dual.py --mode correction --model models/macbert-typo \
        --test-file data/processed/test.tsv --out logs/correction_eval.json
"""

from __future__ import annotations

import argparse
import csv
import json
import sys
from collections import defau

In [52]:
ids = tok.convert_tokens_to_ids(list(noisy))

NameError: name 'tok' is not defined

In [53]:
from pathlib import Path
from collections import defaultdict
import csv
import random
import hashlib

source = Path("data/processed/shuangpin_train_pairs_v2.tsv")
out_dir = Path("data/processed")

outputs = {
    "train": out_dir / "train_v1.tsv",
    "valid": out_dir / "valid_v1.tsv",
    "test": out_dir / "test_v1.tsv",
}

assert source.exists(), f"找不到源文件：{source}"

existing = [str(p) for p in outputs.values() if p.exists()]
if existing:
    raise FileExistsError(
        "为避免覆盖，停止运行。以下输出文件已存在：\n" +
        "\n".join(existing)
    )

# 读取并验证源数据
groups = defaultdict(list)

with source.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f, delimiter="\t")
    header = next(reader, None)
    assert header and header[:2] == ["clean", "noisy"], (
        f"表头异常：{header}"
    )

    for line_no, row in enumerate(reader, 2):
        if len(row) != 2 or not row[0] or not row[1]:
            raise ValueError(f"第 {line_no} 行格式异常：{row}")

        clean, noisy = row
        if len(clean) != len(noisy):
            raise ValueError(f"第 {line_no} 行长度不一致：{row}")

        groups[clean].append((clean, noisy))

# 按 clean 分组随机划分
clean_keys = list(groups)
rng = random.Random(42)
rng.shuffle(clean_keys)

n = len(clean_keys)
n_train = int(n * 0.8)
n_valid = int(n * 0.1)

train_keys = set(clean_keys[:n_train])
valid_keys = set(clean_keys[n_train:n_train + n_valid])
test_keys = set(clean_keys[n_train + n_valid:])

splits = {
    "train": train_keys,
    "valid": valid_keys,
    "test": test_keys,
}

# 写入独立文件
for name, keys in splits.items():
    path = outputs[name]
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f, delimiter="\t", lineterminator="\n")
        writer.writerow(["clean", "noisy"])

        for clean in sorted(keys):
            for pair in groups[clean]:
                writer.writerow(pair)

# 验证 clean 分组没有跨集合泄漏
assert train_keys.isdisjoint(valid_keys)
assert train_keys.isdisjoint(test_keys)
assert valid_keys.isdisjoint(test_keys)
assert train_keys | valid_keys | test_keys == set(groups)

print("===== 分组划分结果 =====")
print("源样本数：", sum(len(v) for v in groups.values()))
print("不同 clean 数：", len(groups))

for name, keys in splits.items():
    path = outputs[name]
    with path.open("r", encoding="utf-8", newline="") as f:
        row_count = sum(1 for _ in csv.reader(f, delimiter="\t")) - 1

    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(
        f"{name}: clean 组数={len(keys)}, "
        f"样本数={row_count}, "
        f"SHA-256={digest}"
    )

print("\n训练集与验证集 clean 重叠：", len(train_keys & valid_keys))
print("训练集与测试集 clean 重叠：", len(train_keys & test_keys))
print("验证集与测试集 clean 重叠：", len(valid_keys & test_keys))
print("[PASS] 分组划分完成，clean 没有跨集合泄漏。")

===== 分组划分结果 =====
源样本数： 5924
不同 clean 数： 984
train: clean 组数=787, 样本数=4691, SHA-256=a2e681fca3a57a571d2fe0ceac775253359b318ebc3d1ee1bd48e4001d9262c3
valid: clean 组数=98, 样本数=583, SHA-256=4911b84fcb0011ee13ccfaff83189cc87689c9e0144233d317af397388e4bb6a
test: clean 组数=99, 样本数=650, SHA-256=e4c603302bca9e1bae4d1173c34d64919fde94cfad2a03190f01adc530d60b77

训练集与验证集 clean 重叠： 0
训练集与测试集 clean 重叠： 0
验证集与测试集 clean 重叠： 0
[PASS] 分组划分完成，clean 没有跨集合泄漏。


In [54]:
import transformers
print("Transformers 版本：", transformers.__version__)

Transformers 版本： 5.18.0


In [55]:
!python scripts/train_correction.py \
  --base hfl/chinese-macbert-base \
  --train-file data/processed/train_v1.tsv \
  --valid-file data/processed/valid_v1.tsv \
  --output-dir models/macbert-typo \
  --preflight-only

中文纠错训练（对齐标签监督）：input=noisy，label=clean
这不是普通 MLM：损失的标签来自 clean 列，而非输入自身。
config.json: 100% 659/659 [00:00<00:00, 3.68MB/s]
tokenizer_config.json: 100% 19.0/19.0 [00:00<00:00, 121kB/s]
vocab.txt: 100% 110k/110k [00:00<00:00, 17.3MB/s]
tokenizer.json: 100% 269k/269k [00:00<00:00, 41.6MB/s]
added_tokens.json: 100% 2.00/2.00 [00:00<00:00, 13.5kB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 733kB/s]
训练文件 train_v1.tsv: 4691 条；验证文件 valid_v1.tsv: 583 条
特殊 token：{'[UNK]': 100, '[SEP]': 102, '[PAD]': 0, '[CLS]': 101, '[MASK]': 103}
[MASK] id = 103；词表大小 = 21128
[train] 读取 4691，可用 4691，长度不一致跳过 0，无错字 482，截断 0
[train] 错字位置 4209 个（平均 0.90/句），其中被 mask 3418；抽查正确位置 3733；最长 token 6
[valid] 读取 583，可用 583，长度不一致跳过 0，无错字 67，截断 0
[valid] 错字位置 516 个（平均 0.89/句），其中被 mask 409；抽查正确位置 474；最长 token 6
[检查] 标签中特殊 token 泄漏：0（应恒为 0）
[样例] 输入: [CLS] [UNK] [MASK] [SEP]
       标签: - - 人 -
[PASS] 预检通过：tokenizer 对齐、特殊 token、截断与标签口径均正常。
--preflight-only 已指定：不加载模型、不训练，退出。


In [56]:
import inspect
import torch
import transformers
from transformers import TrainingArguments, Trainer

print("===== 版本与硬件 =====")
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("显存总量:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GiB")

print("\n===== TrainingArguments 参数 =====")
params = inspect.signature(TrainingArguments.__init__).parameters
print("evaluation_strategy:", "evaluation_strategy" in params)
print("eval_strategy:", "eval_strategy" in params)

print("\n===== Trainer 参数 =====")
trainer_params = inspect.signature(Trainer.__init__).parameters
print("tokenizer:", "tokenizer" in trainer_params)
print("processing_class:", "processing_class" in trainer_params)

===== 版本与硬件 =====
Transformers: 5.18.0
PyTorch: 2.11.0+cu130
CUDA 可用: True
GPU: Tesla T4
显存总量: 14.56 GiB

===== TrainingArguments 参数 =====
evaluation_strategy: False
eval_strategy: True

===== Trainer 参数 =====
tokenizer: False
processing_class: True


In [57]:
!grep -n -E 'evaluation_strategy|Trainer\(|tokenizer=' scripts/train_correction.py

236:        evaluation_strategy="epoch",
248:    trainer = Trainer(
254:        tokenizer=tok,


In [58]:
from pathlib import Path
import shutil
import hashlib

path = Path("scripts/train_correction.py")
backup = Path("scripts/train_correction.py.bak_before_tf518")

assert path.exists(), f"找不到脚本：{path}"
assert not backup.exists(), f"备份已存在，为避免覆盖而停止：{backup}"

original = path.read_text(encoding="utf-8")

old_eval = 'evaluation_strategy="epoch"'
new_eval = 'eval_strategy="epoch"'

old_trainer = "        tokenizer=tok,"
new_trainer = "        processing_class=tok,"

# 修改前校验：确保每个目标只出现一次
assert original.count(old_eval) == 1, (
    f"evaluation_strategy 出现次数不是 1：{original.count(old_eval)}"
)
assert original.count(old_trainer) == 1, (
    f"tokenizer=tok 出现次数不是 1：{original.count(old_trainer)}"
)

# 先备份原文件
shutil.copy2(path, backup)
backup_sha = hashlib.sha256(backup.read_bytes()).hexdigest()

# 只替换指定的两处参数
updated = original.replace(old_eval, new_eval).replace(
    old_trainer, new_trainer
)

# 确认改动范围仅为两处指定替换
assert updated.count(new_eval) == 1
assert updated.count(new_trainer) == 1
assert updated.count(old_eval) == 0
assert updated.count(old_trainer) == 0

path.write_text(updated, encoding="utf-8")

print("[PASS] 训练脚本兼容性参数已修改")
print("原文件备份：", backup)
print("备份 SHA-256：", backup_sha)
print("修改文件：", path)

[PASS] 训练脚本兼容性参数已修改
原文件备份： scripts/train_correction.py.bak_before_tf518
备份 SHA-256： 78639d1aecfaa442e10692506194fdc14e4840fff9b5dd9e97aee3236f9f93b6
修改文件： scripts/train_correction.py


In [59]:
!python -m py_compile scripts/train_correction.py
!grep -n -E 'evaluation_strategy|eval_strategy|Trainer\(|tokenizer=|processing_class=' scripts/train_correction.py

236:        eval_strategy="epoch",
248:    trainer = Trainer(
254:        processing_class=tok,


In [60]:
!python scripts/train_correction.py \
  --base hfl/chinese-macbert-base \
  --train-file data/processed/train_v1.tsv \
  --valid-file data/processed/valid_v1.tsv \
  --output-dir models/macbert-typo \
  --preflight-only

中文纠错训练（对齐标签监督）：input=noisy，label=clean
这不是普通 MLM：损失的标签来自 clean 列，而非输入自身。
训练文件 train_v1.tsv: 4691 条；验证文件 valid_v1.tsv: 583 条
特殊 token：{'[UNK]': 100, '[SEP]': 102, '[PAD]': 0, '[CLS]': 101, '[MASK]': 103}
[MASK] id = 103；词表大小 = 21128
[train] 读取 4691，可用 4691，长度不一致跳过 0，无错字 482，截断 0
[train] 错字位置 4209 个（平均 0.90/句），其中被 mask 3418；抽查正确位置 3733；最长 token 6
[valid] 读取 583，可用 583，长度不一致跳过 0，无错字 67，截断 0
[valid] 错字位置 516 个（平均 0.89/句），其中被 mask 409；抽查正确位置 474；最长 token 6
[检查] 标签中特殊 token 泄漏：0（应恒为 0）
[样例] 输入: [CLS] [UNK] [MASK] [SEP]
       标签: - - 人 -
[PASS] 预检通过：tokenizer 对齐、特殊 token、截断与标签口径均正常。
--preflight-only 已指定：不加载模型、不训练，退出。


In [61]:
from pathlib import Path

p = Path("scripts/train_correction.py")
lines = p.read_text(encoding="utf-8").splitlines()

# 打印训练参数、Trainer 构造和训练保存相关代码
for start, end in [(195, 275), (275, 340)]:
    print(f"\n{'='*20} 第 {start}–{min(end, len(lines))} 行 {'='*20}")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i+1:4}: {lines[i]}")

print("\n===== 输出目录状态 =====")
out = Path("models/macbert-typo")
print("输出目录存在：", out.exists())
if out.exists():
    print("目录内容：", [x.name for x in out.iterdir()][:30])


==================== 第 195–269 行 ====================
 195:         print("[样例] " + describe_example(tok, train_ex[0]).replace("\n", "\n       "))
 196: 
 197:     if problems:
 198:         print("[FAIL] 预检未通过：" + "；".join(problems), file=sys.stderr)
 199:         return 1
 200:     print("[PASS] 预检通过：tokenizer 对齐、特殊 token、截断与标签口径均正常。")
 201: 
 202:     if args.preflight_only:
 203:         print("--preflight-only 已指定：不加载模型、不训练，退出。")
 204:         return 0
 205: 
 206:     model = AutoModelForMaskedLM.from_pretrained(args.base)
 207: 
 208:     class PairDataset(Dataset):
 209:         def __init__(self, examples):
 210:             self.examples = examples
 211: 
 212:         def __len__(self):
 213:             return len(self.examples)
 214: 
 215:         def __getitem__(self, i):
 216:             return self.examples[i]
 217: 
 218:     pad_id = tok.pad_token_id if tok.pad_token_id is not None else 0
 219: 
 220:     def collate_fn(batch):
 221:         b = cl.collate(list(bat

In [62]:
from pathlib import Path

smoke_dir = Path("models/macbert-smoke")
assert not smoke_dir.exists(), (
    f"{smoke_dir} 已存在，请先检查里面是否有需要保留的内容，不要直接覆盖。"
)
print("[PASS] 冒烟测试输出目录可用")

[PASS] 冒烟测试输出目录可用


In [63]:
!python scripts/train_correction.py \
  --base hfl/chinese-macbert-base \
  --train-file data/processed/train_v1.tsv \
  --valid-file data/processed/valid_v1.tsv \
  --output-dir models/macbert-smoke \
  --epochs 1 \
  --max-train-samples 128 \
  --batch-size 8 \
  --grad-accum 1 \
  --seed 42 \
  --fp16

中文纠错训练（对齐标签监督）：input=noisy，label=clean
这不是普通 MLM：损失的标签来自 clean 列，而非输入自身。
训练文件 train_v1.tsv: 4691 条；验证文件 valid_v1.tsv: 583 条
特殊 token：{'[UNK]': 100, '[SEP]': 102, '[PAD]': 0, '[CLS]': 101, '[MASK]': 103}
[MASK] id = 103；词表大小 = 21128
[train] 读取 4691，可用 128，长度不一致跳过 0，无错字 14，截断 0
[train] 错字位置 114 个（平均 0.89/句），其中被 mask 85；抽查正确位置 105；最长 token 5
[valid] 读取 583，可用 583，长度不一致跳过 0，无错字 67，截断 0
[valid] 错字位置 516 个（平均 0.89/句），其中被 mask 409；抽查正确位置 474；最长 token 6
[检查] 标签中特殊 token 泄漏：0（应恒为 0）
[样例] 输入: [CLS] [UNK] [MASK] [SEP]
       标签: - - 人 -
[PASS] 预检通过：tokenizer 对齐、特殊 token、截断与标签口径均正常。

pytorch_model.bin: downloading bytes:  77% 317M/412M [00:01<00:00, 322MB/s, 27.4MB/s  ]
pytorch_model.bin: downloading bytes:  89% 367M/412M [00:01<00:00, 248MB/s, 31.7MB/s  ]
pytorch_model.bin: reconstructing file:  65% 268M/412M [00:04<00:02, 61.6MB/s, 12.8MB/s  ]
pytorch_model.bin: downloading bytes: 100% 390M/390M [00:05<00:00, 70.3MB/s, 35.4MB/s  ]
pytorch_model.bin: reconstructing file: 100% 412M/412M [00:05<00:

In [64]:
import inspect
import transformers
from transformers import TrainingArguments

print("Transformers 版本：", transformers.__version__)

sig = inspect.signature(TrainingArguments.__init__)

print("\n===== 当前版本支持的训练参数 =====")
for name in sig.parameters:
    if any(k in name for k in [
        "warmup", "eval", "save", "best",
        "metric", "greater", "fp16",
        "gradient_accumulation", "report_to"
    ]):
        print(name)

Transformers 版本： 5.18.0

===== 当前版本支持的训练参数 =====
warmup_steps
gradient_accumulation_steps
fp16
bf16_full_eval
fp16_full_eval
report_to
eval_strategy
eval_steps
eval_delay
per_device_eval_batch_size
eval_on_start
eval_do_concat_batches
eval_use_gather_object
eval_accumulation_steps
include_for_metrics
batch_eval_metrics
save_only_model
save_strategy
save_steps
save_on_each_node
save_total_limit
load_best_model_at_end
metric_for_best_model
greater_is_better
skip_memory_metrics
do_eval


In [65]:
from pathlib import Path
import py_compile

p = Path("scripts/train_correction.py")
src = p.read_text(encoding="utf-8")

old = "        warmup_ratio=args.warmup_ratio,\n"
new = "        warmup_steps=warmup_steps,\n"

assert src.count(old) == 1, (
    "没有找到唯一的 warmup_ratio 参数，停止修改。"
)
assert src.count("    train_args = TrainingArguments(\n") == 1, (
    "TrainingArguments 位置不唯一，停止修改。"
)

# 根据训练样本数、批次大小、梯度累积和轮数估算预热步数
insert = (
    "    batches_per_epoch = "
    "(len(train_ex) + args.batch_size - 1) // args.batch_size\n"
    "    updates_per_epoch = "
    "(batches_per_epoch + args.grad_accum - 1) // args.grad_accum\n"
    "    total_steps_est = max(1, round(updates_per_epoch * args.epochs))\n"
    "    warmup_steps = round(total_steps_est * args.warmup_ratio)\n"
    "    print(f\"[训练参数] 预计总更新步数={total_steps_est}，"
    "预热步数={warmup_steps}\")\n\n"
)

src = src.replace(
    "    train_args = TrainingArguments(\n",
    insert + "    train_args = TrainingArguments(\n",
    1,
)
src = src.replace(old, new, 1)

p.write_text(src, encoding="utf-8")

# 语法检查
py_compile.compile(str(p), doraise=True)

print("[PASS] 脚本语法检查通过")
for i, line in enumerate(
    p.read_text(encoding="utf-8").splitlines(), start=1
):
    if "warmup_steps" in line or "warmup_ratio" in line:
        print(f"{i}: {line}")

[PASS] 脚本语法检查通过
231:     warmup_steps = round(total_steps_est * args.warmup_ratio)
232:     print(f"[训练参数] 预计总更新步数={total_steps_est}，预热步数={warmup_steps}")
241:         warmup_steps=warmup_steps,


In [66]:
!python scripts/train_correction.py \
  --base hfl/chinese-macbert-base \
  --train-file data/processed/train_v1.tsv \
  --valid-file data/processed/valid_v1.tsv \
  --output-dir models/macbert-smoke \
  --epochs 1 \
  --max-train-samples 128 \
  --batch-size 8 \
  --grad-accum 1 \
  --seed 42 \
  --fp16

中文纠错训练（对齐标签监督）：input=noisy，label=clean
这不是普通 MLM：损失的标签来自 clean 列，而非输入自身。
训练文件 train_v1.tsv: 4691 条；验证文件 valid_v1.tsv: 583 条
特殊 token：{'[UNK]': 100, '[SEP]': 102, '[PAD]': 0, '[CLS]': 101, '[MASK]': 103}
[MASK] id = 103；词表大小 = 21128
[train] 读取 4691，可用 128，长度不一致跳过 0，无错字 14，截断 0
[train] 错字位置 114 个（平均 0.89/句），其中被 mask 85；抽查正确位置 105；最长 token 5
[valid] 读取 583，可用 583，长度不一致跳过 0，无错字 67，截断 0
[valid] 错字位置 516 个（平均 0.89/句），其中被 mask 409；抽查正确位置 474；最长 token 6
[检查] 标签中特殊 token 泄漏：0（应恒为 0）
[样例] 输入: [CLS] [UNK] [MASK] [SEP]
       标签: - - 人 -
[PASS] 预检通过：tokenizer 对齐、特殊 token、截断与标签口径均正常。
Loading weights: 100% 204/204 [00:00<00:00, 24278.24it/s]
[transformers] BertForMaskedLM LOAD REPORT from: hfl/chinese-macbert-base
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:


In [67]:
from pathlib import Path
import json
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForMaskedLM

model_dir = Path("models/macbert-smoke")

print("===== 模型文件 =====")
assert model_dir.exists(), f"模型目录不存在：{model_dir}"

for p in sorted(model_dir.iterdir()):
    if p.is_file():
        print(f"{p.name}: {p.stat().st_size / 1024**2:.2f} MB")
    elif p.is_dir():
        print(f"{p.name}/")

print("\n===== 模型配置 =====")
config_path = model_dir / "config.json"
assert config_path.exists(), "缺少 config.json"

config = json.loads(config_path.read_text(encoding="utf-8"))
for key in ["model_type", "architectures", "vocab_size", "hidden_size"]:
    print(f"{key}: {config.get(key)}")

print("\n===== 重新加载模型 =====")
tokenizer = AutoTokenizer.from_pretrained(
    str(model_dir), local_files_only=True
)
model = AutoModelForMaskedLM.from_pretrained(
    str(model_dir), local_files_only=True
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

inputs = tokenizer("今天天气很好", return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

print("Transformers:", transformers.__version__)
print("设备:", device)
print("输入形状:", tuple(inputs["input_ids"].shape))
print("输出 logits 形状:", tuple(outputs.logits.shape))
print("[PASS] 模型重新加载并完成前向推理")

===== 模型文件 =====
checkpoint-16/
config.json: 0.00 MB
model.safetensors: 390.23 MB
tokenizer.json: 0.42 MB
tokenizer_config.json: 0.00 MB
training_args.bin: 0.00 MB

===== 模型配置 =====
model_type: bert
architectures: ['BertForMaskedLM']
vocab_size: 21128
hidden_size: 768

===== 重新加载模型 =====


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Transformers: 5.18.0
设备: cuda
输入形状: (1, 8)
输出 logits 形状: (1, 8, 21128)
[PASS] 模型重新加载并完成前向推理


In [68]:
from pathlib import Path
import subprocess

print("===== 评估脚本参数 =====")
result = subprocess.run(
    ["python", "scripts/eval_dual.py", "--help"],
    text=True,
    capture_output=True
)
print(result.stdout)
if result.stderr:
    print("===== stderr =====")
    print(result.stderr)

print("\n===== 数据文件检查 =====")
for name in [
    "data/processed/train_v1.tsv",
    "data/processed/valid_v1.tsv",
    "data/processed/test_v1.tsv",
]:
    p = Path(name)
    print(f"{name}: 存在={p.exists()}", end="")
    if p.exists():
        with p.open(encoding="utf-8") as f:
            print(f"，首行={f.readline().rstrip()!r}")
    else:
        print()

===== 评估脚本参数 =====
usage: eval_dual.py [-h] --mode {shuangpin,correction} [--errors ERRORS]
                    [--model MODEL] [--test-file TEST_FILE]
                    [--max-samples MAX_SAMPLES] [--max-length MAX_LENGTH]
                    [--batch-size BATCH_SIZE]
                    [--mask-correct-ratio MASK_CORRECT_RATIO]
                    [--mask-correct-prob MASK_CORRECT_PROB] [--seed SEED]
                    [--out OUT]

双层评估：双拼键码恢复 + 中文纠错

options:
  -h, --help            show this help message and exit
  --mode {shuangpin,correction}
  --errors ERRORS       shuangpin 模式：errors.jsonl
  --model MODEL         correction 模式：模型目录
  --test-file TEST_FILE
                        correction 模式：test.tsv
  --max-samples MAX_SAMPLES
  --max-length MAX_LENGTH
  --batch-size BATCH_SIZE
  --mask-correct-ratio MASK_CORRECT_RATIO
                        抽查多少比例的'本就正确'位置一起 mask，用于测误纠正率
  --mask-correct-prob MASK_CORRECT_PROB
  --seed SEED
  --out OUT


===== 数据文件检查 =====
data/processed

In [69]:
from pathlib import Path

p = Path("scripts/eval_dual.py")
lines = p.read_text(encoding="utf-8").splitlines()

print("===== 评估逻辑 =====")
keywords = [
    "def evaluate", "def main", "mask_correct",
    "correct_positions", "error_positions",
    "argmax", "predictions", "accuracy",
    "exact", "sentence", "changed", "corrected",
    "wrong", "mask_token_id", "labels",
]

hits = [
    i for i, line in enumerate(lines)
    if any(k in line.lower() for k in keywords)
]

# 打印命中位置前后代码，并合并重叠区间
ranges = []
for i in hits:
    start, end = max(0, i - 3), min(len(lines), i + 5)
    if ranges and start <= ranges[-1][1]:
        ranges[-1] = (ranges[-1][0], max(ranges[-1][1], end))
    else:
        ranges.append((start, end))

for start, end in ranges:
    print(f"\n--- 第 {start + 1}–{end} 行 ---")
    for i in range(start, end):
        print(f"{i + 1:4}: {lines[i]}")

print("\n===== 文件总行数 =====")
print(len(lines))

===== 评估逻辑 =====

--- 第 68–84 行 ---
  68:         b["unique"] += int(r["recovery_unique"])
  69: 
  70:     # "只信键码"的基线：直接把错误键码里 IME 唯一解出的音节当作答案
  71:     typed_unique = sum(1 for r in rows if len(r["wrong_code_candidates"]) == 1)
  72:     typed_correct = sum(1 for r in rows
  73:                         if len(r["wrong_code_candidates"]) == 1
  74:                         and r["wrong_code_candidates"][0] == r["syllable"])
  75: 
  76:     result = {
  77:         "mode": "shuangpin",
  78:         "file": str(errors_path.resolve()),
  79:         "records": n,
  80:         "distinct_wrong_codes": len({r["wrong_code"] for r in rows}),
  81:         "ime_decodable": decodable,
  82:         "ime_decodable_rate": round(decodable / n, 4),
  83:         "ime_ambiguous": ime_ambiguous,
  84:         "ime_ambiguous_rate": round(ime_ambiguous / n, 4),

--- 第 89–108 行 ---
  89:         "baseline_trust_typed_code": {
  90:             "unique_typed": typed_unique,
  91:             "correct"

In [70]:
!python scripts/eval_dual.py \
  --mode correction \
  --model models/macbert-smoke \
  --test-file data/processed/test_v1.tsv \
  --batch-size 16 \
  --max-length 128 \
  --mask-correct-ratio 0.2 \
  --seed 42 \
  --out models/macbert-smoke/eval_test_baseline.json

Loading weights: 100% 202/202 [00:00<00:00, 4941.18it/s]
设备：cuda；测试句：650；模型：/content/xime_work/xime_macbert_colab_ready/models/macbert-smoke
Traceback (most recent call last):
  File "/content/xime_work/xime_macbert_colab_ready/scripts/eval_dual.py", line 304, in <module>
    raise SystemExit(main())
                     ~~~~^^
  File "/content/xime_work/xime_macbert_colab_ready/scripts/eval_dual.py", line 298, in main
    return eval_correction(Path(args.model), Path(args.test_file), args.max_samples,
                           args.max_length, args.batch_size, args.mask_correct_ratio,
                           args.mask_correct_prob, args.seed, out)
  File "/content/xime_work/xime_macbert_colab_ready/scripts/eval_dual.py", line 230, in eval_correction
    run_batch(batch)
    ~~~~~~~~~^^^^^^^
  File "/content/xime_work/xime_macbert_colab_ready/scripts/eval_dual.py", line 185, in run_batch
    pred_ids = torch.topk(logits[i, idx], 3).indices.tolist()
                          ~~~~~~^

In [71]:
from pathlib import Path

p = Path("scripts/eval_dual.py")
lines = p.read_text(encoding="utf-8").splitlines()

for start, end in [(145, 215)]:
    print(f"===== 第 {start}–{end} 行 =====")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i+1:4}: {lines[i]}")

===== 第 145–215 行 =====
 145:         return 3
 146: 
 147:     for p in (model_dir, test_file):
 148:         if not p.exists():
 149:             print(f"[FAIL] 路径不存在：{p.resolve()}", file=sys.stderr)
 150:             return 2
 151: 
 152:     tok = AutoTokenizer.from_pretrained(str(model_dir))
 153:     model = AutoModelForMaskedLM.from_pretrained(str(model_dir)).eval()
 154:     device = "cuda" if torch.cuda.is_available() else "cpu"
 155:     model.to(device)
 156:     mask_id = tok.mask_token_id
 157:     if mask_id is None:
 158:         print("[FAIL] tokenizer 没有 [MASK]", file=sys.stderr)
 159:         return 2
 160: 
 161:     rng = random.Random(seed)
 162:     pairs = load_pairs(test_file, max_samples)
 163:     print(f"设备：{device}；测试句：{len(pairs)}；模型：{model_dir.resolve()}")
 164: 
 165:     typo_total = typo_hit1 = typo_hit3 = 0
 166:     keep_total = keep_hit = 0
 167:     pred_change = true_change = both = 0
 168:     skipped = 0
 169:     examples: list[dict] = []
 170: 

In [72]:
from pathlib import Path
import py_compile
import shutil

p = Path("scripts/eval_dual.py")
src = p.read_text(encoding="utf-8")

old = "                pred_ids = torch.topk(logits[i, idx], 3).indices.tolist()\n"
new = (
    "                position_logits = logits[i][idx]\n"
    "                k = min(3, position_logits.shape[-1])\n"
    "                pred_ids = torch.topk(position_logits, k).indices.tolist()\n"
)

assert src.count(old) == 1, "未找到唯一的目标代码行，停止修改。"

backup = Path("scripts/eval_dual.py.bak_before_topk_fix")
assert not backup.exists(), f"备份文件已存在：{backup}"
shutil.copy2(p, backup)

p.write_text(src.replace(old, new, 1), encoding="utf-8")
py_compile.compile(str(p), doraise=True)

print("[PASS] 已备份原脚本并完成语法检查")
print("备份：", backup)
print("修改：")
for i, line in enumerate(p.read_text(encoding="utf-8").splitlines(), 1):
    if 182 <= i <= 190:
        print(f"{i:4}: {line}")

[PASS] 已备份原脚本并完成语法检查
备份： scripts/eval_dual.py.bak_before_topk_fix
修改：
 182:         for i, x in enumerate(items):
 183:             for pos, kind in x["positions"]:
 184:                 idx = pos + 1  # [CLS] 偏移
 185:                 position_logits = logits[i][idx]
 186:                 k = min(3, position_logits.shape[-1])
 187:                 pred_ids = torch.topk(position_logits, k).indices.tolist()
 188:                 preds = [tok.convert_ids_to_tokens(j) for j in pred_ids]
 189:                 truth = x["clean"][pos]
 190:                 if kind == "typo":


In [73]:
!python scripts/eval_dual.py \
  --mode correction \
  --model models/macbert-smoke \
  --test-file data/processed/test_v1.tsv \
  --batch-size 16 \
  --max-length 128 \
  --mask-correct-ratio 0.2 \
  --seed 42 \
  --out models/macbert-smoke/eval_test_baseline.json

Loading weights: 100% 202/202 [00:00<00:00, 5786.23it/s]
设备：cuda；测试句：650；模型：/content/xime_work/xime_macbert_colab_ready/models/macbert-smoke
Traceback (most recent call last):
  File "/content/xime_work/xime_macbert_colab_ready/scripts/eval_dual.py", line 306, in <module>
    raise SystemExit(main())
                     ~~~~^^
  File "/content/xime_work/xime_macbert_colab_ready/scripts/eval_dual.py", line 300, in main
    return eval_correction(Path(args.model), Path(args.test_file), args.max_samples,
                           args.max_length, args.batch_size, args.mask_correct_ratio,
                           args.mask_correct_prob, args.seed, out)
  File "/content/xime_work/xime_macbert_colab_ready/scripts/eval_dual.py", line 232, in eval_correction
    run_batch(batch)
    ~~~~~~~~~^^^^^^^
  File "/content/xime_work/xime_macbert_colab_ready/scripts/eval_dual.py", line 185, in run_batch
    position_logits = logits[i][idx]
                      ~~~~~~~~~^^^^^
IndexError: index 2 i

In [74]:
from pathlib import Path

p = Path("scripts/eval_dual.py")
lines = p.read_text(encoding="utf-8").splitlines()

print("=" * 70)
print("A. 批次推理与 logits 索引（第 145–195 行）")
print("=" * 70)
for n in range(144, min(195, len(lines))):
    print(f"{n+1:4}: {lines[n]}")

print("\n" + "=" * 70)
print("B. 测试样本与 tokenizer 的构造（第 195–250 行）")
print("=" * 70)
for n in range(194, min(250, len(lines))):
    print(f"{n+1:4}: {lines[n]}")

A. 批次推理与 logits 索引（第 145–195 行）
 145:         return 3
 146: 
 147:     for p in (model_dir, test_file):
 148:         if not p.exists():
 149:             print(f"[FAIL] 路径不存在：{p.resolve()}", file=sys.stderr)
 150:             return 2
 151: 
 152:     tok = AutoTokenizer.from_pretrained(str(model_dir))
 153:     model = AutoModelForMaskedLM.from_pretrained(str(model_dir)).eval()
 154:     device = "cuda" if torch.cuda.is_available() else "cpu"
 155:     model.to(device)
 156:     mask_id = tok.mask_token_id
 157:     if mask_id is None:
 158:         print("[FAIL] tokenizer 没有 [MASK]", file=sys.stderr)
 159:         return 2
 160: 
 161:     rng = random.Random(seed)
 162:     pairs = load_pairs(test_file, max_samples)
 163:     print(f"设备：{device}；测试句：{len(pairs)}；模型：{model_dir.resolve()}")
 164: 
 165:     typo_total = typo_hit1 = typo_hit3 = 0
 166:     keep_total = keep_hit = 0
 167:     pred_change = true_change = both = 0
 168:     skipped = 0
 169:     examples: list[dict] = [

In [75]:
from pathlib import Path
import shutil
import py_compile

p = Path("scripts/eval_dual.py")
src = p.read_text(encoding="utf-8")

backup = Path("scripts/eval_dual.py.bak_before_special_tokens_fix")
assert not backup.exists(), f"备份文件已存在，为避免覆盖先停止：{backup}"

# 1. 在长度检查后，为输入添加与训练一致的特殊 Token
old1 = '''        if len(ids) > max_length - 2:
            skipped += 1
            continue
        typo_pos = [i for i, (a, b) in enumerate(zip(clean, noisy)) if a != b]
'''
new1 = '''        if len(ids) > max_length - 2:
            skipped += 1
            continue
        ids = [tok.cls_token_id] + ids + [tok.sep_token_id]
        typo_pos = [i for i, (a, b) in enumerate(zip(clean, noisy)) if a != b]
'''
assert src.count(old1) == 1, "未找到唯一的特殊 Token 插入位置，停止修改"

# 2. 掩码位置也要加 1，跳过开头的 [CLS]
old2 = '''        for p, _kind in positions:
            ids[p] = mask_id
'''
new2 = '''        for p, _kind in positions:
            ids[p + 1] = mask_id
'''
assert src.count(old2) == 1, "未找到唯一的掩码位置，停止修改"

# 先备份，再修改
shutil.copy2(p, backup)
src = src.replace(old1, new1, 1).replace(old2, new2, 1)
p.write_text(src, encoding="utf-8")

# 语法检查
py_compile.compile(str(p), doraise=True)

print("[PASS] 评估脚本修复完成")
print(f"[PASS] 原脚本备份：{backup}")
print("[PASS] 已添加 [CLS]/[SEP]，并修正字符掩码偏移")
print("[PASS] Python 语法检查通过")

[PASS] 评估脚本修复完成
[PASS] 原脚本备份：scripts/eval_dual.py.bak_before_special_tokens_fix
[PASS] 已添加 [CLS]/[SEP]，并修正字符掩码偏移
[PASS] Python 语法检查通过


In [76]:
!python scripts/eval_dual.py \
  --mode correction \
  --model models/macbert-smoke \
  --test-file data/processed/test_v1.tsv \
  --batch-size 16 \
  --max-length 128 \
  --mask-correct-ratio 0.2 \
  --seed 42 \
  --out models/macbert-smoke/eval_test_baseline.json

Loading weights: 100% 202/202 [00:00<00:00, 4568.07it/s]
设备：cuda；测试句：650；模型：/content/xime_work/xime_macbert_colab_ready/models/macbert-smoke
{
  "mode": "correction",
  "model": "/content/xime_work/xime_macbert_colab_ready/models/macbert-smoke",
  "test_file": "/content/xime_work/xime_macbert_colab_ready/data/processed/test_v1.tsv",
  "device": "cuda",
  "samples_skipped": 0,
  "typo_positions": 650,
  "correction_accuracy_top1": 0.0523,
  "correction_accuracy_top3": 0.1646,
  "kept_positions_sampled": 6,
  "preservation_rate": 0.0,
  "false_correction_rate": 1.0,
  "position_detection": {
    "precision": 0.9909,
    "recall": 1.0,
    "f1": 0.9954,
    "predicted_changes": 656,
    "true_changes": 650,
    "note": "把'模型改动了的位置'当作检测结果；基于本次 mask 位置集合，属代理指标。"
  },
  "mask_correct_ratio": 0.2,
  "metric_note": "错字位置在评估时被显式 mask（oracle 定位），不是端到端准确率。"
}
已写出：/content/xime_work/xime_macbert_colab_ready/models/macbert-smoke/eval_test_baseline.json
[PASS] 中文纠错评估完成。


In [77]:
!python scripts/train_correction.py \
  --base hfl/chinese-macbert-base \
  --train-file data/processed/train_v1.tsv \
  --valid-file data/processed/valid_v1.tsv \
  --output-dir models/macbert-correction-v1 \
  --epochs 3 \
  --batch-size 8 \
  --grad-accum 2 \
  --seed 42 \
  --fp16

中文纠错训练（对齐标签监督）：input=noisy，label=clean
这不是普通 MLM：损失的标签来自 clean 列，而非输入自身。
训练文件 train_v1.tsv: 4691 条；验证文件 valid_v1.tsv: 583 条
特殊 token：{'[UNK]': 100, '[SEP]': 102, '[PAD]': 0, '[CLS]': 101, '[MASK]': 103}
[MASK] id = 103；词表大小 = 21128
[train] 读取 4691，可用 4691，长度不一致跳过 0，无错字 482，截断 0
[train] 错字位置 4209 个（平均 0.90/句），其中被 mask 3418；抽查正确位置 3733；最长 token 6
[valid] 读取 583，可用 583，长度不一致跳过 0，无错字 67，截断 0
[valid] 错字位置 516 个（平均 0.89/句），其中被 mask 409；抽查正确位置 474；最长 token 6
[检查] 标签中特殊 token 泄漏：0（应恒为 0）
[样例] 输入: [CLS] [UNK] [MASK] [SEP]
       标签: - - 人 -
[PASS] 预检通过：tokenizer 对齐、特殊 token、截断与标签口径均正常。
Loading weights: 100% 204/204 [00:00<00:00, 17523.87it/s]
[transformers] BertForMaskedLM LOAD REPORT from: hfl/chinese-macbert-base
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

N

In [78]:
!python scripts/eval_dual.py \
  --mode correction \
  --model models/macbert-correction-v1 \
  --test-file data/processed/test_v1.tsv \
  --batch-size 16 \
  --max-length 128 \
  --mask-correct-ratio 0.2 \
  --seed 42 \
  --out models/macbert-correction-v1/eval_test_v1.json

Loading weights: 100% 202/202 [00:00<00:00, 4786.10it/s]
设备：cuda；测试句：650；模型：/content/xime_work/xime_macbert_colab_ready/models/macbert-correction-v1
{
  "mode": "correction",
  "model": "/content/xime_work/xime_macbert_colab_ready/models/macbert-correction-v1",
  "test_file": "/content/xime_work/xime_macbert_colab_ready/data/processed/test_v1.tsv",
  "device": "cuda",
  "samples_skipped": 0,
  "typo_positions": 650,
  "correction_accuracy_top1": 0.4154,
  "correction_accuracy_top3": 0.5538,
  "kept_positions_sampled": 6,
  "preservation_rate": 0.6667,
  "false_correction_rate": 0.3333,
  "position_detection": {
    "precision": 0.9969,
    "recall": 1.0,
    "f1": 0.9985,
    "predicted_changes": 652,
    "true_changes": 650,
    "note": "把'模型改动了的位置'当作检测结果；基于本次 mask 位置集合，属代理指标。"
  },
  "mask_correct_ratio": 0.2,
  "metric_note": "错字位置在评估时被显式 mask（oracle 定位），不是端到端准确率。"
}
已写出：/content/xime_work/xime_macbert_colab_ready/models/macbert-correction-v1/eval_test_v1.json
[PASS] 中文纠错评估完成。


In [79]:
!python scripts/eval_dual.py --help

usage: eval_dual.py [-h] --mode {shuangpin,correction} [--errors ERRORS]
                    [--model MODEL] [--test-file TEST_FILE]
                    [--max-samples MAX_SAMPLES] [--max-length MAX_LENGTH]
                    [--batch-size BATCH_SIZE]
                    [--mask-correct-ratio MASK_CORRECT_RATIO]
                    [--mask-correct-prob MASK_CORRECT_PROB] [--seed SEED]
                    [--out OUT]

双层评估：双拼键码恢复 + 中文纠错

options:
  -h, --help            show this help message and exit
  --mode {shuangpin,correction}
  --errors ERRORS       shuangpin 模式：errors.jsonl
  --model MODEL         correction 模式：模型目录
  --test-file TEST_FILE
                        correction 模式：test.tsv
  --max-samples MAX_SAMPLES
  --max-length MAX_LENGTH
  --batch-size BATCH_SIZE
  --mask-correct-ratio MASK_CORRECT_RATIO
                        抽查多少比例的'本就正确'位置一起 mask，用于测误纠正率
  --mask-correct-prob MASK_CORRECT_PROB
  --seed SEED
  --out OUT


In [80]:
from pathlib import Path

p = Path("scripts/eval_dual.py")
lines = p.read_text(encoding="utf-8").splitlines()

for start, end in [(1, 145), (235, 310)]:
    print(f"\n{'=' * 24} 第 {start}-{end} 行 {'=' * 24}")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1:4}: {lines[i]}")


======================== 第 1-145 行 ========================
   1: #!/usr/bin/env python
   2: # -*- coding: utf-8 -*-
   3: """双层评估：① 双拼键码恢复  ② 中文纠错（模型）。
   4: 
   5: 模式 shuangpin（纯标准库，本机可跑）：
   6:     读 scripts/generate_shuangpin_errors.py 产出的 JSONL，统计
   7:         - 错误键码在 IME 中是否可解码（decodable）
   8:         - IME 歧义率（同一键码对应多个音节）
   9:         - 仅凭键码能否唯一恢复目标音节（recovery_unique）
  10:         - 按错误类型的分组统计
  11:     结论用于判断"是否需要上下文模型"，不涉及 torch。
  12: 
  13: 模式 correction（需要 torch/transformers，标记为待验证）：
  14:     对测试集逐句：错字位置全部替换为 [MASK]，并随机抽查一部分"本来就正确"的位置也替换为
  15:     [MASK]，用模型还原，报告：
  16:         - 纠正准确率（错字位置 top-1 / top-3 命中正确字）
  17:         - 错误位置指标（把"改动了的位置"当作检测结果，与真实错字位置比 P/R/F1）
  18:         - 误纠正率（本来正确的位置被改坏的比例）
  19:         - 保持率（本来正确的位置保持不变的比例）
  20:     这是"已知候选位置集合"的评估，不是端到端产品指标。
  21: 
  22: 用法：
  23:     python scripts/eval_dual.py --mode shuangpin --errors data/generated/shuangpin_errors.jsonl
  24:     python scripts/eval_dual.py --mode correction --model models/macber

In [81]:
from pathlib import Path
import hashlib

files = [
    "data/raw/wiki_zh.txt",
    "data/raw/wiki_zh_simplified.txt",
    "configs/pinyin_inventory.json",
    "data/processed/wiki_lexicon_v1.tsv",
    "data/processed/train_v1.tsv",
    "data/processed/valid_v1.tsv",
    "data/processed/test_v1.tsv",
    "models/macbert-correction-v1/config.json",
    "models/macbert-correction-v1/model.safetensors",
]

for name in files:
    p = Path(name)
    if not p.is_file():
        print(f"[缺失] {name}")
        continue

    size = p.stat().st_size
    print(f"[存在] {name} | {size:,} bytes")

    if p.name.endswith(".tsv") and p.exists():
        with p.open("r", encoding="utf-8-sig") as f:
            print(f"       数据行数（含表头）: {sum(1 for _ in f):,}")

[存在] data/raw/wiki_zh.txt | 6,133,458 bytes
[存在] data/raw/wiki_zh_simplified.txt | 6,125,855 bytes
[存在] configs/pinyin_inventory.json | 235,499 bytes
[存在] data/processed/wiki_lexicon_v1.tsv | 3,802,823 bytes
       数据行数（含表头）: 100,002
[存在] data/processed/train_v1.tsv | 67,948 bytes
       数据行数（含表头）: 4,692
[存在] data/processed/valid_v1.tsv | 8,642 bytes
       数据行数（含表头）: 584
[存在] data/processed/test_v1.tsv | 9,508 bytes
       数据行数（含表头）: 651
[存在] models/macbert-correction-v1/config.json | 903 bytes
[存在] models/macbert-correction-v1/model.safetensors | 409,184,912 bytes


In [82]:
from pathlib import Path
import py_compile

script = r'''#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""中文纠错端到端基线评估：不使用真实错字位置作为输入。"""

import argparse
import csv
import json
import re
from pathlib import Path

import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

HAN = re.compile(r"^[\u3400-\u4DBF\u4E00-\u9FFF\uF900-\uFAFF]$")


def load_pairs(path, limit):
    rows = []
    with open(path, encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f, delimiter="\t")
        header = next(reader, None)
        if not header or header[:2] != ["clean", "noisy"]:
            raise ValueError(f"TSV 表头错误：{header}")
        for row in reader:
            if len(row) >= 2 and row[0] and row[1]:
                if len(row[0]) != len(row[1]):
                    continue
                rows.append((row[0], row[1]))
                if limit and len(rows) >= limit:
                    break
    return rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--test-file", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--batch-size", type=int, default=64)
    ap.add_argument("--max-samples", type=int, default=0)
    ap.add_argument("--max-length", type=int, default=128)
    ap.add_argument("--thresholds", default="0,1,2,3,4")
    args = ap.parse_args()

    model_dir = Path(args.model)
    data_file = Path(args.test_file)
    if not model_dir.exists() or not data_file.exists():
        raise FileNotFoundError("模型目录或数据文件不存在")

    tok = AutoTokenizer.from_pretrained(str(model_dir))
    model = AutoModelForMaskedLM.from_pretrained(str(model_dir)).eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    if tok.cls_token_id is None or tok.sep_token_id is None:
        raise ValueError("Tokenizer 缺少 CLS/SEP")
    if tok.mask_token_id is None:
        raise ValueError("Tokenizer 缺少 MASK")

    pairs = load_pairs(data_file, args.max_samples)
    thresholds = [float(x) for x in args.thresholds.split(",")]
    if not thresholds:
        raise ValueError("至少需要一个阈值")

    # 限定候选为单个汉字，避免预测出特殊 token 或非汉字 token。
    vocab = tok.get_vocab()
    candidate_ids = sorted({
        idx for token, idx in vocab.items()
        if HAN.fullmatch(token)
    })
    if not candidate_ids:
        raise ValueError("词表中没有找到单汉字候选")

    print(f"设备：{device}；句子：{len(pairs)}；汉字候选：{len(candidate_ids)}")

    # 每条句子的每个字符分别遮盖一次；其他位置保留 noisy 原文。
    jobs = []
    valid_pairs = []
    skipped = 0

    for clean, noisy in pairs:
        char_ids = tok.convert_tokens_to_ids(list(noisy))
        if len(char_ids) != len(noisy):
            skipped += 1
            continue
        if len(char_ids) + 2 > args.max_length:
            skipped += 1
            continue
        if any(x == tok.unk_token_id for x in char_ids):
            skipped += 1
            continue

        sid = len(valid_pairs)
        valid_pairs.append((clean, noisy))
        base = [tok.cls_token_id] + char_ids + [tok.sep_token_id]

        for pos, original_id in enumerate(char_ids):
            masked = base.copy()
            masked[pos + 1] = tok.mask_token_id
            jobs.append((sid, pos, original_id, masked))

    if not valid_pairs or not jobs:
        raise ValueError("没有可评估样本")

    # 每个字符单独遮盖推理，收集候选字分数及原字分数。
    predictions = [[None] * len(noisy) for _, noisy in valid_pairs]
    margins = [[None] * len(noisy) for _, noisy in valid_pairs]
    cand_tensor = torch.tensor(candidate_ids, dtype=torch.long)

    for start in range(0, len(jobs), args.batch_size):
        batch = jobs[start:start + args.batch_size]
        width = max(len(x[3]) for x in batch)
        input_ids = torch.full(
            (len(batch), width), tok.pad_token_id or 0, dtype=torch.long
        )
        attention = torch.zeros((len(batch), width), dtype=torch.long)

        for i, (_, _, _, ids) in enumerate(batch):
            input_ids[i, :len(ids)] = torch.tensor(ids)
            attention[i, :len(ids)] = 1

        with torch.inference_mode():
            logits = model(
                input_ids=input_ids.to(device),
                attention_mask=attention.to(device)
            ).logits.cpu()

        for i, (sid, pos, original_id, _) in enumerate(batch):
            token_logits = logits[i, pos + 1]
            scores = token_logits[cand_tensor]
            best_idx = int(torch.argmax(scores))
            best_id = candidate_ids[best_idx]
            original_score = float(token_logits[original_id])
            best_score = float(scores[best_idx])

            predictions[sid][pos] = tok.convert_ids_to_tokens(best_id)
            margins[sid][pos] = best_score - original_score

        if (start // args.batch_size) % 50 == 0:
            print(f"已处理字符位置：{min(start + len(batch), len(jobs))}/{len(jobs)}")

    results = []
    for threshold in thresholds:
        char_total = char_correct = 0
        typo_total = typo_fixed = 0
        keep_total = keep_preserved = 0
        exact_sentences = any_fixed = all_errors_fixed = 0

        for sid, (clean, noisy) in enumerate(valid_pairs):
            output = list(noisy)

            for pos, original in enumerate(noisy):
                candidate = predictions[sid][pos]
                margin = margins[sid][pos]

                if candidate != original and margin >= threshold:
                    output[pos] = candidate

                if original != clean[pos]:
                    typo_total += 1
                    if output[pos] == clean[pos]:
                        typo_fixed += 1
                else:
                    keep_total += 1
                    if output[pos] == original:
                        keep_preserved += 1

                char_total += 1
                if output[pos] == clean[pos]:
                    char_correct += 1

            output = "".join(output)
            errors = [i for i, (a, b) in enumerate(zip(clean, noisy)) if a != b]
            fixed_count = sum(output[i] == clean[i] for i in errors)
            if errors and fixed_count > 0:
                any_fixed += 1
            if errors and fixed_count == len(errors):
                all_errors_fixed += 1
            if output == clean:
                exact_sentences += 1

        results.append({
            "threshold": threshold,
            "character_accuracy": round(char_correct / char_total, 4),
            "typo_correction_rate": round(typo_fixed / typo_total, 4) if typo_total else None,
            "correct_char_preservation_rate": round(keep_preserved / keep_total, 4) if keep_total else None,
            "false_correction_rate": round(1 - keep_preserved / keep_total, 4) if keep_total else None,
            "sentence_exact_accuracy": round(exact_sentences / len(valid_pairs), 4),
            "sentences_with_any_error_fixed": any_fixed,
            "sentences_with_all_errors_fixed": all_errors_fixed,
            "sentences": len(valid_pairs),
            "typo_positions": typo_total,
            "originally_correct_positions": keep_total,
        })

    report = {
        "mode": "end_to_end_character_mask_sweep",
        "model": str(model_dir.resolve()),
        "data": str(data_file.resolve()),
        "device": device,
        "sentences_read": len(pairs),
        "sentences_evaluated": len(valid_pairs),
        "samples_skipped": skipped,
        "character_candidate_count": len(candidate_ids),
        "threshold_results": results,
        "note": (
            "每个字符分别遮盖预测；阈值按候选汉字相对原字的 logit 差决定是否修改。"
            "这是端到端基线，不是最终产品策略；阈值应在验证集选择，再在独立测试集报告。"
        ),
    }

    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(report, ensure_ascii=False, indent=2))
    print(f"[PASS] 报告已写出：{out.resolve()}")


if __name__ == "__main__":
    main()
'''

p = Path("scripts/eval_end_to_end.py")
assert not p.exists(), f"文件已存在，为避免覆盖停止：{p}"
p.write_text(script, encoding="utf-8")
py_compile.compile(str(p), doraise=True)
print("[PASS] 已创建 scripts/eval_end_to_end.py")
print("[PASS] Python 语法检查通过")

[PASS] 已创建 scripts/eval_end_to_end.py
[PASS] Python 语法检查通过


In [83]:
!python scripts/eval_end_to_end.py \
  --model models/macbert-correction-v1 \
  --test-file data/processed/valid_v1.tsv \
  --batch-size 64 \
  --max-length 128 \
  --thresholds 0,1,2,3,4 \
  --out models/macbert-correction-v1/eval_end_to_end_valid.json

Loading weights: 100% 202/202 [00:00<00:00, 4575.97it/s]
设备：cuda；句子：583；汉字候选：7322
已处理字符位置：64/1101
{
  "mode": "end_to_end_character_mask_sweep",
  "model": "/content/xime_work/xime_macbert_colab_ready/models/macbert-correction-v1",
  "data": "/content/xime_work/xime_macbert_colab_ready/data/processed/valid_v1.tsv",
  "device": "cuda",
  "sentences_read": 583,
  "sentences_evaluated": 516,
  "samples_skipped": 67,
  "character_candidate_count": 7322,
  "threshold_results": [
    {
      "threshold": 0.0,
      "character_accuracy": 0.1644,
      "typo_correction_rate": 0.2674,
      "correct_char_preservation_rate": 0.0735,
      "false_correction_rate": 0.9265,
      "sentence_exact_accuracy": 0.0116,
      "sentences_with_any_error_fixed": 138,
      "sentences_with_all_errors_fixed": 138,
      "sentences": 516,
      "typo_positions": 516,
      "originally_correct_positions": 585
    },
    {
      "threshold": 1.0,
      "character_accuracy": 0.1907,
      "typo_correction_rate": 

In [84]:
from pathlib import Path
from collections import Counter
import json

p = Path("data/raw/wiki_zh_simplified.txt")
lines = [x.strip() for x in p.read_text(encoding="utf-8").splitlines() if x.strip()]

print("语料总行数：", len(lines))
print("不同句子数：", len(set(lines)))
print("长度分布：", Counter(
    "8–19" if len(x) < 20 else
    "20–39" if len(x) < 40 else
    "40–59" if len(x) < 60 else "60+"
    for x in lines
))
print("\n随机展示前 10 条：")
for i, line in enumerate(lines[:10], 1):
    print(f"{i}. {line[:150]}")

语料总行数： 59910
不同句子数： 59910
长度分布： Counter({'20–39': 25718, '40–59': 15204, '8–19': 12379, '60+': 6609})

随机展示前 10 条：
1. 数学，是研究数量、结构以及空间等概念及其变化的一门学科，属于形式科学的一种
2. 数学利用抽象化和逻辑推理，从计数、计算、量度、对物体形状及运动的观察发展而成
3. 数学家们拓展这些概念，以公式化新的猜想，以及从选定的公理及定义出发，严谨地推导出一些定理
4. 基础数学的知识与运用是生活中不可或缺的一环
5. 对数学基本概念的完善，早在古埃及、美索不达米亚及古印度历史上的古代数学文本便可观见，而在古希腊那里有更为严谨的处理
6. 从那时开始，数学的发展便持续不断地小幅进展，至16世纪的文艺复兴时期，因为新的科学发现和数学革新两者的交互，致使数学的加速发展，直至今日
7. 数学并成为许多国家及地区的教育中的一部分
8. 数学在许多领域都有应用，包括科学、工程、医学、经济学和金融学等
9. 数学家也研究纯粹数学，就是数学本身的实质性内容，而不以任何实际应用为目标
10. 许多研究虽然以纯粹数学开始，但其过程中也发现许多可用之处


In [86]:
from pathlib import Path
import ast

files = [
    Path("scripts/xiaohe_codec.py"),
    Path("scripts/generate_shuangpin_errors.py"),
    Path("scripts/bridge_shuangpin_to_text.py"),
]

for path in files:
    print("\n" + "=" * 80)
    print("文件：", path)
    print("=" * 80)

    if not path.exists():
        print("[缺失]")
        continue

    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    print("函数和参数：")
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            args = [a.arg for a in node.args.args]
            print(f"  {node.name}({', '.join(args)})")

    print("\n与编码、解码、邻键、错误生成、候选筛选有关的代码行：")
    lines = source.splitlines()
    keywords = (
        "neighbor", "wrong_code", "encode(", "decode(",
        "is_valid(", "candidate", "perturb", "wrong",
        "generate", "def main", "add_argument"
    )
    hits = [
        i for i, line in enumerate(lines)
        if any(word in line.lower() for word in keywords)
    ]

    shown = set()
    for i in hits:
        for j in range(max(0, i - 1), min(len(lines), i + 2)):
            if j not in shown:
                print(f"{j + 1:4}: {lines[j]}")
                shown.add(j)
        if len(shown) > 180:
            print("  ...（输出达到上限）")
            break

print("\n[PASS] 只读审计完成，没有写入或修改文件。")


文件： scripts/xiaohe_codec.py
函数和参数：
  _reload_stdout()
  normalize_syllable(text)
  load_syllables(path)
  load(config_path, syllables_path)
  selftest()
  main()
  __init__(self, config, syllables)
  is_valid(self, syllable)
  split(self, syllable)
  encode(self, syllable)
  alternative_codes(self, syllable)
  decode(self, code, include_alternatives)
  decode_unique(self, code)

与编码、解码、邻键、错误生成、候选筛选有关的代码行：
  11:     * 映射全部来自 configs/xiaohe_shuangpin.json，代码里不写死任何键位。
  12:     * 与 configs/neighbors.json 完全独立：双拼映射不得混入邻键表，本模块也不读邻键表。
  13:     * 解码不做"猜一个最像的"：返回候选列表，由调用方判断是否唯一。
  76:     # ---------- 基本校验 ----------
  77:     def is_valid(self, syllable: str) -> bool:
  78:         try:
  79:             self.encode(syllable)
  80:         except CodecError:
  98:     # ---------- 编码 ----------
  99:     def encode(self, syllable: str) -> str:
 100:         """返回两键小写键码（与 Rime 输入约定一致；配置文件里存的是大写键位）。"""
 130:         alt = (final[0] + key).lower()
 131:         return [] if alt == self.encode(

In [87]:

from pathlib import Path
from collections import Counter
import csv
import json
import random
import re
import hashlib

from pypinyin import lazy_pinyin, Style
import scripts.xiaohe_codec as xc

# ---------- 路径与安全检查 ----------
root = Path(".")
corpus_path = root / "data/raw/wiki_zh_simplified.txt"
inventory_path = root / "configs/pinyin_inventory.json"
neighbors_path = root / "configs/neighbors.json"

out_tsv = root / "data/processed/context_pairs_pilot_v1.tsv"
out_jsonl = root / "data/processed/context_pairs_pilot_v1.jsonl"
out_report = root / "data/processed/context_pairs_pilot_v1.report.json"

for p in (corpus_path, inventory_path, neighbors_path):
    if not p.exists():
        raise FileNotFoundError(f"缺少必要文件：{p}")

for p in (out_tsv, out_jsonl, out_report):
    if p.exists():
        raise FileExistsError(f"为避免覆盖旧文件，停止运行：{p}")

rng = random.Random(42)
codec = xc.load()

inventory = json.loads(inventory_path.read_text(encoding="utf-8"))
neighbors = json.loads(neighbors_path.read_text(encoding="utf-8"))

chars = inventory["chars"]
by_syllable = inventory["by_syllable"]

HAN = re.compile(r"^[\u3400-\u4DBF\u4E00-\u9FFF\uF900-\uFAFF]$")

sentences = [
    x.strip()
    for x in corpus_path.read_text(encoding="utf-8").splitlines()
    if x.strip()
]
rng.shuffle(sentences)

# ---------- 构建上下文样本 ----------
rows = []
seen = set()
rejects = Counter()
max_sentences = min(500, len(sentences))

for sentence in sentences[:max_sentences]:
    # 逐句拼音有助于处理多音字，但仍不是人工校对结果。
    syllables = lazy_pinyin(
        sentence,
        style=Style.NORMAL,
        strict=False
    )

    if len(syllables) != len(sentence):
        rejects["pinyin_alignment"] += 1
        continue

    possible = []
    for pos, (clean_char, syl) in enumerate(zip(sentence, syllables)):
        if not HAN.fullmatch(clean_char):
            continue

        syl = syl.lower()
        if syl not in chars.get(clean_char, []):
            rejects["inventory_reading_mismatch"] += 1
            continue

        try:
            clean_code = codec.encode(syl)
        except Exception:
            rejects["invalid_clean_syllable"] += 1
            continue

        # 两个键位分别尝试，严格遵守邻键表方向。
        for key_pos, intended_key in enumerate(clean_code.upper()):
            for pressed_key in neighbors.get(intended_key, ""):
                if pressed_key.upper() == intended_key:
                    continue

                wrong_code = (
                    clean_code[:key_pos]
                    + pressed_key.lower()
                    + clean_code[key_pos + 1:]
                )

                # 默认只接受唯一解码，降低错误音节标签歧义。
                decoded = codec.decode(wrong_code)
                if len(decoded) != 1:
                    rejects["wrong_code_not_unique"] += 1
                    continue

                wrong_syl = decoded[0]
                candidates = by_syllable.get(wrong_syl, [])

                # 排除非单汉字候选和原字本身。
                candidates = [
                    c for c in candidates
                    if isinstance(c, str)
                    and HAN.fullmatch(c)
                    and c != clean_char
                    and wrong_syl in chars.get(c, [])
                ]

                if not candidates:
                    rejects["no_valid_noisy_char"] += 1
                    continue

                # 按候选表现有顺序选择；不更改候选表排序。
                noisy_char = candidates[0]

                # 再检查错误字编码必须确实对应错误键码。
                try:
                    if codec.encode(wrong_syl) != wrong_code:
                        rejects["wrong_code_roundtrip_failed"] += 1
                        continue
                except Exception:
                    rejects["wrong_code_roundtrip_failed"] += 1
                    continue

                noisy = sentence[:pos] + noisy_char + sentence[pos + 1:]
                if noisy == sentence:
                    continue

                pair = (sentence, noisy)
                if pair in seen:
                    continue

                possible.append({
                    "clean": sentence,
                    "noisy": noisy,
                    "position": pos,
                    "clean_char": clean_char,
                    "noisy_char": noisy_char,
                    "clean_syllable": syl,
                    "clean_code": clean_code,
                    "wrong_code": wrong_code,
                    "wrong_syllable": wrong_syl,
                    "key_position": key_pos,
                    "intended_key": intended_key,
                    "pressed_key": pressed_key,
                    "neighbor_direction": f"{intended_key}->{pressed_key}",
                    "wrong_code_candidates": decoded,
                    "candidate_count": len(candidates),
                    "source": "wiki_zh_simplified.txt",
                })

    if not possible:
        rejects["sentence_no_valid_mutation"] += 1
        continue

    # 每个原句最多保留一个错误版本，防止相同原句大量重复。
    chosen = rng.choice(possible)
    pair = (chosen["clean"], chosen["noisy"])
    if pair not in seen:
        seen.add(pair)
        rows.append(chosen)

# ---------- 最低样本量保护 ----------
if len(rows) < 50:
    raise RuntimeError(
        f"试验样本只有 {len(rows)} 条，低于安全门槛 50 条；"
        "没有写出文件。请先检查过滤统计。"
    )

# ---------- 写入独立文件 ----------
out_tsv.parent.mkdir(parents=True, exist_ok=True)

with out_tsv.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter="\t", lineterminator="\n")
    writer.writerow(["clean", "noisy"])
    for r in rows:
        writer.writerow([r["clean"], r["noisy"]])

with out_jsonl.open("w", encoding="utf-8", newline="\n") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

report = {
    "source": str(corpus_path),
    "source_sha256": hashlib.sha256(corpus_path.read_bytes()).hexdigest(),
    "inventory": str(inventory_path),
    "neighbors": str(neighbors_path),
    "seed": 42,
    "sentences_examined": max_sentences,
    "samples_written": len(rows),
    "unique_clean_sentences": len({r["clean"] for r in rows}),
    "all_pairs_equal_length": all(
        len(r["clean"]) == len(r["noisy"]) for r in rows
    ),
    "all_pairs_single_char_difference": all(
        sum(a != b for a, b in zip(r["clean"], r["noisy"])) == 1
        for r in rows
    ),
    "all_wrong_codes_uniquely_decoded": all(
        len(r["wrong_code_candidates"]) == 1 for r in rows
    ),
    "rejection_counts": dict(rejects),
    "examples": rows[:10],
}
out_report.write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("试验样本生成完成")
print("检查句子数：", max_sentences)
print("生成样本数：", len(rows))
print("过滤统计：", dict(rejects))
print("等长检查：", report["all_pairs_equal_length"])
print("单字符差异检查：", report["all_pairs_single_char_difference"])
print("错误键码唯一解码：", report["all_wrong_codes_uniquely_decoded"])
print("\n前 10 条样本：")
for i, r in enumerate(rows[:10], 1):
    print(
        f"{i}. clean: {r['clean']}\n"
        f"   noisy: {r['noisy']}\n"
        f"   位置: {r['position']}；"
        f"{r['clean_char']}({r['clean_syllable']}, {r['clean_code']})"
        f" -> {r['noisy_char']}({r['wrong_syllable']}, {r['wrong_code']})；"
        f"邻键: {r['neighbor_direction']}"
    )

print("\n文件：")
print(out_tsv)
print(out_jsonl)
print(out_report)

试验样本生成完成
检查句子数： 500
生成样本数： 306
过滤统计： {'wrong_code_not_unique': 22165, 'pinyin_alignment': 194, 'no_valid_noisy_char': 389}
等长检查： True
单字符差异检查： True
错误键码唯一解码： True

前 10 条样本：
1. clean: 「天文学」和「天体物理学」是同义词
   noisy: 「天文学」和「天体物理学」是容义词
   位置: 14；同(tong, ts) -> 容(rong, rs)；邻键: T->R
2. clean: 鲁迅的毕业执照（毕业证）写着：学生周树人，现年廿一岁，身中面白无须，浙江省绍兴府会稽人，今考得一等第三名
   noisy: 鲁迅的毕业执照（毕业证）写着：学生周树人，现年廿一岁，身中面白苏须，浙江省绍兴府会稽人，今考得一等第三名
   位置: 31；无(wu, wu) -> 苏(su, su)；邻键: W->S
3. clean: 广西壮族自治区属山地丘陵性盆地地貌
   noisy: 广于壮族自治区属山地丘陵性盆地地貌
   位置: 1；西(xi, xi) -> 于(xu, xu)；邻键: I->U
4. clean: 通过限制地下水开采，加强补给区水利建设等措施，现今济南各大泉群均已保证了全年喷涌
   noisy: 通过限制地下水开采，加强补给区水利建设等措施，现今济版各大泉群均已保证了全年喷涌
   位置: 26；南(nan, nj) -> 版(ban, bj)；邻键: N->B
5. clean: 其后子楚在秦王面前展现自己的才华，秦孝文王大悦，在丞相面前称儿子们没人比得上子楚，同时在华阳王后劝说下将子楚立为太子
   noisy: 区后子楚在秦王面前展现自己的才华，秦孝文王大悦，在丞相面前称儿子们没人比得上子楚，同时在华阳王后劝说下将子楚立为太子
   位置: 0；其(qi, qi) -> 区(qu, qu)；邻键: I->U
6. clean: 江南地区，占据两淮吴地的淮南节度使杨行密建吴国，镇海军节度使钱镠建吴越国，威武军节度使王审知建闽国
   noisy: 江南地区，占据两淮吴地的淮南节度使杨行密建吴国，镇海军节度使钱镠建吴越国，威武军节度其王审知建闽国
   位置: 42；使(sh

In [88]:
from pathlib import Path
from collections import Counter
import json
from pypinyin import lazy_pinyin, Style

path = Path("data/processed/context_pairs_pilot_v1.jsonl")
rows = [
    json.loads(line)
    for line in path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

issues = Counter()
examples = []
checked = 0

for r in rows:
    clean = r["clean"]
    noisy = r["noisy"]
    pos = r["position"]

    # 基本结构复核
    if len(clean) != len(noisy):
        issues["length_mismatch"] += 1
        continue

    diffs = [i for i, (a, b) in enumerate(zip(clean, noisy)) if a != b]
    if diffs != [pos]:
        issues["position_mismatch"] += 1
        continue

    clean_py = lazy_pinyin(clean, style=Style.NORMAL, strict=False)
    noisy_py = lazy_pinyin(noisy, style=Style.NORMAL, strict=False)

    if len(clean_py) != len(clean) or len(noisy_py) != len(noisy):
        issues["pinyin_alignment"] += 1
        continue

    checked += 1

    expected_clean = r["clean_syllable"].lower()
    expected_noisy = r["wrong_syllable"].lower()
    actual_clean = clean_py[pos].lower()
    actual_noisy = noisy_py[pos].lower()

    if actual_clean != expected_clean:
        issues["clean_context_reading_mismatch"] += 1
        if len(examples) < 20:
            examples.append({
                "kind": "clean",
                "sentence": clean,
                "position": pos,
                "char": clean[pos],
                "record_syllable": expected_clean,
                "context_pinyin": actual_clean,
            })

    if actual_noisy != expected_noisy:
        issues["noisy_context_reading_mismatch"] += 1
        if len(examples) < 20:
            examples.append({
                "kind": "noisy",
                "sentence": noisy,
                "position": pos,
                "char": noisy[pos],
                "record_syllable": expected_noisy,
                "context_pinyin": actual_noisy,
            })

print("读取样本：", len(rows))
print("完成检查：", checked)
print("问题统计：", dict(issues))
print("\n读音不一致示例：")
for i, x in enumerate(examples, 1):
    print(f"{i}. {x}")

读取样本： 306
完成检查： 306
问题统计： {'noisy_context_reading_mismatch': 105}

读音不一致示例：
1. {'kind': 'noisy', 'sentence': '广于壮族自治区属山地丘陵性盆地地貌', 'position': 1, 'char': '于', 'record_syllable': 'xu', 'context_pinyin': 'yu'}
2. {'kind': 'noisy', 'sentence': '江南地区，占据两淮吴地的淮南节度使杨行密建吴国，镇海军节度使钱镠建吴越国，威武军节度其王审知建闽国', 'position': 42, 'char': '其', 'record_syllable': 'ji', 'context_pinyin': 'qi'}
3. {'kind': 'noisy', 'sentence': '两晋及南北朝时期中原北部转乱频仍，汉人大量移入相对安定的江南地区，另有一定数量的汉人迁入岭南，促进了广州一带的经济开发', 'position': 12, 'char': '转', 'record_syllable': 'zhuai', 'context_pinyin': 'zhuan'}
4. {'kind': 'noisy', 'sentence': '日期可能有农业上、生活英、宗教上或社会上的重要性', 'position': 11, 'char': '英', 'record_syllable': 'yang', 'context_pinyin': 'ying'}
5. {'kind': 'noisy', 'sentence': '黄天源、采芝斋、干生元、松鹤楼、得月部、稻香村、叶受和、朱鸿兴等', 'position': 18, 'char': '部', 'record_syllable': 'pou', 'context_pinyin': 'bu'}
6. {'kind': 'noisy', 'sentence': '每当有战事发生，首先调发各军中的新附军出征，其余则从事屯田和工是造作', 'position': 31, 'char': '是', 'record_syllable': 'ti', 'context_pinyin': 'shi'}
7. {'kin

In [89]:

from pathlib import Path
from collections import Counter
import csv
import json
import hashlib
from pypinyin import lazy_pinyin, Style

src = Path("data/processed/context_pairs_pilot_v1.jsonl")
out_tsv = Path("data/processed/context_pairs_pilot_v2.tsv")
out_jsonl = Path("data/processed/context_pairs_pilot_v2.jsonl")
out_report = Path("data/processed/context_pairs_pilot_v2.report.json")

for p in (src, out_tsv, out_jsonl, out_report):
    if not p.exists() and p == src:
        raise FileNotFoundError(f"缺少输入文件：{p}")
    if p != src and p.exists():
        raise FileExistsError(f"为避免覆盖文件，停止运行：{p}")

rows = [
    json.loads(line)
    for line in src.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

kept = []
rejected = Counter()
examples = []

for r in rows:
    clean, noisy, pos = r["clean"], r["noisy"], r["position"]

    if len(clean) != len(noisy):
        rejected["length_mismatch"] += 1
        continue

    diffs = [
        i for i, (a, b) in enumerate(zip(clean, noisy))
        if a != b
    ]
    if diffs != [pos]:
        rejected["single_position_mismatch"] += 1
        continue

    clean_py = lazy_pinyin(clean, style=Style.NORMAL, strict=False)
    noisy_py = lazy_pinyin(noisy, style=Style.NORMAL, strict=False)

    if len(clean_py) != len(clean) or len(noisy_py) != len(noisy):
        rejected["pinyin_alignment"] += 1
        continue

    expected_clean = r["clean_syllable"].lower()
    expected_noisy = r["wrong_syllable"].lower()

    if clean_py[pos].lower() != expected_clean:
        rejected["clean_context_reading_mismatch"] += 1
        continue

    if noisy_py[pos].lower() != expected_noisy:
        rejected["noisy_context_reading_mismatch"] += 1
        if len(examples) < 10:
            examples.append({
                "clean": clean,
                "noisy": noisy,
                "position": pos,
                "noisy_char": noisy[pos],
                "expected_syllable": expected_noisy,
                "context_syllable": noisy_py[pos],
            })
        continue

    kept.append(r)

if not kept:
    raise RuntimeError("过滤后没有剩余样本；未写出文件。")

with out_tsv.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter="\t", lineterminator="\n")
    writer.writerow(["clean", "noisy"])
    for r in kept:
        writer.writerow([r["clean"], r["noisy"]])

with out_jsonl.open("w", encoding="utf-8", newline="\n") as f:
    for r in kept:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

report = {
    "input": str(src),
    "input_sha256": hashlib.sha256(src.read_bytes()).hexdigest(),
    "input_count": len(rows),
    "kept_count": len(kept),
    "rejected_count": len(rows) - len(kept),
    "rejected_by_reason": dict(rejected),
    "examples_of_reading_mismatch": examples,
    "note": (
        "这是基于 pypinyin 上下文读音的保守过滤；"
        "通过过滤不代表语义正确，也不代表所有样本都符合真实用户输入行为。"
    ),
}
out_report.write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("输入样本：", len(rows))
print("保留样本：", len(kept))
print("过滤样本：", len(rows) - len(kept))
print("过滤原因：", dict(rejected))
print("\n保留样本示例：")
for i, r in enumerate(kept[:10], 1):
    print(f"{i}. clean: {r['clean']}")
    print(f"   noisy: {r['noisy']}")
    print(
        f"   位置 {r['position']}："
        f"{r['clean_char']}({r['clean_syllable']}) -> "
        f"{r['noisy_char']}({r['wrong_syllable']})"
    )

print("\n报告：", out_report)
print("[PASS] v2 已生成，原始 v1 文件未修改。")

输入样本： 306
保留样本： 201
过滤样本： 105
过滤原因： {'noisy_context_reading_mismatch': 105}

保留样本示例：
1. clean: 「天文学」和「天体物理学」是同义词
   noisy: 「天文学」和「天体物理学」是容义词
   位置 14：同(tong) -> 容(rong)
2. clean: 鲁迅的毕业执照（毕业证）写着：学生周树人，现年廿一岁，身中面白无须，浙江省绍兴府会稽人，今考得一等第三名
   noisy: 鲁迅的毕业执照（毕业证）写着：学生周树人，现年廿一岁，身中面白苏须，浙江省绍兴府会稽人，今考得一等第三名
   位置 31：无(wu) -> 苏(su)
3. clean: 通过限制地下水开采，加强补给区水利建设等措施，现今济南各大泉群均已保证了全年喷涌
   noisy: 通过限制地下水开采，加强补给区水利建设等措施，现今济版各大泉群均已保证了全年喷涌
   位置 26：南(nan) -> 版(ban)
4. clean: 其后子楚在秦王面前展现自己的才华，秦孝文王大悦，在丞相面前称儿子们没人比得上子楚，同时在华阳王后劝说下将子楚立为太子
   noisy: 区后子楚在秦王面前展现自己的才华，秦孝文王大悦，在丞相面前称儿子们没人比得上子楚，同时在华阳王后劝说下将子楚立为太子
   位置 0：其(qi) -> 区(qu)
5. clean: 在古代，希腊是西方文化的发源地
   noisy: 在古代，自腊是西方文化的发源地
   位置 4：希(xi) -> 自(zi)
6. clean: 从孔雀王朝开始，佛教分别向南北两个方向传播
   noisy: 从孔雀王朝开始，佛教分别向南北两个唐向传播
   位置 17：方(fang) -> 唐(tang)
7. clean: 操纵部分由转向装置、变速装置、制动装置组成
   noisy: 操纵部分由转向装置、变速装置、制动装置苏成
   位置 19：组(zu) -> 苏(su)
8. clean: 心理学研究倾向于折衷主义，即从其他领域汲取知识以帮助理解、解释心理现象
   noisy: 心理学研究倾向于折衷不义，即从其他领域汲取知识以帮助理解、解释心理现象
   位置 10：主(zhu) -> 不(bu)
9. clean: 又说“左氏之病是以成

In [90]:
from pathlib import Path
from collections import Counter
import json
import random

path = Path("data/processed/context_pairs_pilot_v2.jsonl")
rows = [
    json.loads(line)
    for line in path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print("样本数：", len(rows))
print("记录字段：", sorted(rows[0].keys()))

# 统计替换字符和音节的分布
clean_chars = Counter()
noisy_chars = Counter()
clean_syllables = Counter()
wrong_syllables = Counter()
positions = Counter()
length_buckets = Counter()
sentence_pairs = set()

for r in rows:
    clean = r["clean"]
    noisy = r["noisy"]
    pos = r["position"]

    clean_chars[clean[pos]] += 1
    noisy_chars[noisy[pos]] += 1
    clean_syllables[r.get("clean_syllable", "?")] += 1
    wrong_syllables[r.get("wrong_syllable", "?")] += 1
    positions["句首" if pos < 5 else
              "句中" if pos < len(clean) - 5 else "句尾"] += 1
    length_buckets[
        "8–19" if len(clean) < 20 else
        "20–39" if len(clean) < 40 else
        "40–59" if len(clean) < 60 else "60+"
    ] += 1
    sentence_pairs.add((clean, noisy))

print("\n不同 clean/noisy 对：", len(sentence_pairs))
print("句长分布：", dict(length_buckets))
print("位置分布：", dict(positions))
print("原字音节 Top 15：", clean_syllables.most_common(15))
print("错误字音节 Top 15：", wrong_syllables.most_common(15))
print("原字符 Top 15：", clean_chars.most_common(15))
print("替换字符 Top 15：", noisy_chars.most_common(15))

# 固定随机种子，确保抽样结果可复现
sample = random.Random(20261018).sample(rows, min(30, len(rows)))

print("\n========== 随机抽查样本 ==========")
for i, r in enumerate(sample, 1):
    pos = r["position"]
    print(f"\n{i}. clean: {r['clean']}")
    print(f"   noisy: {r['noisy']}")
    print(
        f"   位置={pos}，"
        f"{r['clean'][pos]} -> {r['noisy'][pos]}，"
        f"音节={r.get('clean_syllable', '?')} -> "
        f"{r.get('wrong_syllable', '?')}"
    )

print("\n[只读审计完成] 未修改任何文件。")

样本数： 201
记录字段： ['candidate_count', 'clean', 'clean_char', 'clean_code', 'clean_syllable', 'intended_key', 'key_position', 'neighbor_direction', 'noisy', 'noisy_char', 'position', 'pressed_key', 'source', 'wrong_code', 'wrong_code_candidates', 'wrong_syllable']

不同 clean/noisy 对： 201
句长分布： {'8–19': 65, '40–59': 51, '20–39': 72, '60+': 13}
位置分布： {'句尾': 42, '句中': 115, '句首': 44}
原字音节 Top 15： [('de', 8), ('shi', 6), ('you', 5), ('tong', 4), ('zhi', 4), ('xing', 4), ('hua', 4), ('nian', 4), ('di', 4), ('yi', 4), ('cheng', 4), ('qi', 3), ('qu', 3), ('zuo', 3), ('hu', 3)]
错误字音节 Top 15： [('shu', 7), ('zhan', 5), ('ding', 5), ('ban', 4), ('dong', 4), ('yong', 4), ('he', 4), ('zi', 3), ('qun', 3), ('re', 3), ('jian', 3), ('dan', 3), ('san', 3), ('zhou', 3), ('hao', 3)]
原字符 Top 15： [('的', 8), ('化', 3), ('作', 3), ('三', 3), ('文', 3), ('地', 3), ('同', 2), ('南', 2), ('其', 2), ('在', 2), ('区', 2), ('念', 2), ('重', 2), ('原', 2), ('兴', 2)]
替换字符 Top 15： [('数', 7), ('战', 5), ('定', 5), ('版', 4), ('东', 4), ('用'

In [91]:

from pathlib import Path
from collections import Counter
import json

data_path = Path("data/processed/context_pairs_pilot_v2.jsonl")
neighbor_path = Path("configs/neighbors.json")

rows = [
    json.loads(line)
    for line in data_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
neighbors = json.loads(neighbor_path.read_text(encoding="utf-8"))

# 兼容邻键值是字符串或列表的配置格式
def get_neighbors(key):
    value = neighbors.get(key, "")
    if isinstance(value, str):
        return set(value)
    if isinstance(value, list):
        return set(value)
    return set()

issues = Counter()
examples = []
direction_counts = Counter()

for r in rows:
    clean_code = str(r.get("clean_code", "")).upper()
    wrong_code = str(r.get("wrong_code", "")).upper()
    pos = r.get("key_position")
    intended = str(r.get("intended_key", "")).upper()
    pressed = str(r.get("pressed_key", "")).upper()

    direction_counts[str(r.get("neighbor_direction", "?"))] += 1

    # 编码长度与键位位置
    if len(clean_code) != 2 or len(wrong_code) != 2:
        issues["code_length_invalid"] += 1
        continue

    if not isinstance(pos, int) or pos not in (0, 1):
        issues["key_position_invalid"] += 1
        continue

    # 两个编码必须只差一个键
    changed = [
        i for i, (a, b) in enumerate(zip(clean_code, wrong_code))
        if a != b
    ]
    if changed != [pos]:
        issues["code_diff_position_mismatch"] += 1
        if len(examples) < 15:
            examples.append({
                "issue": "code_diff_position_mismatch",
                "clean_code": clean_code,
                "wrong_code": wrong_code,
                "key_position": pos,
                "actual_changed_positions": changed,
            })
        continue

    # 记录中的按键必须与编码内容吻合
    if clean_code[pos] != intended:
        issues["intended_key_mismatch"] += 1
        if len(examples) < 15:
            examples.append({
                "issue": "intended_key_mismatch",
                "clean_code": clean_code,
                "key_position": pos,
                "recorded_intended_key": intended,
                "actual_key": clean_code[pos],
            })

    if wrong_code[pos] != pressed:
        issues["pressed_key_mismatch"] += 1
        if len(examples) < 15:
            examples.append({
                "issue": "pressed_key_mismatch",
                "wrong_code": wrong_code,
                "key_position": pos,
                "recorded_pressed_key": pressed,
                "actual_key": wrong_code[pos],
            })

    # 严格按配置中的方向验证，不做反向补全
    if pressed not in get_neighbors(intended):
        issues["not_in_directional_neighbor_map"] += 1
        if len(examples) < 15:
            examples.append({
                "issue": "not_in_directional_neighbor_map",
                "intended_key": intended,
                "pressed_key": pressed,
                "configured_neighbors": sorted(get_neighbors(intended)),
                "clean_code": clean_code,
                "wrong_code": wrong_code,
            })

    # 检查候选数量元数据
    candidates = r.get("wrong_code_candidates")
    count = r.get("candidate_count")

    if isinstance(candidates, list) and isinstance(count, int):
        if len(candidates) != count:
            issues["candidate_count_metadata_mismatch"] += 1

print("样本数：", len(rows))
print("问题统计：", dict(issues))
print("邻键方向字段分布：", direction_counts)
print("\n问题样本（最多 15 条）：")
for i, item in enumerate(examples, 1):
    print(f"{i}. {item}")

print("\n第一条完整记录：")
print(json.dumps(rows[0], ensure_ascii=False, indent=2))

print("\n邻键配置示例：")
for k in ["T", "R", "N", "B"]:
    print(f"{k} -> {sorted(get_neighbors(k))}")

print("\n[只读审计完成] 未修改任何文件。")

样本数： 201
问题统计： {'candidate_count_metadata_mismatch': 201}
邻键方向字段分布： Counter({'I->U': 10, 'I->K': 10, 'I->J': 9, 'D->R': 9, 'Z->S': 6, 'G->Y': 5, 'U->J': 5, 'D->S': 5, 'Z->A': 5, 'X->Z': 4, 'U->Y': 4, 'H->U': 4, 'N->J': 4, 'Y->T': 4, 'G->H': 4, 'E->S': 4, 'S->Z': 4, 'J->K': 4, 'U->H': 4, 'W->S': 3, 'N->B': 3, 'D->E': 3, 'K->M': 3, 'S->D': 3, 'R->T': 3, 'F->R': 3, 'Y->U': 3, 'J->I': 3, 'S->A': 3, 'V->G': 3, 'Y->H': 3, 'M->L': 3, 'T->R': 2, 'H->Y': 2, 'F->D': 2, 'T->F': 2, 'E->R': 2, 'H->J': 2, 'X->D': 2, 'F->G': 2, 'H->B': 2, 'M->N': 2, 'O->L': 2, 'X->C': 2, 'Y->G': 2, 'R->E': 2, 'R->D': 2, 'S->W': 2, 'F->T': 1, 'V->B': 1, 'E->D': 1, 'K->L': 1, 'A->Q': 1, 'O->I': 1, 'C->V': 1, 'B->N': 1, 'A->S': 1, 'I->O': 1, 'G->V': 1, 'Z->X': 1, 'C->X': 1, 'E->W': 1, 'G->F': 1, 'T->Y': 1, 'M->K': 1, 'D->X': 1, 'V->C': 1, 'J->N': 1, 'N->M': 1, 'Q->A': 1, 'D->F': 1, 'K->I': 1, 'U->I': 1, 'H->G': 1})

问题样本（最多 15 条）：

第一条完整记录：
{
  "clean": "「天文学」和「天体物理学」是同义词",
  "noisy": "「天文学」和「天体物理学」是容义词",
  "position": 

In [92]:
from pathlib import Path
import json

data_path = Path("data/processed/context_pairs_pilot_v2.jsonl")
inventory_path = Path("configs/pinyin_inventory.json")

rows = [
    json.loads(line)
    for line in data_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
inventory = json.loads(inventory_path.read_text(encoding="utf-8"))

print("库存顶层字段：", list(inventory.keys()))
print("by_syllable 类型：", type(inventory.get("by_syllable")).__name__)

by_syllable = inventory.get("by_syllable", {})
r = rows[0]
syllable = r["wrong_syllable"]

print("\n第一条样本：")
print("wrong_code：", r["wrong_code"])
print("wrong_code_candidates：", r["wrong_code_candidates"])
print("candidate_count：", r["candidate_count"])
print("wrong_syllable：", syllable)
print("\n库存中 rong 对应的原始数据：")
print(repr(by_syllable.get(syllable)))

value = by_syllable.get(syllable)
if isinstance(value, (list, tuple, set, dict, str)):
    print("库存候选数量（按容器长度计算）：", len(value))
else:
    print("库存候选数量：暂无法直接计算")

print("\n前 10 条记录的字段对照：")
for i, r in enumerate(rows[:10], 1):
    s = r["wrong_syllable"]
    v = by_syllable.get(s)
    inventory_count = (
        len(v) if isinstance(v, (list, tuple, set, dict, str))
        else None
    )
    print({
        "样本": i,
        "wrong_code": r["wrong_code"],
        "解码候选": r["wrong_code_candidates"],
        "错误音节": s,
        "candidate_count": r["candidate_count"],
        "库存候选数量": inventory_count,
    })

库存顶层字段： ['meta', 'chars', 'by_syllable', 'syllables']
by_syllable 类型： dict

第一条样本：
wrong_code： rs
wrong_code_candidates： ['rong']
candidate_count： 15
wrong_syllable： rong

库存中 rong 对应的原始数据：
['容', '荣', '融', '隔', '溶', '熔', '戎', '榕', '镕', '绒', '冗', '蓉', '茸', '嵘', '蝾']
库存候选数量（按容器长度计算）： 15

前 10 条记录的字段对照：
{'样本': 1, 'wrong_code': 'rs', '解码候选': ['rong'], '错误音节': 'rong', 'candidate_count': 15, '库存候选数量': 15}
{'样本': 2, 'wrong_code': 'su', '解码候选': ['su'], '错误音节': 'su', 'candidate_count': 20, '库存候选数量': 20}
{'样本': 3, 'wrong_code': 'bj', '解码候选': ['ban'], '错误音节': 'ban', 'candidate_count': 23, '库存候选数量': 23}
{'样本': 4, 'wrong_code': 'qu', '解码候选': ['qu'], '错误音节': 'qu', 'candidate_count': 27, '库存候选数量': 27}
{'样本': 5, 'wrong_code': 'zi', '解码候选': ['zi'], '错误音节': 'zi', 'candidate_count': 32, '库存候选数量': 32}
{'样本': 6, 'wrong_code': 'th', '解码候选': ['tang'], '错误音节': 'tang', 'candidate_count': 17, '库存候选数量': 17}
{'样本': 7, 'wrong_code': 'su', '解码候选': ['su'], '错误音节': 'su', 'candidate_count': 20, '库存候选数量': 20}
{'样本': 8,

In [93]:
from pathlib import Path
import ast

codec_path = Path("scripts/xiaohe_codec.py")

assert codec_path.exists(), f"找不到文件：{codec_path}"

source = codec_path.read_text(encoding="utf-8")
tree = ast.parse(source)

print("===== 脚本中的函数和类 =====")
for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
        print(f"{type(node).__name__}: {node.name}（第 {node.lineno} 行）")

print("\n===== 脚本源码（前 16000 字符）=====")
print(source[:16000])

===== 脚本中的函数和类 =====
FunctionDef: _reload_stdout（第 35 行）
ClassDef: CodecError（第 43 行）
FunctionDef: normalize_syllable（第 47 行）
ClassDef: XiaoheCodec（第 56 行）
FunctionDef: load_syllables（第 166 行）
FunctionDef: load（第 181 行）
FunctionDef: selftest（第 186 行）
FunctionDef: main（第 212 行）

===== 脚本源码（前 16000 字符）=====
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""小鹤双拼编解码器（XIME 相位一）。

只做三件事：
    1. 合法拼音音节 -> 双拼键码（两键，小写）
    2. 双拼键码 -> 可能的音节集合（**可能不唯一**，必须如实返回全部候选）
    3. 判断一个音节是否合法

设计约束：
    * 映射全部来自 configs/xiaohe_shuangpin.json，代码里不写死任何键位。
    * 与 configs/neighbors.json 完全独立：双拼映射不得混入邻键表，本模块也不读邻键表。
    * 解码不做"猜一个最像的"：返回候选列表，由调用方判断是否唯一。

用法：
    python scripts/xiaohe_codec.py encode zhong guo
    python scripts/xiaohe_codec.py decode vs go
    python scripts/xiaohe_codec.py decode lo --json
    python scripts/xiaohe_codec.py table
    python scripts/xiaohe_codec.py selftest
"""

from __future__ import annotations

import argparse
import json
import re
import sys
from pathlib import Path

DEFAULT

In [94]:
from pathlib import Path

# 自动定位项目根目录
candidates = [
    Path("/content/xime_work/xime_macbert_colab_ready"),
    Path.cwd(),
    Path("/content/xime_macbert_colab_ready"),
]

root = next(
    (
        p.resolve() for p in candidates
        if (p / "scripts/xiaohe_codec.py").exists()
        and (p / "configs/neighbors.json").exists()
    ),
    None
)

if root is None:
    print("没有找到项目目录。当前工作目录：", Path.cwd())
    print("请先运行：%cd /content/xime_work/xime_macbert_colab_ready")
else:
    print("找到项目目录：", root)
    for rel in [
        "data/processed/context_pairs_pilot_v2.jsonl",
        "configs/pinyin_inventory.json",
        "configs/neighbors.json",
        "configs/xiaohe_shuangpin.json",
        "configs/pinyin_syllables.txt",
        "scripts/xiaohe_codec.py",
    ]:
        p = root / rel
        print(("✓ " if p.exists() else "✗ "), rel)

找到项目目录： /content/xime_work/xime_macbert_colab_ready
✓  data/processed/context_pairs_pilot_v2.jsonl
✓  configs/pinyin_inventory.json
✓  configs/neighbors.json
✓  configs/xiaohe_shuangpin.json
✓  configs/pinyin_syllables.txt
✓  scripts/xiaohe_codec.py


In [95]:
import sys
import json
from pathlib import Path
from collections import Counter

# ---------- 1. 定位文件 ----------
ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
DATA = ROOT / "data/processed/context_pairs_pilot_v2.jsonl"
NEIGHBORS = ROOT / "configs/neighbors.json"
INVENTORY = ROOT / "configs/pinyin_inventory.json"
SYLLABLES = ROOT / "configs/pinyin_syllables.txt"
CODEC_CONFIG = ROOT / "configs/xiaohe_shuangpin.json"

for p in [DATA, NEIGHBORS, INVENTORY, SYLLABLES, CODEC_CONFIG]:
    assert p.exists(), f"文件不存在：{p}"

sys.path.insert(0, str(ROOT / "scripts"))
from xiaohe_codec import load as load_codec

codec = load_codec(str(CODEC_CONFIG), str(SYLLABLES))

neighbors = json.loads(NEIGHBORS.read_text(encoding="utf-8"))
inventory = json.loads(INVENTORY.read_text(encoding="utf-8"))
by_syllable = inventory["by_syllable"]

rows = [
    json.loads(line)
    for line in DATA.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print("样本数：", len(rows))
print("双拼编码器：加载成功")
print("库存音节映射数：", len(by_syllable))

# ---------- 2. 逐条检查 ----------
issues = Counter()
examples = {}

def report(code, message, row):
    issues[code] += 1
    examples.setdefault(code, [])
    if len(examples[code]) < 5:
        examples[code].append({
            "clean": row.get("clean"),
            "noisy": row.get("noisy"),
            "position": row.get("position"),
            "clean_syllable": row.get("clean_syllable"),
            "wrong_syllable": row.get("wrong_syllable"),
            "clean_code": row.get("clean_code"),
            "wrong_code": row.get("wrong_code"),
            "detail": message,
        })

for i, r in enumerate(rows, 1):
    clean = r["clean"]
    noisy = r["noisy"]
    cs = r["clean_syllable"]
    ws = r["wrong_syllable"]
    cc = r["clean_code"].lower()
    wc = r["wrong_code"].lower()
    pos = r["position"]

    # A. 长度与单字符差异
    if len(clean) != len(noisy):
        report("length_mismatch", f"{len(clean)} != {len(noisy)}", r)
        continue

    diff = [j for j, (a, b) in enumerate(zip(clean, noisy)) if a != b]

    if len(diff) != 1:
        report("not_single_char_change", f"差异位置={diff}", r)
    elif diff[0] != pos:
        report("position_mismatch", f"实际位置={diff[0]}，记录位置={pos}", r)

    if not (0 <= pos < len(clean)):
        report("position_out_of_range", str(pos), r)
        continue

    if clean[pos] != r["clean_char"]:
        report("clean_char_mismatch", f"实际={clean[pos]!r}", r)

    if noisy[pos] != r["noisy_char"]:
        report("noisy_char_mismatch", f"实际={noisy[pos]!r}", r)

    # B. 字符对应的音节库存
    clean_chars = by_syllable.get(cs, [])
    wrong_chars = by_syllable.get(ws, [])

    if r["clean_char"] not in clean_chars:
        report("clean_char_not_in_syllable_inventory",
               f"{r['clean_char']} 不在 {cs} 的库存中", r)

    if r["noisy_char"] not in wrong_chars:
        report("noisy_char_not_in_syllable_inventory",
               f"{r['noisy_char']} 不在 {ws} 的库存中", r)

    # C. 用项目真实编码器复核音节与键码
    try:
        actual_cc = codec.encode(cs)
        if actual_cc != cc:
            report("clean_code_mismatch",
                   f"编码器结果={actual_cc}，记录={cc}", r)
    except Exception as e:
        report("clean_encode_error", str(e), r)

    try:
        actual_wc = codec.encode(ws)
        allowed_wc = {actual_wc}
        allowed_wc.update(codec.alternative_codes(ws))
        if wc not in allowed_wc:
            report("wrong_code_mismatch",
                   f"音节编码允许值={sorted(allowed_wc)}，记录={wc}", r)
    except Exception as e:
        report("wrong_encode_error", str(e), r)

    # D. 解码候选必须与记录一致
    try:
        actual_candidates = codec.decode(wc)
        recorded_candidates = r.get("wrong_code_candidates", [])

        if actual_candidates != recorded_candidates:
            report("decode_candidates_mismatch",
                   f"实际={actual_candidates}，记录={recorded_candidates}", r)

        if ws not in actual_candidates:
            report("wrong_syllable_not_decoded",
                   f"{ws} 不在 {actual_candidates} 中", r)

    except Exception as e:
        report("decode_error", str(e), r)

    # E. candidate_count 是汉字候选数，不是音节解码数
    actual_count = len(wrong_chars)
    if r.get("candidate_count") != actual_count:
        report("candidate_count_mismatch",
               f"实际库存数={actual_count}，记录={r.get('candidate_count')}", r)

    # F. 邻键替换方向与实际双拼键位
    key_pos = r.get("key_position")
    intended = r.get("intended_key", "").upper()
    pressed = r.get("pressed_key", "").upper()

    if key_pos not in (0, 1):
        report("invalid_key_position", repr(key_pos), r)
    else:
        if cc[key_pos].upper() != intended:
            report("intended_key_mismatch",
                   f"clean_code[{key_pos}]={cc[key_pos].upper()}，记录={intended}", r)

        if wc[key_pos].upper() != pressed:
            report("pressed_key_mismatch",
                   f"wrong_code[{key_pos}]={wc[key_pos].upper()}，记录={pressed}", r)

        other = 1 - key_pos
        if cc[other] != wc[other]:
            report("other_key_changed",
                   f"clean={cc}，wrong={wc}，key_position={key_pos}", r)

        allowed_neighbors = [
            str(k).upper() for k in neighbors.get(intended, [])
        ]
        if pressed not in allowed_neighbors:
            report("directional_neighbor_mismatch",
                   f"{intended}->{pressed} 不在邻键表中；允许={allowed_neighbors}", r)

# ---------- 3. 输出结果 ----------
print("\n" + "=" * 60)
print("审计完成")
print("总样本数：", len(rows))
print("异常类型数：", len(issues))
print("异常记录数（按检查项累计，可能重复计数）：", sum(issues.values()))

if not issues:
    print("所有检查项均通过。")
else:
    print("\n===== 异常统计 =====")
    for code, count in issues.most_common():
        print(f"{code}: {count}")

    print("\n===== 每类异常最多展示 5 条 =====")
    for code, samples in examples.items():
        print(f"\n--- {code} ---")
        for sample in samples:
            print(json.dumps(sample, ensure_ascii=False))

print("\n注意：异常数量按检查项统计，同一条样本可能触发多个异常。")

样本数： 201
双拼编码器：加载成功
库存音节映射数： 409

审计完成
总样本数： 201
异常类型数： 1
异常记录数（按检查项累计，可能重复计数）： 3

===== 异常统计 =====
candidate_count_mismatch: 3

===== 每类异常最多展示 5 条 =====

--- candidate_count_mismatch ---
{"clean": "袁喜欢缠足的女人，所娶的太太和姨太太，除了朝鲜籍的几个姨太太是天足外，其余都是缠足的", "noisy": "袁喜欢缠足的女人，所娶的太太和姨太太，除了朝鲜籍的几个姨太太是天足外，其图都是缠足的", "position": 36, "clean_syllable": "yu", "wrong_syllable": "tu", "clean_code": "yu", "wrong_code": "tu", "detail": "实际库存数=17，记录=16"}
{"clean": "成吉思汗时期分疆裂土给东道诸王与西道诸王，东道诸王是成吉思汗的弟弟，大多分封于塞北东部与东北地区，从属性很高", "noisy": "成吉思汗时期分疆裂土给东道诸王与西道诸王，东道诸王是成吉思汗的弟弟，大多分封于色北东部与东北地区，从属性很高", "position": 39, "clean_syllable": "sai", "wrong_syllable": "se", "clean_code": "sd", "wrong_code": "se", "detail": "实际库存数=10，记录=9"}
{"clean": "中共中央宣传部还规定：国歌要求人人会唱", "noisy": "中共中央宣传部环规定：国歌要求人人会唱", "position": 7, "clean_syllable": "hai", "wrong_syllable": "huan", "clean_code": "hd", "wrong_code": "hr", "detail": "实际库存数=26，记录=25"}

注意：异常数量按检查项统计，同一条样本可能触发多个异常。


In [96]:
import json
from pathlib import Path

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")

data_path = ROOT / "data/processed/context_pairs_pilot_v2.jsonl"
inventory_path = ROOT / "configs/pinyin_inventory.json"

inventory = json.loads(inventory_path.read_text(encoding="utf-8"))
by_syllable = inventory["by_syllable"]

rows = [
    json.loads(line)
    for line in data_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print("===== 候选数量差异明细 =====")

for r in rows:
    syllable = r["wrong_syllable"]
    chars = by_syllable.get(syllable, [])
    actual_count = len(chars)
    recorded_count = r.get("candidate_count")

    if actual_count != recorded_count:
        print("\n原句：", r["clean"])
        print("错误句：", r["noisy"])
        print("错误字符：", r["noisy_char"])
        print("错误音节：", syllable)
        print("记录候选数：", recorded_count)
        print("当前库存候选数：", actual_count)
        print("当前库存候选字：", "".join(chars))
        print("数量差：", actual_count - recorded_count)

===== 候选数量差异明细 =====

原句： 袁喜欢缠足的女人，所娶的太太和姨太太，除了朝鲜籍的几个姨太太是天足外，其余都是缠足的
错误句： 袁喜欢缠足的女人，所娶的太太和姨太太，除了朝鲜籍的几个姨太太是天足外，其图都是缠足的
错误字符： 图
错误音节： tu
记录候选数： 16
当前库存候选数： 17
当前库存候选字： 图土余突途徒杜吐屠涂跌凸菟秃兔荼钍
数量差： 1

原句： 成吉思汗时期分疆裂土给东道诸王与西道诸王，东道诸王是成吉思汗的弟弟，大多分封于塞北东部与东北地区，从属性很高
错误句： 成吉思汗时期分疆裂土给东道诸王与西道诸王，东道诸王是成吉思汗的弟弟，大多分封于色北东部与东北地区，从属性很高
错误字符： 色
错误音节： se
记录候选数： 9
当前库存候选数： 10
当前库存候选字： 色塞寨铯瑟涩啬泣愬穑
数量差： 1

原句： 中共中央宣传部还规定：国歌要求人人会唱
错误句： 中共中央宣传部环规定：国歌要求人人会唱
错误字符： 环
错误音节： huan
记录候选数： 25
当前库存候选数： 26
当前库存候选字： 还环换援欢缓宦患桓灌幻皖唤痪焕瑗寰奂懽郇涣洹浣獾豢鬟
数量差： 1


In [97]:
!pip -q install -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 19.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.2.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.2.0 which is incompatible.
tokenizers 0.23.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.2.0 which is incompatible.
gradio-client 2.7.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.2.0 which is incompatible.


In [98]:
%pip install "huggingface_hub>=0.25.0,<1.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 2.2.0
    Uninstalling huggingface_hub-2.2.0:
      Successfully uninstalled huggingface_hub-2.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 5.18.0 requires huggingface-hub<3.0,>=1.31.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [99]:
%pip install "huggingface_hub>=1.31.0,<2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 19.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2


In [1]:
import huggingface_hub
import transformers
import datasets
import tokenizers

print("huggingface_hub:", huggingface_hub.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("tokenizers:", tokenizers.__version__)

huggingface_hub: 1.33.0
transformers: 5.18.0
datasets: 4.8.5
tokenizers: 0.23.2


In [2]:
%pip check

ipython 7.34.0 requires jedi, which is not installed.


In [3]:
from huggingface_hub import HfApi

repo_id = "shibing624/CSC"

try:
    api = HfApi()
    info = api.dataset_info(repo_id)

    print("数据集：", info.id)
    print("仓库版本：", info.sha)
    print("\n===== 文件列表 =====")

    for f in info.siblings:
        print(f.rfilename)

except Exception as e:
    print("读取数据集信息失败：")
    print(type(e).__name__, str(e))

数据集： shibing624/CSC
仓库版本： 40f6a4b1a42c0bcb74aaf1ea3c8bba5ce140b021

===== 文件列表 =====
.gitattributes
README.md
dev.json
test.json
train.json


In [4]:
from pathlib import Path
from huggingface_hub import hf_hub_download

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
OUT = ROOT / "data/raw/opensource_csc"
OUT.mkdir(parents=True, exist_ok=True)

REVISION = "40f6a4b1a42c0bcb74aaf1ea3c8bba5ce140b021"

readme_path = hf_hub_download(
    repo_id="shibing624/CSC",
    repo_type="dataset",
    filename="README.md",
    revision=REVISION,
)

readme = Path(readme_path).read_text(encoding="utf-8")

print(readme)

README.md:   0%|          | 0.00/2.77k [00:00<?, ?B/s]

---
license: apache-2.0
language:
- zh
tags:
- text-correction
pretty_name: CSC
task_categories:
- text-generation
---

# Dataset Card for CSC
中文拼写纠错数据集
- **Repository:** https://github.com/shibing624/pycorrector

## Dataset Description

Chinese Spelling Correction (CSC) is a task to detect and correct misspelled characters in Chinese texts. 

CSC is challenging since many Chinese characters are visually or phonologically similar but with quite different semantic meanings.

中文拼写纠错数据集，共27万条，是通过原始SIGHAN13、14、15年数据集和Wang271k数据集合并整理后得到，json格式，带错误字符位置信息。


### Original Dataset Summary

- test.json 和 dev.json 为 **SIGHAN数据集**， 包括SIGHAN13 14 15，来自 [官方csc.html](http://nlp.ee.ncu.edu.tw/resource/csc.html) ，文件大小：339kb，4千条。
- train.json 为 **Wang271k数据集**，包括 Wang271k ，来自 [Automatic-Corpus-Generation dimmywang提供](https://github.com/wdimmy/Automatic-Corpus-Generation/blob/master/corpus/train.sgml) ，文件大小：93MB，27万条。

如果只想用SIGHAN数据集，可以这样取数据：
```python
from datasets import load_dataset
dev_ds = load_data

In [5]:
from datasets import load_dataset

ds = load_dataset(
    "shibing624/CSC",
    revision=REVISION,
)

print(ds)

for split in ds:
    print(f"\n===== {split} =====")
    print("样本数：", len(ds[split]))
    if len(ds[split]) > 0:
        print(ds[split][0])

train.json: reconstructing file:   0%|          |  0.00B / 99.0MB            

train.json: downloading bytes:           |  0.00B            

dev.json: reconstructing file:   0%|          |  0.00B / 11.1MB            

dev.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'original_text', 'wrong_ids', 'correct_text'],
        num_rows: 251835
    })
    validation: Dataset({
        features: ['id', 'original_text', 'wrong_ids', 'correct_text'],
        num_rows: 27981
    })
    test: Dataset({
        features: ['id', 'original_text', 'wrong_ids', 'correct_text'],
        num_rows: 1100
    })
})

===== train =====
样本数： 251835
{'id': '-', 'original_text': '目前区次事件的细节还不清楚，伤亡人数也未确定。', 'wrong_ids': [2], 'correct_text': '目前这次事件的细节还不清楚，伤亡人数也未确定。'}

===== validation =====
样本数： 27981
{'id': 'A2-0079-4', 'original_text': '第二个象片儿：四个学生都在教室。', 'wrong_ids': [3], 'correct_text': '第二个相片儿：四个学生都在教室。'}

===== test =====
样本数： 1100
{'id': 'A2-0011-1', 'original_text': '你好！我是张爱文。', 'wrong_ids': [], 'correct_text': '你好！我是张爱文。'}


In [6]:
from collections import Counter

assert "ds" in globals(), "没有找到 ds，请先重新运行 load_dataset 单元。"

train = ds["train"]

stats = Counter()
samples = {}
seen_pairs = set()
duplicate_pairs = 0

def record_issue(name, row, detail):
    stats[name] += 1
    samples.setdefault(name, [])
    if len(samples[name]) < 5:
        samples[name].append({
            "id": row.get("id"),
            "original_text": row.get("original_text"),
            "correct_text": row.get("correct_text"),
            "wrong_ids": row.get("wrong_ids"),
            "detail": detail,
        })

for row in train:
    original = row.get("original_text")
    correct = row.get("correct_text")
    wrong_ids = row.get("wrong_ids")

    # 基础字段检查
    if not isinstance(original, str) or not isinstance(correct, str):
        record_issue("invalid_text_type", row, "文本字段不是字符串")
        continue

    if not original or not correct:
        record_issue("empty_text", row, "原句或正句为空")

    if not isinstance(wrong_ids, list):
        record_issue("invalid_wrong_ids_type", row, "wrong_ids 不是列表")
        wrong_ids = []

    # 统计原句与正句的字符差异
    actual_diff = [
        i for i, (a, b) in enumerate(zip(original, correct))
        if a != b
    ]

    # 不等长时，先记录长度问题；后续不把补齐位置
    # 简单当成普通单字符差异
    if len(original) != len(correct):
        record_issue(
            "length_mismatch", row,
            f"原句长度={len(original)}，正句长度={len(correct)}"
        )
    else:
        if not actual_diff:
            stats["no_text_difference"] += 1
        elif len(actual_diff) == 1:
            stats["single_char_edit"] += 1
        else:
            stats["multi_char_edit"] += 1

        # 核对标注位置是否恰好等于字符差异位置
        valid_ids = (
            all(isinstance(x, int) and not isinstance(x, bool)
                and 0 <= x < len(original) for x in wrong_ids)
            and len(set(wrong_ids)) == len(wrong_ids)
        )

        if not valid_ids:
            record_issue("invalid_wrong_ids", row, "位置越界、类型错误或重复")
        elif sorted(wrong_ids) != actual_diff:
            record_issue(
                "wrong_ids_mismatch", row,
                f"实际差异位置={actual_diff}，标注位置={sorted(wrong_ids)}"
            )

    if not wrong_ids:
        stats["empty_wrong_ids"] += 1

    # 统计完全相同的原句/正句对
    pair = (original, correct)
    if pair in seen_pairs:
        duplicate_pairs += 1
    else:
        seen_pairs.add(pair)

print("===== 训练集质量审计 =====")
print("总样本数：", len(train))
print("原句/正句对的唯一数量：", len(seen_pairs))
print("重复对出现次数（不含首次）：", duplicate_pairs)

for key in [
    "single_char_edit",
    "multi_char_edit",
    "no_text_difference",
    "empty_wrong_ids",
    "length_mismatch",
    "invalid_text_type",
    "empty_text",
    "invalid_wrong_ids_type",
    "invalid_wrong_ids",
    "wrong_ids_mismatch",
]:
    print(f"{key}: {stats[key]}")

print("\n===== 异常样本示例（每类最多 5 条）=====")
for name, examples in samples.items():
    print(f"\n--- {name} ---")
    for item in examples:
        print(item)

===== 训练集质量审计 =====
总样本数： 251835
原句/正句对的唯一数量： 248782
重复对出现次数（不含首次）： 3053
single_char_edit: 170328
multi_char_edit: 78639
no_text_difference: 2868
empty_wrong_ids: 2759
length_mismatch: 0
invalid_text_type: 0
empty_text: 0
invalid_wrong_ids_type: 0
invalid_wrong_ids: 13
wrong_ids_mismatch: 248

===== 异常样本示例（每类最多 5 条）=====

--- wrong_ids_mismatch ---
{'id': 'B1-2392-1', 'original_text': '我来台湾以前住在日本大坂，大坂的值钱是比台湾贵，租房子也一样贵，可是我觉得日本的房子有小有贵也干净。', 'correct_text': '我来台湾以前住在日本大坂，大坂的值钱是比台湾贵，租房子也一样贵，可是我觉得日本的房子又小又小又贵净。', 'wrong_ids': [42, 44], 'detail': '实际差异位置=[42, 44, 45, 46, 47]，标注位置=[42, 44]'}
{'id': 'B1-3561-1', 'original_text': '有一次我回到我房间看到一本「讲话秘决」，我翻来翻去不晓的是谁放的或是别人忘记拿走？问同学旁边才知道老师特别送给一个冷冷静静、少说话的我。', 'correct_text': '有一次我回到我房间看到一本「讲话秘诀」，我翻来翻去不晓得是谁放的或是别人忘记拿走？问同学旁边才知道老师特别送给一个冷冷静静、少说话的我。', 'wrong_ids': [4, 17, 27], 'detail': '实际差异位置=[17, 27]，标注位置=[4, 17, 27]'}
{'id': 'B1-1574-1', 'original_text': '除了我有的时候没办法接受老师课的方式以外，我老师的课什么方面都喜欢。', 'correct_text': '除了我有的时候没办法接受老师课的方式以外，我老师的课什么方面都喜欢。', 'wrong_ids':

In [7]:
from collections import Counter
from pprint import pprint

assert "ds" in globals(), "请先运行成功加载数据集的单元。"

train = ds["train"]

counts = Counter()
examples = {}
pair_counts = Counter()

def save_example(category, row, detail):
    bucket = examples.setdefault(category, [])
    if len(bucket) < 5:
        bucket.append({
            "id": row.get("id"),
            "original_text": row.get("original_text"),
            "correct_text": row.get("correct_text"),
            "wrong_ids": row.get("wrong_ids"),
            "detail": detail,
        })

for row in train:
    original = row["original_text"]
    correct = row["correct_text"]
    ids = row["wrong_ids"]

    pair_counts[(original, correct)] += 1

    # 数据集已审计过：文本类型有效，且原句与正句等长
    actual = {
        i for i, (a, b) in enumerate(zip(original, correct))
        if a != b
    }

    counts["total"] += 1

    if not actual:
        counts["no_text_diff"] += 1
    elif len(actual) == 1:
        counts["single_diff"] += 1
    else:
        counts["multi_diff"] += 1

    if not ids:
        counts["empty_ids"] += 1
        if actual:
            save_example(
                "empty_ids_but_text_diff", row,
                f"实际差异位置={sorted(actual)}，wrong_ids 为空"
            )
        else:
            counts["no_diff_and_empty_ids"] += 1
        continue

    # 先验证标注格式，避免把重复位置直接转成集合后掩盖问题
    ids_format_ok = (
        isinstance(ids, list)
        and all(
            isinstance(x, int)
            and not isinstance(x, bool)
            and 0 <= x < len(original)
            for x in ids
        )
        and len(set(ids)) == len(ids)
    )

    if not ids_format_ok:
        counts["invalid_ids"] += 1
        save_example(
            "invalid_ids",
            row,
            f"标注存在重复、越界或类型问题：{ids}"
        )
        continue

    marked = set(ids)
    missing = actual - marked
    extra = marked - actual

    if not missing and not extra:
        counts["ids_exact_match"] += 1
    elif missing and not extra:
        counts["ids_missing_some_diffs"] += 1
        save_example(
            "ids_missing_some_diffs",
            row,
            f"未标注的实际差异位置={sorted(missing)}"
        )
    elif extra and not missing:
        counts["ids_include_unchanged_positions"] += 1
        save_example(
            "ids_include_unchanged_positions",
            row,
            f"标注中没有发生变化的位置={sorted(extra)}"
        )
    else:
        counts["ids_both_missing_and_extra"] += 1
        save_example(
            "ids_both_missing_and_extra",
            row,
            f"漏标位置={sorted(missing)}；多标位置={sorted(extra)}"
        )

    if not actual and marked:
        counts["no_diff_but_ids_nonempty"] += 1
        save_example(
            "no_diff_but_ids_nonempty",
            row,
            f"文本没有差异，但标注位置={sorted(marked)}"
        )

# 精确重复对统计
duplicate_excess = sum(n - 1 for n in pair_counts.values() if n > 1)
duplicate_groups = sum(1 for n in pair_counts.values() if n > 1)

print("===== 细分审计统计 =====")
for key, value in counts.most_common():
    print(f"{key}: {value}")

print("\n===== 重复样本 =====")
print("不同原句/正句对数量：", len(pair_counts))
print("出现重复的句子对组数：", duplicate_groups)
print("重复出现次数（不含首次）：", duplicate_excess)

print("\n===== 异常样本示例 =====")
for category, rows in examples.items():
    print(f"\n--- {category}（最多展示 5 条）---")
    pprint(rows, sort_dicts=False)

===== 细分审计统计 =====
total: 251835
ids_exact_match: 248815
single_diff: 170328
multi_diff: 78639
no_text_diff: 2868
empty_ids: 2759
no_diff_and_empty_ids: 2759
ids_include_unchanged_positions: 157
no_diff_but_ids_nonempty: 109
ids_missing_some_diffs: 70
ids_both_missing_and_extra: 21
invalid_ids: 13

===== 重复样本 =====
不同原句/正句对数量： 248782
出现重复的句子对组数： 1534
重复出现次数（不含首次）： 3053

===== 异常样本示例 =====

--- ids_missing_some_diffs（最多展示 5 条）---
[{'id': 'B1-2392-1',
  'original_text': '我来台湾以前住在日本大坂，大坂的值钱是比台湾贵，租房子也一样贵，可是我觉得日本的房子有小有贵也干净。',
  'correct_text': '我来台湾以前住在日本大坂，大坂的值钱是比台湾贵，租房子也一样贵，可是我觉得日本的房子又小又小又贵净。',
  'wrong_ids': [42, 44],
  'detail': '未标注的实际差异位置=[45, 46, 47]'},
 {'id': 'B1-1174-2',
  'original_text': '那里的动物，对我很好，连狮子也对我好。那只狮子带我去山外面。那个时候他把我放在他的上面，所以我可能坐跑的狮子上。',
  'correct_text': '那里的动物，对我很好，连狮子也对我好。那只狮子带我去山外面。那个时候牠把我放在牠把我放在牠的上面能坐跑的狮子上。',
  'wrong_ids': [34, 39],
  'detail': '未标注的实际差异位置=[40, 41, 42, 43, 44, 45, 46, 47]'},
 {'id': 'B2-2236-3',
  'original_text': '一份耕耘一份收获，只要努力的从事自己的工作，不管收入的高低未来一

In [8]:
from collections import Counter

assert "ds" in globals(), "请先加载 Hugging Face 数据集。"

train = ds["train"]

unique_pairs = {}
stats = Counter()

for row in train:
    original = row["original_text"]
    correct = row["correct_text"]

    if original == correct:
        stats["same_text"] += 1
        continue

    # 以原句和正句组成唯一键
    pair = (original, correct)

    if pair in unique_pairs:
        stats["duplicate_excess"] += 1
        continue

    unique_pairs[pair] = True

    diffs = sum(a != b for a, b in zip(original, correct))
    stats["unique_changed_pairs"] += 1

    if diffs == 1:
        stats["unique_single_char"] += 1
    else:
        stats["unique_multi_char"] += 1

print("===== 第一版候选规模 =====")
print("原始训练集：", len(train))
print("完全相同的文本对：", stats["same_text"])
print("重复出现次数（已变化的文本对）：", stats["duplicate_excess"])
print("去重后、有实际差异的句子对：", len(unique_pairs))
print("其中单字符差异：", stats["unique_single_char"])
print("其中多字符差异：", stats["unique_multi_char"])

===== 第一版候选规模 =====
原始训练集： 251835
完全相同的文本对： 2868
重复出现次数（已变化的文本对）： 3048
去重后、有实际差异的句子对： 245919
其中单字符差异： 167790
其中多字符差异： 78129


In [9]:
import random

assert "ds" in globals(), "请先运行加载数据集的单元。"

# 按原句/正句对去重，并重新计算真实字符差异
seen = set()
single_samples = []
multi_samples = []

for row in ds["train"]:
    original = row["original_text"]
    correct = row["correct_text"]

    if not original or original == correct:
        continue

    pair = (original, correct)
    if pair in seen:
        continue
    seen.add(pair)

    # 前面审计确认所有文本长度相同；这里仍显式检查
    if len(original) != len(correct):
        continue

    diffs = [
        i for i, (a, b) in enumerate(zip(original, correct))
        if a != b
    ]

    if len(diffs) == 1:
        single_samples.append((row, diffs))
    elif len(diffs) > 1:
        multi_samples.append((row, diffs))

rng = random.Random(20261018)

chosen_single = rng.sample(single_samples, min(12, len(single_samples)))
chosen_multi = rng.sample(multi_samples, min(18, len(multi_samples)))

def show_samples(title, samples):
    print(f"\n{'=' * 24} {title} {'=' * 24}")

    for n, (row, diffs) in enumerate(samples, 1):
        original = row["original_text"]
        correct = row["correct_text"]

        print(f"\n[{n}] ID={row.get('id')} | 长度={len(original)} | 差异字符数={len(diffs)}")
        print("原句：", original)
        print("正句：", correct)

        # 展示差异位置附近的字符，方便定位
        details = []
        for i in diffs[:12]:
            left = max(0, i - 3)
            right = min(len(original), i + 4)
            details.append(
                f"位置{i}: 原句[{original[left:right]}] -> 正句[{correct[left:right]}]"
            )

        print("差异上下文：")
        for detail in details:
            print("  ", detail)

show_samples("单字符差异样本", chosen_single)
show_samples("多字符差异样本", chosen_multi)

print("\n===== 抽样规模 =====")
print("唯一单字符差异候选：", len(single_samples))
print("唯一多字符差异候选：", len(multi_samples))


======================== 单字符差异样本 ========================

[1] ID=- | 长度=24 | 差异字符数=1
原句： 个别单位提出在标准中设定重金属锰的迁移线量指标。
正句： 个别单位提出在标准中设定重金属锰的迁移限量指标。
差异上下文：
   位置19: 原句[的迁移线量指标] -> 正句[的迁移限量指标]

[2] ID=- | 长度=21 | 差异字符数=1
原句： 主办单位请望索普能够和胡根班再度同台较量。
正句： 主办单位希望索普能够和胡根班再度同台较量。
差异上下文：
   位置4: 原句[办单位请望索普] -> 正句[办单位希望索普]

[3] ID=- | 长度=35 | 差异字符数=1
原句： 他说：我不能继续目睹人展处于痛苦中，因此我决定离开赖比瑞亚，流亡海外。
正句： 他说：我不能继续目睹人民处于痛苦中，因此我决定离开赖比瑞亚，流亡海外。
差异上下文：
   位置11: 原句[目睹人展处于痛] -> 正句[目睹人民处于痛]

[4] ID=- | 长度=44 | 差异字符数=1
原句： 臼宫在布希今晚九时台北时间二十一日上午九时向国会联席会议发表演说之前，先发布了讲稿摘要。
正句： 白宫在布希今晚九时台北时间二十一日上午九时向国会联席会议发表演说之前，先发布了讲稿摘要。
差异上下文：
   位置0: 原句[臼宫在布] -> 正句[白宫在布]

[5] ID=- | 长度=46 | 差异字符数=1
原句： 当艾尔斯演出他职业生涯几乎是最差的九洞时，全场都怀疑他们将见证高尔天史上最大的一次崩盘演出。
正句： 当艾尔斯演出他职业生涯几乎是最差的九洞时，全场都怀疑他们将见证高尔夫史上最大的一次崩盘演出。
差异上下文：
   位置33: 原句[证高尔天史上最] -> 正句[证高尔夫史上最]

[6] ID=- | 长度=10 | 差异字符数=1
原句： 而且常常会翘家除去玩
正句： 而且常常会翘家出去玩
差异上下文：
   位置7: 原句[会翘家除去玩] -> 正句[会翘家出去玩]

[7] ID=- | 长度=19 | 差异字符数=1
原句： 财政部财科所国有资本是主任陈少强建议。
正句： 财政部财科所国有资本室主任陈少强建议。
差异上下文：
   位置10: 原句[有资本是主任陈] ->

In [10]:
from collections import Counter

assert "ds" in globals(), "请先加载数据集。"

seen = set()
diff_distribution = Counter()

for row in ds["train"]:
    original = row["original_text"]
    correct = row["correct_text"]

    if not original or original == correct:
        continue

    pair = (original, correct)
    if pair in seen:
        continue
    seen.add(pair)

    # 之前已确认长度一致；这里仍然进行防御性检查
    if len(original) != len(correct):
        diff_distribution["length_mismatch"] += 1
        continue

    n_diff = sum(
        a != b for a, b in zip(original, correct)
    )

    if n_diff == 1:
        diff_distribution["1"] += 1
    elif n_diff == 2:
        diff_distribution["2"] += 1
    elif n_diff == 3:
        diff_distribution["3"] += 1
    elif n_diff <= 5:
        diff_distribution["4-5"] += 1
    elif n_diff <= 10:
        diff_distribution["6-10"] += 1
    else:
        diff_distribution["11+"] += 1

print("===== 唯一句子对的字符差异分布 =====")
for category in ["1", "2", "3", "4-5", "6-10", "11+", "length_mismatch"]:
    print(f"{category}: {diff_distribution[category]}")

print("合计：", sum(diff_distribution.values()))

===== 唯一句子对的字符差异分布 =====
1: 167790
2: 57885
3: 18083
4-5: 1911
6-10: 239
11+: 11
length_mismatch: 0
合计： 245919


In [11]:
from pathlib import Path
from collections import Counter
import json

assert "ds" in globals(), "请先加载成功的 Hugging Face 数据集。"

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
OUT = ROOT / "data" / "processed" / "csc_candidates"
OUT.mkdir(parents=True, exist_ok=True)

paths = {
    "single": OUT / "single_char.jsonl",
    "multi_2_3": OUT / "multi_2_3_char.jsonl",
    "review": OUT / "review_4plus_char.jsonl",
}

counts = Counter()
seen = set()

with (
    paths["single"].open("w", encoding="utf-8") as f1,
    paths["multi_2_3"].open("w", encoding="utf-8") as f23,
    paths["review"].open("w", encoding="utf-8") as fr,
):
    for row in ds["train"]:
        original = row["original_text"]
        correct = row["correct_text"]

        if not original or not correct:
            counts["empty_text_skipped"] += 1
            continue

        if len(original) != len(correct):
            counts["length_mismatch_skipped"] += 1
            continue

        if original == correct:
            counts["identical_skipped"] += 1
            continue

        pair = (original, correct)
        if pair in seen:
            counts["duplicate_skipped"] += 1
            continue
        seen.add(pair)

        diff_positions = [
            i for i, (a, b) in enumerate(zip(original, correct))
            if a != b
        ]
        n_diff = len(diff_positions)

        # 保存重新计算的差异位置，不把 wrong_ids 当作标签
        record = {
            "source": "shibing624/CSC",
            "source_split": "train",
            "source_id": row.get("id"),
            "original_text": original,
            "correct_text": correct,
            "diff_count": n_diff,
            "diff_positions": diff_positions,
        }

        line = json.dumps(record, ensure_ascii=False) + "\n"

        if n_diff == 1:
            f1.write(line)
            counts["single_char"] += 1
        elif n_diff <= 3:
            f23.write(line)
            counts["multi_2_3_char"] += 1
        else:
            fr.write(line)
            counts["review_4plus_char"] += 1

print("===== 导出统计 =====")
for key, value in counts.items():
    print(f"{key}: {value}")

print("\n===== 输出文件 =====")
for key, path in paths.items():
    print(f"{key}: {path}")
    print(f"  文件存在：{path.exists()}")
    if path.exists():
        print(f"  文件大小：{path.stat().st_size / 1024 / 1024:.2f} MB")

expected = {
    "single_char": 167790,
    "multi_2_3_char": 75968,
    "review_4plus_char": 2161,
}

print("\n===== 数量核对 =====")
for key, target in expected.items():
    actual = counts[key]
    print(f"{key}: 实际={actual}, 预期={target}, 一致={actual == target}")

assert all(counts[k] == v for k, v in expected.items()), \
    "候选数量与审计结果不一致，请先停止后续处理。"

print("\n候选文件已导出，原有训练文件未被覆盖。")

===== 导出统计 =====
single_char: 167790
multi_2_3_char: 75968
identical_skipped: 2868
review_4plus_char: 2161
duplicate_skipped: 3048

===== 输出文件 =====
single: /content/xime_work/xime_macbert_colab_ready/data/processed/csc_candidates/single_char.jsonl
  文件存在：True
  文件大小：64.95 MB
multi_2_3: /content/xime_work/xime_macbert_colab_ready/data/processed/csc_candidates/multi_2_3_char.jsonl
  文件存在：True
  文件大小：30.41 MB
review: /content/xime_work/xime_macbert_colab_ready/data/processed/csc_candidates/review_4plus_char.jsonl
  文件存在：True
  文件大小：1.10 MB

===== 数量核对 =====
single_char: 实际=167790, 预期=167790, 一致=True
multi_2_3_char: 实际=75968, 预期=75968, 一致=True
review_4plus_char: 实际=2161, 预期=2161, 一致=True

候选文件已导出，原有训练文件未被覆盖。


In [12]:
from pathlib import Path
from collections import Counter
import json

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
OUT = ROOT / "data" / "processed" / "csc_candidates"

FILES = {
    "single": (OUT / "single_char.jsonl", 1, 1),
    "multi_2_3": (OUT / "multi_2_3_char.jsonl", 2, 3),
    "review_4plus": (OUT / "review_4plus_char.jsonl", 4, None),
}

total = 0
global_pairs = set()
errors = Counter()
file_stats = {}

for name, (path, min_diff, max_diff) in FILES.items():
    assert path.exists(), f"文件不存在：{path}"

    count = 0
    diff_counts = Counter()

    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            try:
                row = json.loads(line)
            except Exception:
                errors["invalid_json"] += 1
                print(f"[JSON错误] {name}:{line_no}")
                continue

            original = row.get("original_text")
            correct = row.get("correct_text")
            positions = row.get("diff_positions")
            stored_count = row.get("diff_count")

            if not isinstance(original, str) or not isinstance(correct, str):
                errors["invalid_text"] += 1
                continue

            if len(original) != len(correct):
                errors["length_mismatch"] += 1
                continue

            actual_positions = [
                i for i, (a, b) in enumerate(zip(original, correct))
                if a != b
            ]

            if not isinstance(positions, list) or positions != actual_positions:
                errors["position_mismatch"] += 1

            if stored_count != len(actual_positions):
                errors["diff_count_mismatch"] += 1

            n = len(actual_positions)
            if n < min_diff or (max_diff is not None and n > max_diff):
                errors["wrong_bucket"] += 1

            pair = (original, correct)
            if pair in global_pairs:
                errors["duplicate_across_files"] += 1
            global_pairs.add(pair)

            diff_counts[n] += 1
            count += 1

    file_stats[name] = (count, diff_counts)
    total += count

print("===== 文件回读统计 =====")
for name, (count, diff_counts) in file_stats.items():
    print(f"\n{name}: {count} 条")
    print("差异数量分布：", dict(sorted(diff_counts.items())))

print("\n===== 总体审计 =====")
print("回读总条数：", total)
print("跨文件唯一句对数：", len(global_pairs))
print("错误计数：", dict(errors))
print("审计通过：", total == 245919 and not errors)

===== 文件回读统计 =====

single: 167790 条
差异数量分布： {1: 167790}

multi_2_3: 75968 条
差异数量分布： {2: 57885, 3: 18083}

review_4plus: 2161 条
差异数量分布： {4: 1453, 5: 458, 6: 140, 7: 49, 8: 28, 9: 14, 10: 8, 11: 3, 12: 4, 13: 1, 14: 3}

===== 总体审计 =====
回读总条数： 245919
跨文件唯一句对数： 245919
错误计数： {}
审计通过： True


In [13]:
from pathlib import Path
import csv

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")

print("===== 查找现有数据划分 =====")

targets = ["train_v1.tsv", "valid_v1.tsv", "test_v1.tsv"]

found = {}
for name in targets:
    matches = list(ROOT.rglob(name))
    if len(matches) == 1:
        found[name] = matches[0]
        print(f"\n✓ {name}")
        print(f"  路径：{matches[0]}")
    elif len(matches) == 0:
        print(f"\n✗ 未找到：{name}")
    else:
        print(f"\n⚠ 找到多个同名文件：{name}")
        for p in matches:
            print(" ", p)

print("\n===== 检查 TSV 格式 =====")

for name, path in found.items():
    print(f"\n--- {name} ---")
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f, delimiter="\t")
        for i, row in enumerate(reader):
            print(f"第 {i + 1} 行：{row[:8]}")
            if i >= 2:
                break

print("\n===== 候选文件 =====")
candidate_dir = ROOT / "data" / "processed" / "csc_candidates"
for p in sorted(candidate_dir.glob("*.jsonl")):
    print(f"{p.name}: {p.stat().st_size / 1024 / 1024:.2f} MB")

===== 查找现有数据划分 =====

✓ train_v1.tsv
  路径：/content/xime_work/xime_macbert_colab_ready/data/processed/train_v1.tsv

✓ valid_v1.tsv
  路径：/content/xime_work/xime_macbert_colab_ready/data/processed/valid_v1.tsv

✓ test_v1.tsv
  路径：/content/xime_work/xime_macbert_colab_ready/data/processed/test_v1.tsv

===== 检查 TSV 格式 =====

--- train_v1.tsv ---
第 1 行：['clean', 'noisy']
第 2 行：['万人', '馓人']
第 3 行：['万人', '午人']

--- valid_v1.tsv ---
第 1 行：['clean', 'noisy']
第 2 行：['不满', '不磡']
第 3 行：['不满', '不篮']

--- test_v1.tsv ---
第 1 行：['clean', 'noisy']
第 2 行：['三个', '莞个']
第 3 行：['三个', '聃个']

===== 候选文件 =====
multi_2_3_char.jsonl: 30.41 MB
review_4plus_char.jsonl: 1.10 MB
single_char.jsonl: 64.95 MB


In [14]:
from pathlib import Path
import csv
import json

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
OLD_DIR = ROOT / "data" / "processed"
CAND_DIR = OLD_DIR / "csc_candidates"

old_paths = {
    "train": OLD_DIR / "train_v1.tsv",
    "valid": OLD_DIR / "valid_v1.tsv",
    "test": OLD_DIR / "test_v1.tsv",
}
cand_paths = {
    "single": CAND_DIR / "single_char.jsonl",
    "multi_2_3": CAND_DIR / "multi_2_3_char.jsonl",
    "review_4plus": CAND_DIR / "review_4plus_char.jsonl",
}

# 先检查所有文件，避免审计跑到一半才发现路径错误
missing = [
    str(p) for p in list(old_paths.values()) + list(cand_paths.values())
    if not p.is_file()
]
if missing:
    print("以下文件不存在，请先检查 Colab 运行时和项目目录：")
    for p in missing:
        print(" -", p)
    raise FileNotFoundError("审计已停止，没有修改任何文件。")

def read_old(path):
    result = []
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        if not {"clean", "noisy"}.issubset(reader.fieldnames or []):
            raise ValueError(f"TSV列名异常：{path}；列名={reader.fieldnames}")
        for row in reader:
            clean, noisy = row["clean"], row["noisy"]
            if clean and noisy:
                result.append((noisy, clean))
    return result

def read_candidates(path):
    result = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            row = json.loads(line)
            result.append((row["original_text"], row["correct_text"]))
    return result

old = {k: read_old(p) for k, p in old_paths.items()}
cand = {k: read_candidates(p) for k, p in cand_paths.items()}

old_pairs = {k: set(v) for k, v in old.items()}
old_clean = {k: {clean for _, clean in v} for k, v in old.items()}
old_noisy = {k: {noisy for noisy, _ in v} for k, v in old.items()}

print("===== 旧数据规模 =====")
for split, pairs in old.items():
    print(f"{split}: 行数={len(pairs):,}，唯一句对={len(set(pairs)):,}")

print("\n===== 新候选与旧数据重叠 =====")
for group, pairs in cand.items():
    pair_set = set(pairs)
    clean_set = {clean for _, clean in pairs}
    noisy_set = {noisy for noisy, _ in pairs}

    print(f"\n[{group}] 候选行数={len(pairs):,}")
    for split in ("train", "valid", "test"):
        exact = pair_set & old_pairs[split]
        reverse = {
            (clean, noisy) for noisy, clean in pairs
        } & old_pairs[split]
        same_clean = clean_set & old_clean[split]
        same_noisy = noisy_set & old_noisy[split]

        print(
            f"{split}: 同向句对={len(exact):,}，"
            f"反向句对={len(reverse):,}，"
            f"相同clean文本={len(same_clean):,}，"
            f"相同noisy文本={len(same_noisy):,}"
        )

print("\n===== 旧验证集与测试集 =====")
print(
    "clean 文本交集：",
    len(old_clean["valid"] & old_clean["test"])
)

print("\n===== 新候选组之间的句对重复 =====")
groups = list(cand)
for i in range(len(groups)):
    for j in range(i + 1, len(groups)):
        a, b = groups[i], groups[j]
        set_a, set_b = set(cand[a]), set(cand[b])
        reverse_b = {(clean, noisy) for noisy, clean in cand[b]}
        print(
            f"{a} vs {b}: "
            f"同向重复={len(set_a & set_b):,}，"
            f"反向重复={len(set_a & reverse_b):,}"
        )

print("\n只读审计完成；没有修改任何文件。")

===== 旧数据规模 =====
train: 行数=4,691，唯一句对=4,691
valid: 行数=583，唯一句对=583
test: 行数=650，唯一句对=650

===== 新候选与旧数据重叠 =====

[single] 候选行数=167,790
train: 同向句对=0，反向句对=0，相同clean文本=0，相同noisy文本=0
valid: 同向句对=0，反向句对=0，相同clean文本=0，相同noisy文本=0
test: 同向句对=0，反向句对=0，相同clean文本=0，相同noisy文本=0

[multi_2_3] 候选行数=75,968
train: 同向句对=0，反向句对=0，相同clean文本=0，相同noisy文本=0
valid: 同向句对=0，反向句对=0，相同clean文本=0，相同noisy文本=0
test: 同向句对=0，反向句对=0，相同clean文本=0，相同noisy文本=0

[review_4plus] 候选行数=2,161
train: 同向句对=0，反向句对=0，相同clean文本=0，相同noisy文本=0
valid: 同向句对=0，反向句对=0，相同clean文本=0，相同noisy文本=0
test: 同向句对=0，反向句对=0，相同clean文本=0，相同noisy文本=0

===== 旧验证集与测试集 =====
clean 文本交集： 0

===== 新候选组之间的句对重复 =====
single vs multi_2_3: 同向重复=0，反向重复=0
single vs review_4plus: 同向重复=0，反向重复=0
multi_2_3 vs review_4plus: 同向重复=0，反向重复=0

只读审计完成；没有修改任何文件。


In [15]:
from pathlib import Path
import json
import random
from collections import Counter

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
CAND_DIR = ROOT / "data" / "processed" / "csc_candidates"
rng = random.Random(20261018)

groups = {
    "单字符差异": (CAND_DIR / "single_char.jsonl", 30),
    "2–3 字符差异": (CAND_DIR / "multi_2_3_char.jsonl", 30),
    "4 个及以上差异": (CAND_DIR / "review_4plus_char.jsonl", 15),
}

for group_name, (path, sample_size) in groups.items():
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))

    samples = rng.sample(rows, min(sample_size, len(rows)))

    print(f"\n{'=' * 24} {group_name} {'=' * 24}")
    print(f"文件总数：{len(rows):,}；本次抽样：{len(samples)}")

    for i, row in enumerate(samples, 1):
        original = row["original_text"]
        correct = row["correct_text"]
        positions = row["diff_positions"]

        print(f"\n{i}. 原句：{original}")
        print(f"   正句：{correct}")
        print(f"   差异数：{row['diff_count']}；位置：{positions}")

        # 显示每个差异位置的字符变化
        changes = [
            f"{original[p]}→{correct[p]}"
            for p in positions
        ]
        print("   字符变化：" + "，".join(changes))

print("\n抽样完成：仅读取文件，没有修改数据。")


======================== 单字符差异 ========================
文件总数：167,790；本次抽样：30

1. 原句：个别单位提出在标准中设定重金属锰的迁移线量指标。
   正句：个别单位提出在标准中设定重金属锰的迁移限量指标。
   差异数：1；位置：[19]
   字符变化：线→限

2. 原句：主办单位请望索普能够和胡根班再度同台较量。
   正句：主办单位希望索普能够和胡根班再度同台较量。
   差异数：1；位置：[4]
   字符变化：请→希

3. 原句：他说：我不能继续目睹人展处于痛苦中，因此我决定离开赖比瑞亚，流亡海外。
   正句：他说：我不能继续目睹人民处于痛苦中，因此我决定离开赖比瑞亚，流亡海外。
   差异数：1；位置：[11]
   字符变化：展→民

4. 原句：臼宫在布希今晚九时台北时间二十一日上午九时向国会联席会议发表演说之前，先发布了讲稿摘要。
   正句：白宫在布希今晚九时台北时间二十一日上午九时向国会联席会议发表演说之前，先发布了讲稿摘要。
   差异数：1；位置：[0]
   字符变化：臼→白

5. 原句：当艾尔斯演出他职业生涯几乎是最差的九洞时，全场都怀疑他们将见证高尔天史上最大的一次崩盘演出。
   正句：当艾尔斯演出他职业生涯几乎是最差的九洞时，全场都怀疑他们将见证高尔夫史上最大的一次崩盘演出。
   差异数：1；位置：[33]
   字符变化：天→夫

6. 原句：而且常常会翘家除去玩
   正句：而且常常会翘家出去玩
   差异数：1；位置：[7]
   字符变化：除→出

7. 原句：财政部财科所国有资本是主任陈少强建议。
   正句：财政部财科所国有资本室主任陈少强建议。
   差异数：1；位置：[10]
   字符变化：是→室

8. 原句：奔跑中国系列劲跑赛事将转战广州。
   正句：奔跑中国系列竞跑赛事将转战广州。
   差异数：1；位置：[6]
   字符变化：劲→竞

9. 原句：卜波天曾两度在奥运的五十公尺和一百公尺自由式比赛摘金，这是他在这次世泳赛拿到的第二面金牌。
   正句：卜波夫曾两度在奥运的五十公尺和一百公尺自由式比赛摘金，这是他在这次世泳赛拿到的第二面金牌。
   差异数：1；位置：[2]
   字符变化：天→夫

10. 原句：七国渠

In [16]:
%pip install opencc-python-reimplemented

In [17]:
from opencc import OpenCC

cc = OpenCC("t2s")

tests = [
    "繁體中文",
    "網際網路",
    "計畫",
]

for text in tests:
    print(f"{text} -> {cc.convert(text)}")

繁體中文 -> 繁体中文
網際網路 -> 网际网路
計畫 -> 计划


In [18]:
from pathlib import Path
from collections import Counter
import json
from opencc import OpenCC

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
SRC = ROOT / "data" / "processed" / "csc_candidates"
DST = ROOT / "data" / "processed" / "csc_candidates_simplified"
DST.mkdir(parents=True, exist_ok=True)

cc = OpenCC("t2s")

sources = [
    SRC / "single_char.jsonl",
    SRC / "multi_2_3_char.jsonl",
    SRC / "review_4plus_char.jsonl",
]

outputs = {
    "single": DST / "single_char.jsonl",
    "multi_2_3": DST / "multi_2_3_char.jsonl",
    "review_4plus": DST / "review_4plus_char.jsonl",
}

stats = Counter()
seen = set()

with (
    outputs["single"].open("w", encoding="utf-8") as f1,
    outputs["multi_2_3"].open("w", encoding="utf-8") as f23,
    outputs["review_4plus"].open("w", encoding="utf-8") as fr,
):
    writers = {
        "single": f1,
        "multi_2_3": f23,
        "review_4plus": fr,
    }

    for src in sources:
        with src.open("r", encoding="utf-8") as f:
            for line in f:
                row = json.loads(line)

                original = cc.convert(row["original_text"])
                correct = cc.convert(row["correct_text"])
                stats["input_rows"] += 1

                if len(original) != len(correct):
                    stats["length_mismatch_after_conversion"] += 1
                    continue

                positions = [
                    i for i, (a, b) in enumerate(zip(original, correct))
                    if a != b
                ]

                if not positions:
                    stats["no_diff_after_conversion"] += 1
                    continue

                pair = (original, correct)
                if pair in seen:
                    stats["duplicate_after_conversion"] += 1
                    continue
                seen.add(pair)

                n = len(positions)
                new_row = {
                    **row,
                    "original_text": original,
                    "correct_text": correct,
                    "diff_count": n,
                    "diff_positions": positions,
                    "normalization": "OpenCC t2s",
                }

                if n == 1:
                    bucket = "single"
                elif n <= 3:
                    bucket = "multi_2_3"
                else:
                    bucket = "review_4plus"

                writers[bucket].write(
                    json.dumps(new_row, ensure_ascii=False) + "\n"
                )
                stats[bucket] += 1

print("===== 简体规范化统计 =====")
for key, value in stats.items():
    print(f"{key}: {value}")

print("\n===== 输出文件 =====")
for key, path in outputs.items():
    print(f"{key}: {path}；存在={path.exists()}")
    if path.exists():
        print(f"  大小：{path.stat().st_size / 1024 / 1024:.2f} MB")

print("\n原始候选文件未修改。")

===== 简体规范化统计 =====
input_rows: 245919
single: 167841
no_diff_after_conversion: 83
multi_2_3: 75834
review_4plus: 2160
duplicate_after_conversion: 1

===== 输出文件 =====
single: /content/xime_work/xime_macbert_colab_ready/data/processed/csc_candidates_simplified/single_char.jsonl；存在=True
  大小：69.93 MB
multi_2_3: /content/xime_work/xime_macbert_colab_ready/data/processed/csc_candidates_simplified/multi_2_3_char.jsonl；存在=True
  大小：32.59 MB
review_4plus: /content/xime_work/xime_macbert_colab_ready/data/processed/csc_candidates_simplified/review_4plus_char.jsonl；存在=True
  大小：1.16 MB

原始候选文件未修改。


In [19]:
from pathlib import Path
from collections import Counter
import json

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
DATA = ROOT / "data" / "processed" / "csc_candidates_simplified"

FILES = {
    "single": DATA / "single_char.jsonl",
    "multi_2_3": DATA / "multi_2_3_char.jsonl",
    "review_4plus": DATA / "review_4plus_char.jsonl",
}

stats = Counter()
errors = []
seen_pairs = set()
bucket_counts = Counter()
sample_rows = []

for bucket, path in FILES.items():
    if not path.exists():
        errors.append(f"文件不存在：{path}")
        continue

    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            try:
                row = json.loads(line)
            except Exception as e:
                errors.append(f"{path.name} 第 {line_no} 行 JSON 错误：{e}")
                continue

            stats["total_rows"] += 1
            stats[f"{bucket}_rows"] += 1

            original = row.get("original_text", "")
            correct = row.get("correct_text", "")
            positions = row.get("diff_positions")
            diff_count = row.get("diff_count")

            if not isinstance(original, str) or not isinstance(correct, str):
                errors.append(f"{path.name}:{line_no} 文本字段不是字符串")
                continue

            if len(original) != len(correct):
                stats["length_mismatch"] += 1
                errors.append(f"{path.name}:{line_no} 两句长度不一致")
                continue

            actual_positions = [
                i for i, (a, b) in enumerate(zip(original, correct))
                if a != b
            ]

            if not actual_positions:
                stats["no_diff"] += 1
                errors.append(f"{path.name}:{line_no} 两句没有差异")
                continue

            if positions != actual_positions:
                stats["position_mismatch"] += 1
                errors.append(f"{path.name}:{line_no} diff_positions 不正确")

            if diff_count != len(actual_positions):
                stats["diff_count_mismatch"] += 1
                errors.append(f"{path.name}:{line_no} diff_count 不正确")

            if row.get("normalization") != "OpenCC t2s":
                errors.append(f"{path.name}:{line_no} 缺少预期的规范化标记")

            n = len(actual_positions)
            expected_bucket = (
                "single" if n == 1
                else "multi_2_3" if n <= 3
                else "review_4plus"
            )

            if expected_bucket != bucket:
                stats["wrong_bucket"] += 1
                errors.append(
                    f"{path.name}:{line_no} 分桶错误："
                    f"实际差异数={n}，文件分桶={bucket}"
                )

            pair = (original, correct)
            if pair in seen_pairs:
                stats["duplicate_pairs"] += 1
                errors.append(f"{path.name}:{line_no} 出现重复文本对")
            seen_pairs.add(pair)

            bucket_counts[expected_bucket] += 1

            if len(sample_rows) < 12:
                sample_rows.append({
                    "bucket": bucket,
                    "original": original,
                    "correct": correct,
                    "diff_count": n,
                    "diff_positions": actual_positions,
                })

print("===== 完整性审计统计 =====")
for key, value in sorted(stats.items()):
    print(f"{key}: {value}")

print("\n===== 分桶核对 =====")
for bucket in FILES:
    print(
        f"{bucket}: 文件行数={stats[f'{bucket}_rows']}，"
        f"按实际差异数计算={bucket_counts[bucket]}"
    )

print("\n===== 审计错误 =====")
if errors:
    for error in errors[:30]:
        print("✗", error)
    if len(errors) > 30:
        print(f"...另有 {len(errors) - 30} 条错误")
else:
    print("✓ 未发现错误")

print("\n===== 总结 =====")
print("文件总行数：", stats["total_rows"])
print("唯一文本对：", len(seen_pairs))
print("审计通过：", len(errors) == 0)

print("\n===== 随机性不足的前 12 条示例（仅供初步查看） =====")
for i, row in enumerate(sample_rows, 1):
    print(
        f"{i}. [{row['bucket']}] "
        f"{row['original']} -> {row['correct']} "
        f"(差异数={row['diff_count']}，位置={row['diff_positions']})"
    )

===== 完整性审计统计 =====
multi_2_3_rows: 75834
review_4plus_rows: 2160
single_rows: 167841
total_rows: 245835

===== 分桶核对 =====
single: 文件行数=167841，按实际差异数计算=167841
multi_2_3: 文件行数=75834，按实际差异数计算=75834
review_4plus: 文件行数=2160，按实际差异数计算=2160

===== 审计错误 =====
✓ 未发现错误

===== 总结 =====
文件总行数： 245835
唯一文本对： 245835
审计通过： True

===== 随机性不足的前 12 条示例（仅供初步查看） =====
1. [single] 目前区次事件的细节还不清楚，伤亡人数也未确定。 -> 目前这次事件的细节还不清楚，伤亡人数也未确定。 (差异数=1，位置=[2])
2. [single] 报导中并未说明出口量，但据引述药厂主管的话指出，每一种药物最大的庄度出口量都达到十二吨之谱。 -> 报导中并未说明出口量，但据引述药厂主管的话指出，每一种药物最大的年度出口量都达到十二吨之谱。 (差异数=1，位置=[32])
3. [single] 丈夫拒不承认家暴庭审一结束就打期子。 -> 丈夫拒不承认家暴庭审一结束就打妻子。 (差异数=1，位置=[15])
4. [single] 报导并末说明事故发生的原因。 -> 报导并未说明事故发生的原因。 (差异数=1，位置=[3])
5. [single] 美国贸易代表左里克和农业部长威尼晏发表联合声明说：我们很高兴中国似乎已察觉到我们的关切。 -> 美国贸易代表左里克和农业部长威尼曼发表联合声明说：我们很高兴中国似乎已察觉到我们的关切。 (差异数=1，位置=[16])
6. [single] 根据这项调查，至三月三十一日止，日本的人口为一亿二千六百二十八万四干八百零五人，比一年前增加二十一万三千五百人，增幅为百分之零点一六九。 -> 根据这项调查，至三月三十一日止，日本的人口为一亿二千六百二十八万四千八百零五人，比一年前增加二十一万三千五百人，增幅为百分之零点一六九。 (差异数=1，位置=[33])
7. [single] 香港新闻统萼专员林瑞麟十四日在

In [20]:
from pathlib import Path
from collections import Counter
import csv
import json

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
CANDIDATE_DIR = ROOT / "data" / "processed" / "csc_candidates_simplified"
OLD_DIR = ROOT / "data" / "processed"

CANDIDATE_FILES = [
    CANDIDATE_DIR / "single_char.jsonl",
    CANDIDATE_DIR / "multi_2_3_char.jsonl",
    CANDIDATE_DIR / "review_4plus_char.jsonl",
]

OLD_FILES = {
    "train": OLD_DIR / "train_v1.tsv",
    "valid": OLD_DIR / "valid_v1.tsv",
    "test": OLD_DIR / "test_v1.tsv",
}

# 读取新候选语料；不修改任何文件
candidate_pairs = set()
candidate_clean = set()
candidate_noisy = set()
candidate_rows = 0

for path in CANDIDATE_FILES:
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            noisy = row["original_text"]
            clean = row["correct_text"]

            candidate_rows += 1
            candidate_pairs.add((noisy, clean))
            candidate_noisy.add(noisy)
            candidate_clean.add(clean)

# 读取旧数据集
old_data = {}
for split, path in OLD_FILES.items():
    if not path.exists():
        raise FileNotFoundError(
            f"找不到旧数据文件：{path}\n"
            "请检查 train_v1.tsv、valid_v1.tsv、test_v1.tsv 的实际位置。"
        )

    rows = []
    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")

        if not {"clean", "noisy"}.issubset(reader.fieldnames or []):
            raise ValueError(
                f"{path.name} 列名异常：{reader.fieldnames}"
            )

        for row in reader:
            rows.append((row["noisy"], row["clean"]))

    old_data[split] = rows

print("===== 数据规模 =====")
print("新候选文件总行数：", candidate_rows)
print("新候选唯一文本对：", len(candidate_pairs))

print("\n===== 新候选与旧数据集重叠审计 =====")

for split, rows in old_data.items():
    old_pairs = set(rows)
    old_noisy = {noisy for noisy, clean in rows}
    old_clean = {clean for noisy, clean in rows}

    same_direction = candidate_pairs & old_pairs
    reverse_direction = {
        (noisy, clean)
        for noisy, clean in candidate_pairs
        if (clean, noisy) in old_pairs
    }

    # 相同角色的文本重叠
    same_noisy = candidate_noisy & old_noisy
    same_clean = candidate_clean & old_clean

    # 跨角色重叠：新候选输入与旧目标相同，或新目标与旧输入相同
    candidate_noisy_old_clean = candidate_noisy & old_clean
    candidate_clean_old_noisy = candidate_clean & old_noisy

    print(f"\n[{split}]")
    print("旧数据行数：", len(rows))
    print("同方向文本对重叠：", len(same_direction))
    print("反方向文本对重叠：", len(reverse_direction))
    print("相同 noisy 文本：", len(same_noisy))
    print("相同 clean 文本：", len(same_clean))
    print("新 noisy 与旧 clean 相同：", len(candidate_noisy_old_clean))
    print("新 clean 与旧 noisy 相同：", len(candidate_clean_old_noisy))

    if same_direction:
        print("同方向重叠示例：", list(same_direction)[:3])
    if reverse_direction:
        print("反方向重叠示例：", list(reverse_direction)[:3])

print("\n审计完成：以上操作仅读取文件，没有修改原始数据。")

===== 数据规模 =====
新候选文件总行数： 245835
新候选唯一文本对： 245835

===== 新候选与旧数据集重叠审计 =====

[train]
旧数据行数： 4691
同方向文本对重叠： 0
反方向文本对重叠： 0
相同 noisy 文本： 0
相同 clean 文本： 0
新 noisy 与旧 clean 相同： 0
新 clean 与旧 noisy 相同： 0

[valid]
旧数据行数： 583
同方向文本对重叠： 0
反方向文本对重叠： 0
相同 noisy 文本： 0
相同 clean 文本： 0
新 noisy 与旧 clean 相同： 0
新 clean 与旧 noisy 相同： 0

[test]
旧数据行数： 650
同方向文本对重叠： 0
反方向文本对重叠： 0
相同 noisy 文本： 0
相同 clean 文本： 0
新 noisy 与旧 clean 相同： 0
新 clean 与旧 noisy 相同： 0

审计完成：以上操作仅读取文件，没有修改原始数据。


In [21]:
from pathlib import Path
import json
import csv
import random

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
SRC = ROOT / "data" / "processed" / "csc_candidates_simplified"
OUT_DIR = ROOT / "data" / "processed" / "csc_quality_review"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / "review_250.tsv"
rng = random.Random(20261018)

groups = [
    ("single", SRC / "single_char.jsonl", 100),
    ("multi_2_3", SRC / "multi_2_3_char.jsonl", 100),
    ("review_4plus", SRC / "review_4plus_char.jsonl", 50),
]

selected = []

for group_name, path, sample_size in groups:
    rows = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            rows.append(row)

    if len(rows) < sample_size:
        raise ValueError(
            f"{group_name} 只有 {len(rows)} 条，"
            f"不足以抽取 {sample_size} 条"
        )

    samples = rng.sample(rows, sample_size)

    for row in samples:
        selected.append({
            "分组": group_name,
            "来源": row.get("source", ""),
            "原始划分": row.get("source_split", ""),
            "来源ID": row.get("source_id", ""),
            "差异数": row["diff_count"],
            "错误句_original": row["original_text"],
            "目标句_correct": row["correct_text"],
            "判定": "",
            "备注": "",
        })

# 打乱审核顺序，减少审核者受到分组顺序的影响
rng.shuffle(selected)

columns = [
    "分组", "来源", "原始划分", "来源ID", "差异数",
    "错误句_original", "目标句_correct", "判定", "备注"
]

with OUT_FILE.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=columns,
        delimiter="\t",
        extrasaction="ignore",
    )
    writer.writeheader()
    writer.writerows(selected)

print("===== 抽样完成 =====")
print("抽样总数：", len(selected))
print("单字符：", sum(r["分组"] == "single" for r in selected))
print("2–3 字符：", sum(r["分组"] == "multi_2_3" for r in selected))
print("4 字符及以上：", sum(r["分组"] == "review_4plus" for r in selected))
print("审核表：", OUT_FILE)
print("\n判定建议：")
print("正确      = 目标句正确，适合简体中文")
print("错误      = 目标句本身错误或修改不合理")
print("有歧义    = 缺少上下文，无法确认")
print("地区用语  = 目标句主要是繁体地区表达或不符合目标语言习惯")
print("专有名词  = 人名、地名、机构名等需要核实")
print("\n前 10 条预览：")
for i, row in enumerate(selected[:10], 1):
    print(f"\n{i}. [{row['分组']}] 差异数={row['差异数']}")
    print("错误句：", row["错误句_original"])
    print("目标句：", row["目标句_correct"])

===== 抽样完成 =====
抽样总数： 250
单字符： 100
2–3 字符： 100
4 字符及以上： 50
审核表： /content/xime_work/xime_macbert_colab_ready/data/processed/csc_quality_review/review_250.tsv

判定建议：
正确      = 目标句正确，适合简体中文
错误      = 目标句本身错误或修改不合理
有歧义    = 缺少上下文，无法确认
地区用语  = 目标句主要是繁体地区表达或不符合目标语言习惯
专有名词  = 人名、地名、机构名等需要核实

前 10 条预览：

1. [single] 差异数=1
错误句： 数百名法国的抗议群众今天破坏马萨特镇附近种植基因改造玉米的农田，并与一百多名生物科技支持者及一百多名誓察对峙。
目标句： 数百名法国的抗议群众今天破坏马萨特镇附近种植基因改造玉米的农田，并与一百多名生物科技支持者及一百多名警察对峙。

2. [review_4plus] 差异数=4
错误句： 他的第第小舒马贺以一分二十秒七0八排第三，同扁威廉斯车队的盂托亚则以一分二十一秒0七六排石第四位。
目标句： 他的弟第小舒马贺以一分二十秒七0八排第三，同属威廉斯车队的孟托亚则以一分二十一秒0七六排在第四位。

3. [single] 差异数=1
错误句： 委内瑞拉政府坚持，选务当局二月二日举办的公投不莫约束力，查维斯已表示，即使在此项公投落败，他也不会下台。
目标句： 委内瑞拉政府坚持，选务当局二月二日举办的公投不具约束力，查维斯已表示，即使在此项公投落败，他也不会下台。

4. [multi_2_3] 差异数=3
错误句： 辛松说：艾斯特拉答的作法是将这些贿款留存在某处，儅金额达到两亿披索时--我想是在四、五月间，他下令尤莉前往会见担任回教青年将学金基金会秘书的塞拉皮欧律师，并要求塞拉皮欧将钱存入银行。
目标句： 辛松说：艾斯特拉达的作法是将这些贿款留存在某处，当金额达到两亿披索时--我想是在四、五月间，他下令尤莉前往会见担任回教青年奖学金基金会秘书的塞拉皮欧律师，并要求塞拉皮欧将钱存入银行。

5. [review_4plus] 差异数=4
错误句： 华盛顿天主教大学的校长欧康奈尔神父说：他会是一位把忠诚对代教会当作中心教悔的教宗，但与一些人

In [22]:
from pathlib import Path
from collections import Counter
import csv

path = Path(
    "/content/xime_work/xime_macbert_colab_ready/"
    "data/processed/csc_quality_review/review_250.tsv"
)

counts = Counter()
by_group = Counter()
unreviewed = 0
invalid = []

valid_labels = {
    "正确", "错误", "有歧义", "地区用语", "专有名词"
}

with path.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")

    for row in reader:
        group = row["分组"]
        label = row["判定"].strip()

        if not label:
            unreviewed += 1
            continue

        if label not in valid_labels:
            invalid.append((row["来源ID"], label))
            continue

        counts[label] += 1
        by_group[(group, label)] += 1

print("===== 全部审核统计 =====")
print("已审核：", sum(counts.values()))
print("未审核：", unreviewed)
print("无效标签：", len(invalid))

for label in sorted(valid_labels):
    print(f"{label}: {counts[label]}")

print("\n===== 按差异类型统计 =====")
for group in ["single", "multi_2_3", "review_4plus"]:
    print(f"\n[{group}]")
    for label in sorted(valid_labels):
        print(f"{label}: {by_group[(group, label)]}")

if invalid:
    print("\n无效标签示例：", invalid[:10])

===== 全部审核统计 =====
已审核： 0
未审核： 250
无效标签： 0
专有名词: 0
地区用语: 0
有歧义: 0
正确: 0
错误: 0

===== 按差异类型统计 =====

[single]
专有名词: 0
地区用语: 0
有歧义: 0
正确: 0
错误: 0

[multi_2_3]
专有名词: 0
地区用语: 0
有歧义: 0
正确: 0
错误: 0

[review_4plus]
专有名词: 0
地区用语: 0
有歧义: 0
正确: 0
错误: 0


In [23]:
from pathlib import Path
import csv

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
SRC = ROOT / "data/processed/csc_quality_review/review_250.tsv"
OUT = ROOT / "data/processed/csc_quality_review/batches"
OUT.mkdir(parents=True, exist_ok=True)

with SRC.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

columns = list(rows[0].keys())

batch_size = 25

for start in range(0, len(rows), batch_size):
    batch_no = start // batch_size + 1
    batch = rows[start:start + batch_size]
    out_path = OUT / f"review_batch_{batch_no:02d}.tsv"

    with out_path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=columns,
            delimiter="\t",
        )
        writer.writeheader()
        writer.writerows(batch)

    print(f"第 {batch_no:02d} 批：{len(batch)} 条 -> {out_path}")

print(f"\n共生成 {len(rows) // batch_size} 批，每批最多 {batch_size} 条。")

第 01 批：25 条 -> /content/xime_work/xime_macbert_colab_ready/data/processed/csc_quality_review/batches/review_batch_01.tsv
第 02 批：25 条 -> /content/xime_work/xime_macbert_colab_ready/data/processed/csc_quality_review/batches/review_batch_02.tsv
第 03 批：25 条 -> /content/xime_work/xime_macbert_colab_ready/data/processed/csc_quality_review/batches/review_batch_03.tsv
第 04 批：25 条 -> /content/xime_work/xime_macbert_colab_ready/data/processed/csc_quality_review/batches/review_batch_04.tsv
第 05 批：25 条 -> /content/xime_work/xime_macbert_colab_ready/data/processed/csc_quality_review/batches/review_batch_05.tsv
第 06 批：25 条 -> /content/xime_work/xime_macbert_colab_ready/data/processed/csc_quality_review/batches/review_batch_06.tsv
第 07 批：25 条 -> /content/xime_work/xime_macbert_colab_ready/data/processed/csc_quality_review/batches/review_batch_07.tsv
第 08 批：25 条 -> /content/xime_work/xime_macbert_colab_ready/data/processed/csc_quality_review/batches/review_batch_08.tsv
第 09 批：25 条 -> /content/xime_wor

In [24]:
from pathlib import Path
import csv

path = Path(
    "/content/xime_work/xime_macbert_colab_ready/"
    "data/processed/csc_quality_review/batches/review_batch_01.tsv"
)

with path.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

print(f"第 1 批，共 {len(rows)} 条\n")

for i, row in enumerate(rows, 1):
    print(f"===== 第 {i} 条 =====")
    print(f"分组：{row['分组']}；差异数：{row['差异数']}")
    print(f"来源：{row['来源']}；来源划分：{row['原始划分']}；来源 ID：{row['来源ID']}")
    print(f"错误句：{row['错误句_original']}")
    print(f"目标句：{row['目标句_correct']}")
    print()

第 1 批，共 25 条

===== 第 1 条 =====
分组：single；差异数：1
来源：shibing624/CSC；来源划分：train；来源 ID：-
错误句：数百名法国的抗议群众今天破坏马萨特镇附近种植基因改造玉米的农田，并与一百多名生物科技支持者及一百多名誓察对峙。
目标句：数百名法国的抗议群众今天破坏马萨特镇附近种植基因改造玉米的农田，并与一百多名生物科技支持者及一百多名警察对峙。

===== 第 2 条 =====
分组：review_4plus；差异数：4
来源：shibing624/CSC；来源划分：train；来源 ID：-
错误句：他的第第小舒马贺以一分二十秒七0八排第三，同扁威廉斯车队的盂托亚则以一分二十一秒0七六排石第四位。
目标句：他的弟第小舒马贺以一分二十秒七0八排第三，同属威廉斯车队的孟托亚则以一分二十一秒0七六排在第四位。

===== 第 3 条 =====
分组：single；差异数：1
来源：shibing624/CSC；来源划分：train；来源 ID：-
错误句：委内瑞拉政府坚持，选务当局二月二日举办的公投不莫约束力，查维斯已表示，即使在此项公投落败，他也不会下台。
目标句：委内瑞拉政府坚持，选务当局二月二日举办的公投不具约束力，查维斯已表示，即使在此项公投落败，他也不会下台。

===== 第 4 条 =====
分组：multi_2_3；差异数：3
来源：shibing624/CSC；来源划分：train；来源 ID：-
错误句：辛松说：艾斯特拉答的作法是将这些贿款留存在某处，儅金额达到两亿披索时--我想是在四、五月间，他下令尤莉前往会见担任回教青年将学金基金会秘书的塞拉皮欧律师，并要求塞拉皮欧将钱存入银行。
目标句：辛松说：艾斯特拉达的作法是将这些贿款留存在某处，当金额达到两亿披索时--我想是在四、五月间，他下令尤莉前往会见担任回教青年奖学金基金会秘书的塞拉皮欧律师，并要求塞拉皮欧将钱存入银行。

===== 第 5 条 =====
分组：review_4plus；差异数：4
来源：shibing624/CSC；来源划分：train；来源 ID：-
错误句：华盛顿天主教大学的校长欧康奈尔神父说：他会是一位把忠诚对代教会当作中心教悔的教宗，但与一些人想法相反的是，我不认为他会灌喻人们这种关念。
目标句：华

In [25]:
from pathlib import Path
import csv

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
BASE = ROOT / "data/processed/csc_quality_review/batches"

SRC = BASE / "review_batch_01.tsv"
OUT = BASE / "review_batch_01_suggested.tsv"

# 顺序对应第 1 批的 25 条
labels = [
    "正确", "有歧义", "正确", "专有名词", "有歧义",
    "正确", "正确", "有歧义", "正确", "地区用语",
    "正确", "正确", "专有名词", "正确", "专有名词",
    "正确", "有歧义", "专有名词", "地区用语", "正确",
    "专有名词", "有歧义", "正确", "正确", "正确",
]

reasons = [
    "誓察改为警察，明显错字修正",
    "多处修正合理，但弟第小舒马贺仍不通顺",
    "不莫约束力改为不具约束力",
    "涉及人名及基金会名称，建议核实",
    "多处修正合理，但中心教诲表达不自然",
    "备地改为各地",
    "狗、长长、棉质等修正符合语境",
    "巨有改为互有合理，但整句表达生硬",
    "茌改为在，并目改为并且",
    "涉及台湾地名、机构及交通用语",
    "情爱改为亲爱，书信称呼合理",
    "涠田改为油田，原涠改为原油",
    "涉及足球人物译名，建议核实",
    "营销纪录符合此处表达习惯",
    "涉及地名、国名，建议核实",
    "三十曰茌改为三十日在",
    "原句损坏较严重，整体准确性难确认",
    "涉及人名译名，建议核实",
    "有地区用语差异，建议单独评估",
    "主要错字修正合理，仍有地区用语",
    "涉及戴高乐机场等专名，建议核实",
    "部分修正合理，但目标句语义仍不够自然",
    "开理改为开离，电砚改为电视",
    "小是改为小时，萨牲改为牺牲",
    "玫击改为攻击，明显错字修正",
]

with SRC.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

if len(rows) != 25:
    raise ValueError(f"预期 25 条，实际读取到 {len(rows)} 条")

for row, label, reason in zip(rows, labels, reasons):
    row["判定"] = label
    row["备注"] = reason

columns = list(rows[0].keys())

with OUT.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=columns,
        delimiter="\t",
    )
    writer.writeheader()
    writer.writerows(rows)

from collections import Counter
counts = Counter(labels)

print("建议审核结果已保存：", OUT)
print("\n===== 第 1 批建议标签统计 =====")
for label, count in counts.most_common():
    print(f"{label}: {count}")

print("\n注意：这些是初步建议，尚未进行外部专名核实。")
print("原始审核表和语料文件均未修改。")

建议审核结果已保存： /content/xime_work/xime_macbert_colab_ready/data/processed/csc_quality_review/batches/review_batch_01_suggested.tsv

===== 第 1 批建议标签统计 =====
正确: 13
有歧义: 5
专有名词: 5
地区用语: 2

注意：这些是初步建议，尚未进行外部专名核实。
原始审核表和语料文件均未修改。


In [26]:
from pathlib import Path
import csv

path = Path(
    "/content/xime_work/xime_macbert_colab_ready/"
    "data/processed/csc_quality_review/batches/review_batch_02.tsv"
)

with path.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

print(f"第 2 批，共 {len(rows)} 条\n")

for i, row in enumerate(rows, 1):
    print(f"===== 第 {i} 条 =====")
    print(f"分组：{row['分组']}；差异数：{row['差异数']}")
    print(f"来源 ID：{row['来源ID']}")
    print(f"错误句：{row['错误句_original']}")
    print(f"目标句：{row['目标句_correct']}")
    print()

第 2 批，共 25 条

===== 第 1 条 =====
分组：single；差异数：1
来源 ID：-
错误句：而这个紧箍咒和护身服是否真正管用。
目标句：而这个紧箍咒和护身符是否真正管用。

===== 第 2 条 =====
分组：single；差异数：1
来源 ID：-
错误句：克莉丝特丝全年赢得的奚金总额如今已超过四百万美元，也是女子网坛第一人。
目标句：克莉丝特丝全年赢得的奖金总额如今已超过四百万美元，也是女子网坛第一人。

===== 第 3 条 =====
分组：single；差异数：1
来源 ID：-
错误句：波兰是个天主教国家，教会人员纷纷利用主日弥撒的机会提醒信徒，别忘了尽公民的意务。
目标句：波兰是个天主教国家，教会人员纷纷利用主日弥撒的机会提醒信徒，别忘了尽公民的义务。

===== 第 4 条 =====
分组：review_4plus；差异数：4
来源 ID：-
错误句：普罗金也表示，这总疫苗耍获批准上市至少是要五年事间。
目标句：普罗金也表示，这种疫苗要获批准上市至少需要五年时间。

===== 第 5 条 =====
分组：single；差异数：1
来源 ID：-
错误句：伊拉克和中东和平进程的黯淡消息笼罩着联合国大会的第二天议程。
目标句：伊拉克和中东和平进程的黯淡消息笼罩着联合国大会的笫二天议程。

===== 第 6 条 =====
分组：multi_2_3；差异数：3
来源 ID：-
错误句：莱斯说：我们相信吹洲联盟的谈判正导向正雀的方象，因为他们目前所做的是使伊朗面对一个抉择，即它是否准备给予国际社会对于伊朗活动所需的信任。
目标句：莱斯说：我们相信欧洲联盟的谈判正导向正确的方向，因为他们目前所做的是使伊朗面对一个抉择，即它是否准备给予国际社会对于伊朗活动所需的信任。

===== 第 7 条 =====
分组：multi_2_3；差异数：2
来源 ID：-
错误句：有一与推广和普及这项运动。
目标句：有益于推广和普及这项运动。

===== 第 8 条 =====
分组：multi_2_3；差异数：2
来源 ID：-
错误句：犹太垦民与巴勒斯坦人在约旦河西岸的那不鲁斯附近爆发数小时激烈鎗战后，以色列总理巴瑞克指控巴勒斯坦人严重伟反十七日签订的夏姆锡克协定。
目标句：犹太垦民与巴勒斯坦人在约旦河西岸的那不鲁斯附近爆发

In [27]:
from pathlib import Path
import csv
from collections import Counter

root = Path("/content/xime_work/xime_macbert_colab_ready")
source = root / "data/processed/csc_quality_review/batches/review_batch_02.tsv"
output = root / "data/processed/csc_quality_review/batches/review_batch_02_suggested.tsv"

labels = [
    "正确", "专有名词", "正确", "正确", "有歧义",
    "正确", "正确", "专有名词", "正确", "地区用语",
    "正确", "正确", "专有名词", "正确", "正确",
    "正确", "正确", "正确", "正确", "有歧义",
    "有歧义", "有歧义", "正确", "正确", "正确",
]

reasons = [
    "护身服改为护身符，明显错字修正。",
    "奖金修正合理；克莉丝特丝涉及人物译名，建议核实。",
    "意务改为义务，搭配自然。",
    "多处明显错字得到修正，目标句整体合理。",
    "目标句将原本正确的第二天改为笫二天，存在目标标签错误风险。",
    "欧洲、正确、方向等修正合理。",
    "有一与改为有益于，符合语义。",
    "鎗战、伟反的修正合理；巴瑞克、夏姆锡克等译名需核实。",
    "组织、问题、时间、已经等修正合理。",
    "做为属于地区用语差异；若统一大陆规范，可另行考虑规范化。",
    "茌改为在，明显错字修正。",
    "痘家改为度假，符合旅游语境。",
    "阿望改为阿里合理；击技运动奖名称仍需核实。",
    "多处明显错误得到修正，目标句整体合理。",
    "外交宫改为外交官，语义明确。",
    "英国、二零零零年等修正合理。",
    "敢兴趣改为感兴趣，明显错字修正。",
    "朐钞改为伪钞，符合纸币语境。",
    "转账改为转涨，符合经济指标语境。",
    "目标句仍有祸首对象等不自然表达，建议复核完整原文。",
    "外交政策、确信修正合理，但布希两位专家的表述关系不协调。",
    "曰改为日合理；上升18的单位或百分号可能缺失，需核对原文。",
    "咸为改为成为，符合语境。",
    "希望、土耳其等修正合理。",
    "并末改为并未，明显错字修正。",
]

assert len(labels) == len(reasons) == 25

with source.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

assert len(rows) == 25, f"预期 25 条，实际 {len(rows)} 条"

for row, label, reason in zip(rows, labels, reasons):
    row["判定"] = label
    row["备注"] = reason

with output.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=list(rows[0].keys()),
        delimiter="\t",
        lineterminator="\n",
    )
    writer.writeheader()
    writer.writerows(rows)

counts = Counter(row["判定"] for row in rows)

print("建议审核结果已保存：", output)
print("\n===== 第 2 批建议标签统计 =====")
for label in ["正确", "有歧义", "专有名词", "地区用语"]:
    print(f"{label}: {counts[label]}")

print("\n原始审核表和语料文件均未修改。")
print("注意：以上仍是初步建议，专有名词尚未外部核实。")

建议审核结果已保存： /content/xime_work/xime_macbert_colab_ready/data/processed/csc_quality_review/batches/review_batch_02_suggested.tsv

===== 第 2 批建议标签统计 =====
正确: 17
有歧义: 4
专有名词: 3
地区用语: 1

原始审核表和语料文件均未修改。
注意：以上仍是初步建议，专有名词尚未外部核实。


In [28]:
from pathlib import Path
import csv

path = Path(
    "/content/xime_work/xime_macbert_colab_ready/"
    "data/processed/csc_quality_review/batches/review_batch_03.tsv"
)

with path.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

print(f"第 3 批，共 {len(rows)} 条\n")

for i, row in enumerate(rows, 1):
    print(f"===== 第 {i} 条 =====")
    print(f"分组：{row['分组']}；差异数：{row['差异数']}")
    print(f"来源 ID：{row['来源ID']}")
    print(f"错误句：{row['错误句_original']}")
    print(f"目标句：{row['目标句_correct']}")
    print()

第 3 批，共 25 条

===== 第 1 条 =====
分组：review_4plus；差异数：4
来源 ID：-
错误句：马其顿脱离南斯拉天独立久后，与安卡拉当局维持想当亲密的关西。
目标句：马其顿脱离南斯拉夫独立之后，与安卡拉当局维持相当亲密的关系。

===== 第 2 条 =====
分组：single；差异数：1
来源 ID：-
错误句：国际传真社报导，俄罗斯总统蒲亭在总统大选即将于十四日投票前夕披露新内阁阵容，宣布长期担任外交部长的伊戈尔伊凡诺夫已由俄罗斯驻联台国大使拉夫罗夫取而代之。
目标句：国际传真社报导，俄罗斯总统蒲亭在总统大选即将于十四日投票前夕披露新内阁阵容，宣布长期担任外交部长的伊戈尔伊凡诺夫已由俄罗斯驻联合国大使拉夫罗夫取而代之。

===== 第 3 条 =====
分组：multi_2_3；差异数：3
来源 ID：-
错误句：花莲这件自杀乌龙案件发生于花莲市中美路和俯前路口，四十二岁男子后先皓上午九时三十分及乘车号-047的机车行经路口时被警方拦检盘查，经查对失窃车辆资料后，发现这辆机车是今年十二月十日在花莲市国连路失窃的赃车，警方随即将后先皓带回警局侦讯。
目标句：花莲这件自杀乌龙案件发生于花莲市中美路和府前路口，四十二岁男子后先皓上午九时三十分骑乘车号-047的机车行经路口时被警方拦检盘查，经查对失窃车辆资料后，发现这辆机车是今年十二月十日在花莲市国联路失窃的赃车，警方随即将后先皓带回警局侦讯。

===== 第 4 条 =====
分组：single；差异数：1
来源 ID：-
错误句：决定忡的第三盘，摩纳哥在五比二领先之际拿下关键性的破发，最后以六比三获胜。
目标句：决定性的第三盘，摩纳哥在五比二领先之际拿下关键性的破发，最后以六比三获胜。

===== 第 5 条 =====
分组：single；差异数：1
来源 ID：-
错误句：中国大陆已经成为全球圣诞玩具和装施商品的最大生产国，而且就算不是最大国、也会是最大国之一。
目标句：中国大陆已经成为全球圣诞玩具和装饰商品的最大生产国，而且就算不是最大国、也会是最大国之一。

===== 第 6 条 =====
分组：multi_2_3；差异数：2
来源 ID：-
错误句：伦敦市政官员说，一名男子挥动铁棒，豉下前英国首相佘契尔夫人朔像的头。
目标句：伦敦市政官员说，一名男子

In [29]:
from pathlib import Path
import csv
from collections import Counter

root = Path("/content/xime_work/xime_macbert_colab_ready")
source = root / "data/processed/csc_quality_review/batches/review_batch_03.tsv"
output = root / "data/processed/csc_quality_review/batches/review_batch_03_suggested.tsv"

labels = [
    "正确", "正确", "专有名词", "正确", "正确",
    "专有名词", "正确", "正确", "正确", "有歧义",
    "正确", "有歧义", "正确", "专有名词", "正确",
    "有歧义", "正确", "正确", "有歧义", "正确",
    "有歧义", "正确", "正确", "正确", "正确",
]

reasons = [
    "南斯拉天、久后、想当、关西等明显错字得到修正。",
    "联台国改为联合国，明显错字修正。",
    "府前路、国联路及人物姓名涉及专名，需核实。",
    "决定忡改为决定性的，符合语法和语境。",
    "装施改为装饰，明显错字修正。",
    "豉下、朔像得到修正；佘契尔涉及历史人物译名。",
    "饮食习愤改为饮食习惯，搭配自然。",
    "人庚币改为人民币，明显错字修正。",
    "教意、墓本法得到修正；原句仍有标点和表达问题。",
    "首度夺冠的表述需要结合球队历史核对。",
    "这顶计画改为这项计画，量词搭配更合理。",
    "认到关键字草莓飞飞的具体语境不明，建议复核。",
    "一昧、贪涓无厌、尽界得到合理修正。",
    "其他的修正合理；峇里岛涉及地区译名差异。",
    "黄国、正雀改为美国、正确，修正合理。",
    "多处错字得到修正，但烟子仍可能有问题，需核实。",
    "并末改为并未，明显错字修正。",
    "所面临、恐怖、表示等修正合理。",
    "目标句仍有弟第等异常，且事件叙述需要复核。",
    "获释、被美军的修正符合语境。",
    "很哩改为很累可能合理，但需上下文确认语义。",
    "见义勇为基金组织的表达更完整。",
    "菖接接触改为直接接触，明显错字修正。",
    "撤旦改为撒旦合理；纳粹旧党羽的表达仍略显生硬。",
    "人菅、程受、庄力得到合理修正。",
]

assert len(labels) == 25 and len(reasons) == 25

with source.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

assert len(rows) == 25, f"预期 25 条，实际 {len(rows)} 条"

for row, label, reason in zip(rows, labels, reasons):
    row["判定"] = label
    row["备注"] = reason

with output.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=list(rows[0].keys()),
        delimiter="\t",
        lineterminator="\n",
    )
    writer.writeheader()
    writer.writerows(rows)

counts = Counter(row["判定"] for row in rows)

print("建议审核结果已保存：", output)
print("\n===== 第 3 批建议标签统计 =====")
for label in ["正确", "有歧义", "专有名词", "地区用语"]:
    print(f"{label}: {counts[label]}")

print("\n原始审核表和语料文件均未修改。")
print("注意：这些仍是初步建议，不代表已经完成外部核实。")

建议审核结果已保存： /content/xime_work/xime_macbert_colab_ready/data/processed/csc_quality_review/batches/review_batch_03_suggested.tsv

===== 第 3 批建议标签统计 =====
正确: 17
有歧义: 5
专有名词: 3
地区用语: 0

原始审核表和语料文件均未修改。
注意：这些仍是初步建议，不代表已经完成外部核实。


In [30]:
from pathlib import Path
import csv

path = Path(
    "/content/xime_work/xime_macbert_colab_ready/"
    "data/processed/csc_quality_review/batches/review_batch_04.tsv"
)

with path.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

print(f"第 4 批，共 {len(rows)} 条\n")

for i, row in enumerate(rows, 1):
    print(f"===== 第 {i} 条 =====")
    print(f"分组：{row['分组']}；差异数：{row['差异数']}")
    print(f"错误句：{row['错误句_original']}")
    print(f"目标句：{row['目标句_correct']}")
    print()

第 4 批，共 25 条

===== 第 1 条 =====
分组：multi_2_3；差异数：3
错误句：政府因该负担其他对他们的照护的责任。
目标句：政府应该负担起他对他们的照顾的责任。

===== 第 2 条 =====
分组：single；差异数：1
错误句：他发表上述谈话之前，人展协商会议出席的五百九十二名代表，包括军方派系的三十八名代表，投票支持这项建议。
目标句：他发表上述谈话之前，人民协商会议出席的五百九十二名代表，包括军方派系的三十八名代表，投票支持这项建议。

===== 第 3 条 =====
分组：review_4plus；差异数：6
错误句：美国运通喧布，有于九一一恐布玫击事件导至旅游人数锐减，该公词将以高达三亿五千万美元的代价，再裁员五千五百至六千五百人。
目标句：美国运通宣布，由于九一一恐怖攻击事件导致旅游人数锐减，该公司将以高达三亿五千万美元的代价，再裁员五千五百至六千五百人。

===== 第 4 条 =====
分组：multi_2_3；差异数：2
错误句：联合国官员说，大约三千五百名惊慌失措的阿富汗难民今天越过边养附近，涌入巴基斯坦，这也是自从本月七日贡国开始轰炸阿富汗以来最大的逃亡潮。
目标句：联合国官员说，大约三千五百名惊慌失措的阿富汗难民今天越过边界附近，涌入巴基斯坦，这也是自从本月七日美国开始轰炸阿富汗以来最大的逃亡潮。

===== 第 5 条 =====
分组：multi_2_3；差异数：3
错误句：政府报告，今年第二季经产帐赤字由第一季的一千一百二十五亿美元暴增至创纪录的一干两百九十九亿六干万美元。
目标句：政府报告，今年第二季经常帐赤字由第一季的一千一百二十五亿美元暴增至创纪录的一千两百九十九亿六千万美元。

===== 第 6 条 =====
分组：multi_2_3；差异数：2
错误句：他们在英国医学周刊莿胳针杂志中报导，威而钢如同另外两种药剂般能使肺部获得氧气，并使肺郜动脉的血压降低，但它是口服药而更为方便。
目标句：他们在英国医学周刊刺胳针杂志中报导，威而钢如同另外两种药剂般能使肺部获得氧气，并使肺部动脉的血压降低，但它是口服药而更为方便。

===== 第 7 条 =====
分组：multi_2_3；差异数：3
错误句：首相联盟的足球明星：克威尔、维杜卡即波斯尼斯都名列十大兼钱王之内，今年进帐都照

In [31]:
from pathlib import Path
import csv
from collections import Counter

root = Path("/content/xime_work/xime_macbert_colab_ready")
source = root / "data/processed/csc_quality_review/batches/review_batch_04.tsv"
output = root / "data/processed/csc_quality_review/batches/review_batch_04_suggested.tsv"

labels = [
    "有歧义", "正确", "正确", "有歧义", "正确",
    "专有名词", "正确", "有歧义", "专有名词", "正确",
    "正确", "正确", "正确", "正确", "正确",
    "正确", "正确", "正确", "正确", "有歧义",
    "正确", "正确", "正确", "正确", "正确",
]

reasons = [
    "因该改为应该合理，但其他变成他，主语和语义发生变化。",
    "人展协商会议改为人民协商会议，修正合理。",
    "多处明显错字得到修正，目标句整体合理。",
    "贡国改为美国符合语境，但越过边界附近仍不自然。",
    "经产帐及中文数字表达得到合理修正。",
    "涉及医学期刊和药品名称；肺部动脉的表述也建议复核。",
    "及、赚钱王、超过等修正合理。",
    "末改为未正确，但整句存在语义重复。",
    "涉及人物译名及电影名称，建议核实专名。",
    "练、得的修正符合语法结构。",
    "两间事改为两件事，量词修正合理。",
    "各地、头牲口的修正符合语境。",
    "一亩改为一直、不需耍改为不需要，修正合理。",
    "多处明显错字得到修正，目标句整体合理。",
    "多处明显错字得到修正，句意基本连贯。",
    "美国、清楚、觉得、精神等修正合理。",
    "事早、全世界等修正合理。",
    "男乒符合乒乓球选拔赛语境。",
    "爆竹符合节庆安全新闻语境。",
    "目标句仍有这种名小说家等不自然表达，建议复核。",
    "多为可能符合原意，但建议结合上下文确认。",
    "曰改为日，明显字符错误修正。",
    "茼定改为肯定，符合语境。",
    "预算金额、五千万等修正合理。",
    "打工、领不到等修正符合语境。",
]

assert len(labels) == len(reasons) == 25

with source.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

assert len(rows) == 25, f"预期 25 条，实际 {len(rows)} 条"

for row, label, reason in zip(rows, labels, reasons):
    row["判定"] = label
    row["备注"] = reason

with output.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=list(rows[0].keys()),
        delimiter="\t",
        lineterminator="\n",
    )
    writer.writeheader()
    writer.writerows(rows)

counts = Counter(row["判定"] for row in rows)

print("建议审核结果已保存：", output)
print("\n===== 第 4 批建议标签统计 =====")
for label in ["正确", "有歧义", "专有名词", "地区用语"]:
    print(f"{label}: {counts[label]}")

print("\n原始审核表和语料文件均未修改。")
print("以上标签属于初步建议，尚未完成外部专名核实。")

建议审核结果已保存： /content/xime_work/xime_macbert_colab_ready/data/processed/csc_quality_review/batches/review_batch_04_suggested.tsv

===== 第 4 批建议标签统计 =====
正确: 19
有歧义: 4
专有名词: 2
地区用语: 0

原始审核表和语料文件均未修改。
以上标签属于初步建议，尚未完成外部专名核实。


In [32]:
from pathlib import Path
import csv

path = Path(
    "/content/xime_work/xime_macbert_colab_ready/"
    "data/processed/csc_quality_review/batches/review_batch_05.tsv"
)

with path.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

print(f"第 5 批，共 {len(rows)} 条\n")

for i, row in enumerate(rows, 1):
    print(f"===== 第 {i} 条 =====")
    print(f"分组：{row['分组']}；差异数：{row['差异数']}")
    print(f"错误句：{row['错误句_original']}")
    print(f"目标句：{row['目标句_correct']}")
    print()

第 5 批，共 25 条

===== 第 1 条 =====
分组：single；差异数：1
错误句：赖昌星若被遣这回到中国，几可确定必然面临死刑判决。
目标句：赖昌星若被遣送回到中国，几可确定必然面临死刑判决。

===== 第 2 条 =====
分组：single；差异数：1
错误句：恩戈朗说，在阿必尚莫他区有三名警察受伤，并有一名路人遭一名狙击手击毙。
目标句：恩戈朗说，在阿必尚其他区有三名警察受伤，并有一名路人遭一名狙击手击毙。

===== 第 3 条 =====
分组：single；差异数：1
错误句：南韩情报通信部五曰说，由于韩国被指控收买政府官员以取得政府的合约，情报部将针对其采购部门进行调查，彻查是否有任何贪污行为。
目标句：南韩情报通信部五日说，由于韩国被指控收买政府官员以取得政府的合约，情报部将针对其采购部门进行调查，彻查是否有任何贪污行为。

===== 第 4 条 =====
分组：single；差异数：1
错误句：科学家指出，过去二十年来，生长在泥泞沿岸地区的热带灌木红树林已经消失百分之三十五，两成的珊瑚谯已遭到破坏，百分之二十五的针叶树与百分之三十五的两栖动物也有灭绝之虞。
目标句：科学家指出，过去二十年来，生长在泥泞沿岸地区的热带灌木红树林已经消失百分之三十五，两成的珊瑚礁已遭到破坏，百分之二十五的针叶树与百分之三十五的两栖动物也有灭绝之虞。

===== 第 5 条 =====
分组：single；差异数：1
错误句：他说：我耍求贝鲁斯柯尼考虑这项构想，率队前往特拉维夫，并邀请以色列当局和巴勒斯坦当局和他一起观赏球赛。
目标句：他说：我要求贝鲁斯柯尼考虑这项构想，率队前往特拉维夫，并邀请以色列当局和巴勒斯坦当局和他一起观赏球赛。

===== 第 6 条 =====
分组：single；差异数：1
错误句：旧美方宣称海珊重新制造了这些武器。
目标句：但美方宣称海珊重新制造了这些武器。

===== 第 7 条 =====
分组：single；差异数：1
错误句：她说：首相如今交付我们的任务就是，确旧大家都会全力以赴。
目标句：她说：首相如今交付我们的任务就是，确保大家都会全力以赴。

===== 第 8 条 =====
分组：multi_2_3；差异数：2
错误句：英国白金汊官周五表示，住院第三天的英国玛格丽特公主，进食的情况已

In [33]:
from pathlib import Path
import csv
from collections import Counter

root = Path("/content/xime_work/xime_macbert_colab_ready")
source = root / "data/processed/csc_quality_review/batches/review_batch_05.tsv"
output = root / "data/processed/csc_quality_review/batches/review_batch_05_suggested.tsv"

labels = [
    "正确", "专有名词", "正确", "正确", "正确",
    "有歧义", "正确", "专有名词", "正确", "正确",
    "有歧义", "正确", "正确", "有歧义", "有歧义",
    "正确", "正确", "正确", "正确", "专有名词",
    "专有名词", "正确", "正确", "正确", "正确",
]

reasons = [
    "遣这回到改为遣送回到，修正合理。",
    "其他区在语义上合理，但阿必尚涉及地名译法，建议核实。",
    "五曰改为五日，明显字符错误修正。",
    "珊瑚谯改为珊瑚礁，符合生态环境语境。",
    "耍求改为要求，明显错字修正。",
    "旧美方改为但美方可能合理，但需要前文确认转折关系。",
    "确旧改为确保，符合句意。",
    "白金汊官改为白金汉宫，符合机构名称，建议核实来源。",
    "十佳改为市价，符合苹果公司市值语境。",
    "南讳改为南韩，符合上下文。",
    "遭遇、事件的修正合理；麦卡锡译名及一九五年代仍需复核。",
    "琅琅改为朗朗，符合钢琴演奏家朗朗的姓名写法。",
    "果帐改为呆帐，符合金融语境。",
    "位试飞改为为试飞合理，但整句像残缺的新闻片段。",
    "多处错字得到修正，但地图是美国科罗拉多州仍存在语法问题。",
    "多处明显错字得到修正，目标句整体合理。",
    "明查暗访改为明察暗访，成语用字修正。",
    "人道缓肋改为人道援助，符合救灾语境。",
    "俞来愈改为愈来愈，表达修正合理。",
    "好钢改为郝刚涉及具体人名，缺少依据，需核实。",
    "遵从、项目的修正合理；佘契尔涉及人物译名。",
    "二十曰改为二十日，明显字符错误修正。",
    "曰东改为日本，符合上下文。",
    "具名、想要、其中等修正合理。",
    "靖国神社、日本的修正符合语境。",
]

assert len(labels) == len(reasons) == 25

with source.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

assert len(rows) == 25, f"预期 25 条，实际 {len(rows)} 条"

for row, label, reason in zip(rows, labels, reasons):
    row["判定"] = label
    row["备注"] = reason

with output.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=list(rows[0].keys()),
        delimiter="\t",
        lineterminator="\n",
    )
    writer.writeheader()
    writer.writerows(rows)

counts = Counter(row["判定"] for row in rows)

print("建议审核结果已保存：", output)
print("\n===== 第 5 批建议标签统计 =====")
for label in ["正确", "有歧义", "专有名词", "地区用语"]:
    print(f"{label}: {counts[label]}")

print("\n原始审核表和语料文件均未修改。")
print("以上是初步建议，尚未完成外部专名核实。")

建议审核结果已保存： /content/xime_work/xime_macbert_colab_ready/data/processed/csc_quality_review/batches/review_batch_05_suggested.tsv

===== 第 5 批建议标签统计 =====
正确: 17
有歧义: 4
专有名词: 4
地区用语: 0

原始审核表和语料文件均未修改。
以上是初步建议，尚未完成外部专名核实。


In [34]:
from pathlib import Path
import csv

path = Path(
    "/content/xime_work/xime_macbert_colab_ready/"
    "data/processed/csc_quality_review/batches/review_batch_06.tsv"
)

with path.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

print(f"第 6 批，共 {len(rows)} 条\n")

for i, row in enumerate(rows, 1):
    print(f"===== 第 {i} 条 =====")
    print(f"分组：{row['分组']}；差异数：{row['差异数']}")
    print(f"错误句：{row['错误句_original']}")
    print(f"目标句：{row['目标句_correct']}")
    print()

第 6 批，共 25 条

===== 第 1 条 =====
分组：multi_2_3；差异数：2
错误句：油纸近期楼价急跌两成多。
目标句：有指近期楼价急跌两成多。

===== 第 2 条 =====
分组：single；差异数：1
错误句：欧盟执行委员会在一项报告中指出，在一九六0至二000年之间，欧洲人的平均寿命增长了八岁，目前男性平均寿命为七十五岁，女性为八十一岁，都比豪国人多一岁。
目标句：欧盟执行委员会在一项报告中指出，在一九六0至二000年之间，欧洲人的平均寿命增长了八岁，目前男性平均寿命为七十五岁，女性为八十一岁，都比美国人多一岁。

===== 第 3 条 =====
分组：single；差异数：1
错误句：中央气象局表示，台湾北部、东北部和中部地区末来两天可能会持续降豪大雨，山区应严防坍方和土石流等灾害。
目标句：中央气象局表示，台湾北部、东北部和中部地区未来两天可能会持续降豪大雨，山区应严防坍方和土石流等灾害。

===== 第 4 条 =====
分组：single；差异数：1
错误句：十四日凌晨河南省人民医院向其加入下达病危通知书。
目标句：十四日凌晨河南省人民医院向其家入下达病危通知书。

===== 第 5 条 =====
分组：single；差异数：1
错误句：布希没有点名是那些国家未出力铲除恐怖主义，以及协助缉捕被指为九月十一日对世捍贸易中心和国防部发动恐怖攻击的主嫌欧萨玛宾拉登。
目标句：布希没有点名是那些国家未出力铲除恐怖主义，以及协助缉捕被指为九月十一日对世界贸易中心和国防部发动恐怖攻击的主嫌欧萨玛宾拉登。

===== 第 6 条 =====
分组：multi_2_3；差异数：2
错误句：三十九岁的杨斌三曰在一项临时记者会中表示，他的公司积欠了大约一百二十万美元税款，但保证将迅速缴清，他坚称，这不罡个大问题。
目标句：三十九岁的杨斌三日在一项临时记者会中表示，他的公司积欠了大约一百二十万美元税款，但保证将迅速缴清，他坚称，这不是个大问题。

===== 第 7 条 =====
分组：review_4plus；差异数：5
错误句：在此同时，许多贸易伙伴非常不满美国颁行十年提共一干七百三十五亿美元农业津贴的法案，这将波坏消除贸易障拟的怒力。
目标句：在此同时，许多贸易伙伴非常不满美国颁行十年提供一千七百三十五亿美元农业津贴的

In [35]:
from pathlib import Path
import csv
from collections import Counter

root = Path("/content/xime_work/xime_macbert_colab_ready")

source = (
    root / "data/processed/csc_quality_review/batches/"
    "review_batch_06.tsv"
)
output = (
    root / "data/processed/csc_quality_review/batches/"
    "review_batch_06_suggested.tsv"
)

labels = [
    "正确", "正确", "有歧义", "有歧义", "有歧义",
    "正确", "正确", "有歧义", "正确", "正确",
    "正确", "正确", "正确", "正确", "正确",
    "正确", "有歧义", "正确", "正确", "正确",
    "正确", "有歧义", "正确", "正确", "有歧义",
]

reasons = [
    "油纸改为有指，结合楼价新闻语境，修正方向合理。",
    "豪国人改为美国人，符合上下文。",
    "末来改为未来正确，但持续降豪大雨仍不自然，目标句可能残留错误。",
    "加入改为家入仍不通顺，结合句意可能想表达向其家人下达。",
    "世捍改为世界正确，但是那些国家可能仍有用词问题，需要上下文确认。",
    "三曰改为三日、不罡改为不是，两处修正均合理。",
    "多处明显错字得到修正，目标句整体基本通顺。",
    "机器改为及其可能符合上下文，但整句是残缺片段，无法独立确认。",
    "影导天燃气改为引导天然气，修正合理。",
    "土耳莫改为土耳其，符合地名及上下文。",
    "直明事实真相改为查明事实真相，搭配合理。",
    "北京考鸭改为北京烤鸭、外国采改为外国菜，均符合语境。",
    "乐观肘改为乐观时，明显错字修正。",
    "恐怖玫击改为恐怖攻击，符合语境。",
    "堆持、忍何、导至、冲秃等错误得到合理修正。",
    "传毓武器改为传统武器，符合军事语境。",
    "企业界改为企业借后目标句仍不自然，商品卖四千一百元也需要结合原意确认。",
    "石涠部门改为石油部门，符合语境。",
    "问题袖认为改为问题被认为，语法合理。",
    "法宫改为法官，明显错字修正。",
    "部要、肘、开使、一值等错误得到合理修正。",
    "它改为他可能合理，但需要上下文确认代词指代。",
    "最亲一次改为最新一次，符合语义。",
    "报子改为报纸、遢送改为递送，修正合理。",
    "多处错字已修正，但主流此外缺少标点，不余遗力疑似应为不遗余力；人物和著述名称也需核实。",
]

assert len(labels) == 25
assert len(reasons) == 25

with source.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

assert len(rows) == 25, f"预期 25 条，实际 {len(rows)} 条"

for row, label, reason in zip(rows, labels, reasons):
    row["判定"] = label
    row["备注"] = reason

with output.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=list(rows[0].keys()),
        delimiter="\t",
        lineterminator="\n",
    )
    writer.writeheader()
    writer.writerows(rows)

counts = Counter(row["判定"] for row in rows)

print("建议审核结果已保存：", output)
print("\n===== 第 6 批建议标签统计 =====")
for label in ["正确", "有歧义", "专有名词", "地区用语"]:
    print(f"{label}: {counts[label]}")

print("\n原始审核表和语料文件均未修改。")

建议审核结果已保存： /content/xime_work/xime_macbert_colab_ready/data/processed/csc_quality_review/batches/review_batch_06_suggested.tsv

===== 第 6 批建议标签统计 =====
正确: 18
有歧义: 7
专有名词: 0
地区用语: 0

原始审核表和语料文件均未修改。


In [36]:
from pathlib import Path
import csv

path = Path(
    "/content/xime_work/xime_macbert_colab_ready/"
    "data/processed/csc_quality_review/batches/review_batch_07.tsv"
)

with path.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

print(f"第 7 批，共 {len(rows)} 条\n")

for i, row in enumerate(rows, 1):
    print(f"===== 第 {i} 条 =====")
    print(f"分组：{row['分组']}；差异数：{row['差异数']}")
    print(f"错误句：{row['错误句_original']}")
    print(f"目标句：{row['目标句_correct']}")
    print()

第 7 批，共 25 条

===== 第 1 条 =====
分组：review_4plus；差异数：4
错误句：紫雷回忆小时候初次接触的漫画就是曾经引起台湾轰动的日本漫画好小子。大该是幼智院吧，当时觉得好小子漫画的构图及情节好好玩，就这样开始尝试涂鸭，画自己最喜欢的人物图样，至于会走进漫画界，完全是因缘际会，帮一个日本漫画家画插图后才开始，也算巧合吧，紫雷带著甜美的笑容叙述这段经历。
目标句：紫雷回忆小时候初次接触的漫画就是曾经引起台湾轰动的日本漫画好小子。大概是幼稚园吧，当时觉得好小子漫画的构图及情节好好玩，就这样开始尝试涂鸦，画自己最喜欢的人物图样，至于会走进漫画界，完全是因缘际会，帮一个日本漫画家画插图后才开始，也算巧合吧，紫雷带著甜美的笑容叙述这段经历。

===== 第 2 条 =====
分组：multi_2_3；差异数：2
错误句：金吉奇说：现阶段我们只能侬滥权和武装叛乱霏名起诉他。
目标句：金吉奇说：现阶段我们只能依滥权和武装叛乱罪名起诉他。

===== 第 3 条 =====
分组：review_4plus；差异数：11
错误句：不懂中文真麻烦！现在我的中文说的比以前好多了。我真的很喜欢学中文！
目标句：不懂中文真麻烦！现在我的中文我的中文说得比以前好多了很喜欢学中文！

===== 第 4 条 =====
分组：multi_2_3；差异数：2
错误句：矢志横扫满贯大赛的索伦丝坦与两位少女业余选手魏成黄帼普蕾索，以及美国资深女将英克丝特与瑞典的柯克，同以七十杆作收。
目标句：矢志横扫满贯大赛的索伦丝坦与两位少女业余选手魏成美和普蕾索，以及美国资深女将英克丝特与瑞典的柯克，同以七十杆作收。

===== 第 5 条 =====
分组：multi_2_3；差异数：2
错误句：他说，在预定九月十一日于墨西哥坎肯召开的世养贸易组织会议上，日本将针对降低农业贸易关税的问题，与欧盟及开发中国家维持密切合作，和以豪国为首的粮食出口大国对抗。
目标句：他说，在预定九月十一日于墨西哥坎肯召开的世界贸易组织会议上，日本将针对降低农业贸易关税的问题，与欧盟及开发中国家维持密切合作，和以美国为首的粮食出口大国对抗。

===== 第 6 条 =====
分组：single；差异数：1
错误句：全天推杆均末超过十八呎的朴祉垠说：我一整天的感觉都很好，感觉有获胜机会。


In [37]:
from pathlib import Path
import csv
from collections import Counter

root = Path("/content/xime_work/xime_macbert_colab_ready")

source = (
    root / "data/processed/csc_quality_review/batches/"
    "review_batch_07.tsv"
)
output = (
    root / "data/processed/csc_quality_review/batches/"
    "review_batch_07_suggested.tsv"
)

labels = [
    "正确", "有歧义", "有歧义", "专有名词", "正确",
    "正确", "正确", "专有名词", "正确", "正确",
    "正确", "正确", "专有名词", "专有名词", "正确",
    "正确", "有歧义", "正确", "正确", "有歧义",
    "正确", "专有名词", "正确", "专有名词", "正确",
]

reasons = [
    "大该、幼智院、涂鸭改为大概、幼稚园、涂鸦，修正合理。",
    "依滥权和武装叛乱罪名起诉不够自然，可能应为以相关罪名起诉。",
    "目标句出现我的中文重复，且好多了很喜欢缺少衔接。",
    "涉及高尔夫球员姓名及断句，需要核实姓名和原文。",
    "世养、以豪国改为世界、以美国，符合国际贸易语境。",
    "末改为未，符合未超过十八呎的表达。",
    "虽让、最犬、胜力、么有改为虽然、最大、胜利、没有，整体合理。",
    "涉及香港机构名称，空运牌照局的正式名称需要核实。",
    "糟地震信袭改为再遭地震侵袭，语义合理。",
    "姜味改为将为，符合提供服务的语境。",
    "威软公司改为微软公司，公司名称修正合理。",
    "多处明显错字得到修正，整体可读。",
    "涉及人物译名欧兹奥斯朋，需核实；摩托车修正合理。",
    "涉及伍德布里吉的人名译法，其余多处错字修正合理。",
    "过的、相当不措、着、归昝改为过得、相当不错、这、归咎，修正合理。",
    "浸示改为表示，明显错字修正。",
    "多处错字修正合理，但人口占全国23缺少百分号或其他计量单位。",
    "荀许多改为有许多，符合句意。",
    "罹雷膀胱癌改为罹患膀胱癌，表达合理。",
    "直询改为质询合理，但该派改为改派可能改变原意。",
    "必耍改为必要，明显错字修正。",
    "墓德改为基德涉及篮球人物译名，建议核实。",
    "己改为已、名子改为名字，修正合理。",
    "若望保禄二世涉及教宗姓名译法，圣徙改为圣徒合理。",
    "那是改为那时，符合时间语境。",
]

assert len(labels) == 25
assert len(reasons) == 25

with source.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

assert len(rows) == 25, f"预期 25 条，实际 {len(rows)} 条"

for row, label, reason in zip(rows, labels, reasons):
    row["判定"] = label
    row["备注"] = reason

with output.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=list(rows[0].keys()),
        delimiter="\t",
        lineterminator="\n",
    )
    writer.writeheader()
    writer.writerows(rows)

counts = Counter(row["判定"] for row in rows)

print("建议审核结果已保存：", output)
print("\n===== 第 7 批建议标签统计 =====")
for label in ["正确", "有歧义", "专有名词", "地区用语"]:
    print(f"{label}: {counts[label]}")

print("\n原始审核表和语料文件均未修改。")

建议审核结果已保存： /content/xime_work/xime_macbert_colab_ready/data/processed/csc_quality_review/batches/review_batch_07_suggested.tsv

===== 第 7 批建议标签统计 =====
正确: 15
有歧义: 4
专有名词: 6
地区用语: 0

原始审核表和语料文件均未修改。


In [38]:
from pathlib import Path
import csv

path = Path(
    "/content/xime_work/xime_macbert_colab_ready/"
    "data/processed/csc_quality_review/batches/review_batch_08.tsv"
)

with path.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

print(f"第 8 批，共 {len(rows)} 条\n")

for i, row in enumerate(rows, 1):
    print(f"===== 第 {i} 条 =====")
    print(f"分组：{row['分组']}；差异数：{row['差异数']}")
    print(f"错误句：{row['错误句_original']}")
    print(f"目标句：{row['目标句_correct']}")
    print()

第 8 批，共 25 条

===== 第 1 条 =====
分组：review_4plus；差异数：4
错误句：他说，即然我们双方领导人将要轮流互访，我们就有必要营造一些良好气氛来配合我希望今天出席的美国朋友们，都能踊跃的把握机会，做出积极贡现，进一步促进双方的友宜。不是连柯林顿总统自己都一再明白说，今后美国不会孤立、围堵、更不会对抗中国，而是要交往吗
目标句：他说，既然我们双方领导人将要轮流互访，我们就有必要营造一些良好气氛来配合我希望今天出席地美国朋友们，都能踊跃的把握机会，做出积极贡献，进一步促进双方的友谊。不是连柯林顿总统自己都一再明白说，今后美国不会孤立、围堵、更不会对抗中国，而是要交往吗

===== 第 2 条 =====
分组：multi_2_3；差异数：2
错误句：主亦单位表示，二十八日的活动有一万五千人参加，预估二十九曰的示威游行则将有十万人参加。
目标句：主办单位表示，二十八日的活动有一万五千人参加，预估二十九日的示威游行则将有十万人参加。

===== 第 3 条 =====
分组：multi_2_3；差异数：2
错误句：他称许巴勒斯坦领袖阿拉法特，说他是对抗忧太复国运动的阿拉旧斗士之一。
目标句：他称许巴勒斯坦领袖阿拉法特，说他是对抗犹太复国运动的阿拉伯斗士之一。

===== 第 4 条 =====
分组：multi_2_3；差异数：2
错误句：福斯汽车公司直事长皮契八月二十曰告诉德国金融时报说，这份薪资高过福斯通常付给员工的薪资。
目标句：福斯汽车公司董事长皮契八月二十日告诉德国金融时报说，这份薪资高过福斯通常付给员工的薪资。

===== 第 5 条 =====
分组：multi_2_3；差异数：2
错误句：投票将在晚间八时结束，初步的正式开祟结果料将在晚间十一时三十分台北时间三日清晨五肘三十分揭晓。
目标句：投票将在晚间八时结束，初步的正式开票结果料将在晚间十一时三十分台北时间三日清晨五时三十分揭晓。

===== 第 6 条 =====
分组：multi_2_3；差异数：2
错误句：北韩讥讽华府家反恐怖战争之名扮供南韩新式武器，无异在朝鲜半岛制造战争情势。
目标句：北韩讥讽华府假反恐怖战争之名提供南韩新式武器，无异在朝鲜半岛制造战争情势。

===== 第 7 条 =====
分组：multi_2_3；差异数：2
错误句：九连

In [39]:
from pathlib import Path
import csv
from collections import Counter

root = Path("/content/xime_work/xime_macbert_colab_ready")

source = (
    root / "data/processed/csc_quality_review/batches/"
    "review_batch_08.tsv"
)
output = (
    root / "data/processed/csc_quality_review/batches/"
    "review_batch_08_suggested.tsv"
)

labels = [
    "有歧义", "正确", "正确", "专有名词", "正确",
    "正确", "正确", "正确", "正确", "正确",
    "专有名词", "有歧义", "正确", "正确", "正确",
    "正确", "有歧义", "有歧义", "有歧义", "专有名词",
    "有歧义", "正确", "正确", "正确", "有歧义",
]

reasons = [
    "出席地美国朋友们仍有用词错误，目标句尚未完全修正。",
    "主亦、二十九曰改为主办、二十九日，修正合理。",
    "忧太、阿拉旧改为犹太、阿拉伯，符合语境。",
    "董事长修正合理，但公司名称和人物译名仍需核实。",
    "开祟、五肘改为开票、五时，符合选举新闻语境。",
    "家反、扮供改为假、提供，修正合理。",
    "九连七改为就连其，语法和语义基本合理。",
    "三万三干、上办庄改为三万三千、上半年，修正合理。",
    "尼涸、八千致改为尼泊尔、八千至，修正合理。",
    "拟视改为凝视，搭配正确。",
    "涉及国家名称斯洛伐克，建议核实译名。",
    "南辕北辙的各个国家语义不自然，目标句仍需复核。",
    "恐怖玫击、像片改为恐怖攻击、相片，修正合理。",
    "所为、原意、朋唷、影像改为所谓、原因、朋友、影响，合理。",
    "也董的改为也懂得，符合语境。",
    "多处明显错字得到修正，整体意思基本清楚。",
    "多处错字已修正，但部分语句不够自然，需要上下文确认。",
    "同肘改为同时合理，但推毁仍未修正，药物名称也疑似缺失。",
    "城煌庙疑似仍应为城隍庙，整句还需要进一步复核。",
    "涉及公司和企业名称，建议核实专名；普通股等修正合理。",
    "弟弟弟存在重复，要是我有时间所以等表达仍不通顺。",
    "废埔改为废墟，符合语境。",
    "多处明显错字修正合理，整体表达通顺。",
    "遭遍改为遭遇，修正合理。",
    "关萦企业改为关系企业合理，但公司净值的25疑似缺少百分号。",
]

assert len(labels) == len(reasons) == 25

with source.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

assert len(rows) == 25, f"预期 25 条，实际 {len(rows)} 条"

for row, label, reason in zip(rows, labels, reasons):
    row["判定"] = label
    row["备注"] = reason

with output.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=list(rows[0].keys()),
        delimiter="\t",
        lineterminator="\n",
    )
    writer.writeheader()
    writer.writerows(rows)

counts = Counter(row["判定"] for row in rows)

print("建议审核结果已保存：", output)
print("\n===== 第 8 批建议标签统计 =====")
for label in ["正确", "有歧义", "专有名词", "地区用语"]:
    print(f"{label}: {counts[label]}")

print("\n原始审核表和语料文件均未修改。")

建议审核结果已保存： /content/xime_work/xime_macbert_colab_ready/data/processed/csc_quality_review/batches/review_batch_08_suggested.tsv

===== 第 8 批建议标签统计 =====
正确: 15
有歧义: 7
专有名词: 3
地区用语: 0

原始审核表和语料文件均未修改。


In [40]:
from pathlib import Path
import csv

path = Path(
    "/content/xime_work/xime_macbert_colab_ready/"
    "data/processed/csc_quality_review/batches/"
    "review_batch_09.tsv"
)

with path.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

print(f"第 9 批，共 {len(rows)} 条\n")

for i, row in enumerate(rows, 1):
    print(f"===== 第 {i} 条 =====")
    print(f"分组：{row['分组']}；差异数：{row['差异数']}")
    print(f"错误句：{row['错误句_original']}")
    print(f"目标句：{row['目标句_correct']}")
    print()

第 9 批，共 25 条

===== 第 1 条 =====
分组：multi_2_3；差异数：2
错误句：新任联合国驻柯索伏行政长官、前丹麦国防部长海克鲁普已正式展开工作，希望能影导这个南斯拉天暴力猖獗的省份重新迈向和平之路。
目标句：新任联合国驻柯索伏行政长官、前丹麦国防部长海克鲁普已正式展开工作，希望能引导这个南斯拉夫暴力猖獗的省份重新迈向和平之路。

===== 第 2 条 =====
分组：single；差异数：1
错误句：二十五岁的泰勤在资格赛成绩虽然普通，但坚信自己实力与二000年无异。
目标句：二十五岁的泰勒在资格赛成绩虽然普通，但坚信自己实力与二000年无异。

===== 第 3 条 =====
分组：single；差异数：1
错误句：由于在地而配合作战的阿富汗部队不让记者进入战区采访，目前还无法评估这些攻击造成的损失情形。
目标句：由于在地面配合作战的阿富汗部队不让记者进入战区采访，目前还无法评估这些攻击造成的损失情形。

===== 第 4 条 =====
分组：multi_2_3；差异数：3
错误句：哈维随坦承道底那些国家会被邀请加入仍属末定之天，但同时亦表示，斯洛伐克和斯洛温尼亚明年获邀加入北约的机会仍然很高。
目标句：哈维虽坦承到底那些国家会被邀请加入仍属未定之天，但同时亦表示，斯洛伐克和斯洛温尼亚明年获邀加入北约的机会仍然很高。

===== 第 5 条 =====
分组：single；差异数：1
错误句：经建会分析，上述统计如分作二期观察，前五年劳动生产力平均年增率为百分之四点五，主要是产出原由比增加百分之三点九所致，后五年成长率则高达百分之九点二，这接段产出原料比平均增加率仍有百分之三点一，资本劳动比则高达百分之十二点六，企业的自动化使得劳动生产力大幅提升。
目标句：经建会分析，上述统计如分作二期观察，前五年劳动生产力平均年增率为百分之四点五，主要是产出原由比增加百分之三点九所致，后五年成长率则高达百分之九点二，这阶段产出原料比平均增加率仍有百分之三点一，资本劳动比则高达百分之十二点六，企业的自动化使得劳动生产力大幅提升。

===== 第 6 条 =====
分组：multi_2_3；差异数：2
错误句：他因为力促恢复勉学费高等教盲而令史卓颇为难堪。
目标句：他因为力促恢复免学费高等教育而令史卓颇为难堪。

===== 第 7

In [41]:
from pathlib import Path
import csv
from collections import Counter

root = Path("/content/xime_work/xime_macbert_colab_ready")

source = (
    root / "data/processed/csc_quality_review/batches/"
    "review_batch_09.tsv"
)
output = (
    root / "data/processed/csc_quality_review/batches/"
    "review_batch_09_suggested.tsv"
)

labels = [
    "专有名词", "专有名词", "正确", "正确", "有歧义",
    "正确", "正确", "有歧义", "有歧义", "正确",
    "正确", "正确", "正确", "有歧义", "正确",
    "正确", "正确", "正确", "有歧义", "有歧义",
    "正确", "正确", "正确", "有歧义", "正确",
]

reasons = [
    "引导修正合理，但地名、人物译名及南斯拉夫相关表述值得核实。",
    "泰勤改为泰勒可能正确，但属于人物姓名，需要上下文确认。",
    "地而改为地面，符合军事语境。",
    "随、道底、末定之天改为虽、到底、未定之天，整体可读。",
    "这接段改为这阶段合理，但产出原由比、产出原料比仍疑似有误。",
    "勉学费高等教盲改为免学费高等教育，合理。",
    "结荣、程受改为结束、承受，符合语境。",
    "公民营机构可能是正确术语，但阿裔部长和代表的含义需要上下文。",
    "技艺校育、技艺中心学教等表达仍疑似错误，目标句需要复核。",
    "妞约、重一、曰圆改为纽约、从一、日圆，合理。",
    "比须改为必须，修正合理。",
    "一枝之长改为一技之长，成语修正正确。",
    "但任改为担任，合理。",
    "多处错字已修正，但过时间后我很了解这种情况仍不自然。",
    "金正曰、恬统改为金正日、传统，合理。",
    "重央改为中央，合理。",
    "十肘改为十时，符合时间表达。",
    "朗货改为期货，符合原油交易语境。",
    "预定、二日的修正合理，但具体日期背景缺失，整句略显生硬。",
    "规范机构名称通常是国际货币基金组织，目标句仍可能有专名错误。",
    "君手改为着手，符合着手补强生产线的搭配。",
    "撮告、即然、收到改为报告、既然、受到，合理。",
    "件况、罡否、记续、虾去改为情况、是否、继续、下去，合理。",
    "不旦改为不但合理，但句首结构不完整，可能依赖前文。",
    "多处明显错字修正合理，目标句整体通顺。",
]

assert len(labels) == len(reasons) == 25

with source.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

assert len(rows) == 25, f"预期 25 条，实际 {len(rows)} 条"

for row, label, reason in zip(rows, labels, reasons):
    row["判定"] = label
    row["备注"] = reason

with output.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=list(rows[0].keys()),
        delimiter="\t",
        lineterminator="\n",
    )
    writer.writeheader()
    writer.writerows(rows)

counts = Counter(row["判定"] for row in rows)

print("建议审核结果已保存：", output)
print("\n===== 第 9 批建议标签统计 =====")
for label in ["正确", "有歧义", "专有名词", "地区用语"]:
    print(f"{label}: {counts[label]}")

print("\n原始审核表和语料文件均未修改。")

建议审核结果已保存： /content/xime_work/xime_macbert_colab_ready/data/processed/csc_quality_review/batches/review_batch_09_suggested.tsv

===== 第 9 批建议标签统计 =====
正确: 16
有歧义: 7
专有名词: 2
地区用语: 0

原始审核表和语料文件均未修改。


In [42]:
from pathlib import Path
import csv

path = Path(
    "/content/xime_work/xime_macbert_colab_ready/"
    "data/processed/csc_quality_review/batches/"
    "review_batch_10.tsv"
)

with path.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

print(f"第 10 批，共 {len(rows)} 条\n")

for i, row in enumerate(rows, 1):
    print(f"===== 第 {i} 条 =====")
    print(f"分组：{row['分组']}；差异数：{row['差异数']}")
    print(f"错误句：{row['错误句_original']}")
    print(f"目标句：{row['目标句_correct']}")
    print()

第 10 批，共 25 条

===== 第 1 条 =====
分组：review_4plus；差异数：4
错误句：我为他、我们以及所有牺生奉献促成此韦的人感到卑哀，他是一个很怀的榜样。
目标句：我为他、我们以及所有牺牲奉献促成此事的人感到悲哀，他是一个很坏的榜样。

===== 第 2 条 =====
分组：multi_2_3；差异数：3
错误句：原世养第一小威廉丝在一周前拿下中国公开萤冠军后，排名由第十及到第九，另一美国女将卡普莉提也进步一名到第八。
目标句：原世界第一小威廉丝在一周前拿下中国公开赛冠军后，排名由第十挤到第九，另一美国女将卡普莉提也进步一名到第八。

===== 第 3 条 =====
分组：single；差异数：1
错误句：这些国际投资菖在中国市场的份额，只限于开设合资企业及外商独资企业。
目标句：这些国际投资者在中国市场的份额，只限于开设合资企业及外商独资企业。

===== 第 4 条 =====
分组：review_4plus；差异数：4
错误句：瓜地马拉贫穷的原族民虽占人口多数，但长期以来遭到歧视和排斥，并且常遭暴力侵害，教宗这项谈话茌这个国家别冥意议。
目标句：瓜地马拉贫穷的原住民虽占人口多数，但长期以来遭到歧视和排斥，并且常遭暴力侵害，教宗这项谈话在这个国家别具意义。

===== 第 5 条 =====
分组：single；差异数：1
错误句：李文史东表示：有些人以为，有关欧元的问题可以在无形中就获得胜利，后者甚至根本不提这个问题。李文史东已经被驱逐出执政党工党，但是仍然获得该党草根民众的热烈支持。
目标句：李文史东表示：有些人以为，有关欧元的问题可以在无形中就获得胜利，或者甚至根本不提这个问题。李文史东已经被驱逐出执政党工党，但是仍然获得该党草根民众的热烈支持。

===== 第 6 条 =====
分组：multi_2_3；差异数：2
错误句：三和总研的资深经济学家说：对美汽车出口是曰本的三项昂重要外销项目之一，另两项是电子零组件和一般机械，美国对曰本汽车的需求减弱令人忧心。
目标句：三和总研的资深经济学家说：对美汽车出口是日本的三项最重要外销项目之一，另两项是电子零组件和一般机械，美国对曰本汽车的需求减弱令人忧心。

===== 第 7 条 =====
分组：multi_2_3；差异数：2
错误句：道底是外甥女像舅舅借钱买

In [43]:
from pathlib import Path
import csv
from collections import Counter

root = Path("/content/xime_work/xime_macbert_colab_ready")

source = (
    root / "data/processed/csc_quality_review/batches/"
    "review_batch_10.tsv"
)
output = (
    root / "data/processed/csc_quality_review/batches/"
    "review_batch_10_suggested.tsv"
)

labels = [
    "有歧义", "专有名词", "正确", "正确", "正确",
    "有歧义", "正确", "正确", "有歧义", "专有名词",
    "专有名词", "专有名词", "专有名词", "正确", "专有名词",
    "有歧义", "专有名词", "正确", "正确", "有歧义",
    "正确", "正确", "正确", "正确", "有歧义",
]

reasons = [
    "多处错字已修正，但所有牺牲奉献促成此事的人搭配不自然。",
    "涉及小威廉丝、卡普莉提等人物译名，需要核实。",
    "投资菖改为投资者，修正合理。",
    "原族民、茌、别冥意议改为原住民、在、别具意义，整体通顺。",
    "后者改为或者，符合前后文逻辑。",
    "昂重要改为最重要合理，但后文仍有对曰本这一残留错误。",
    "道底、像改为到底、向，搭配合理。",
    "撮导、旱年改为报导、早年，符合新闻语境。",
    "而机器被改成二机器，目标句引入明显错误。",
    "土耳莫改为土耳其合理，但国防部长姓名译法需要核实。",
    "刘红波改为刘洪波涉及人名，缺少上下文无法确认。",
    "哈雯里改为哈里里涉及人物姓名，需要结合新闻背景核实。",
    "加拿大皇家警骑可能仍有机构名称错误，建议核实规范译名。",
    "多处明显错字得到修正，整体表达合理。",
    "在瑞士的修正合理，但地名及人物背景建议结合上下文核实。",
    "通明点灯仍不自然，可能应为通宵点灯等表达，需核实。",
    "涉及布希、夏隆等人物译名，且时间表达略显混乱。",
    "这顷计画己获信改为这项计画已获得，修正合理。",
    "彗施改为措施，合理；联准会、葛老属于特定语境称呼。",
    "如粟改为如果合理，但人员不足和有所需要表达不自然。",
    "佛罗里达打学、汁划改为佛罗里达大学、计划，合理。",
    "根墓改为根基，合理。",
    "尚末改为尚未，合理。",
    "多处明显错字得到修正，整体语义通顺。",
    "多处错字得到修正，但尽量自制保护平民的搭配不自然。",
]

assert len(labels) == len(reasons) == 25

with source.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

assert len(rows) == 25, f"预期 25 条，实际 {len(rows)} 条"

for row, label, reason in zip(rows, labels, reasons):
    row["判定"] = label
    row["备注"] = reason

with output.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=list(rows[0].keys()),
        delimiter="\t",
        lineterminator="\n",
    )
    writer.writeheader()
    writer.writerows(rows)

counts = Counter(row["判定"] for row in rows)

print("建议审核结果已保存：", output)
print("\n===== 第 10 批建议标签统计 =====")
for label in ["正确", "有歧义", "专有名词", "地区用语"]:
    print(f"{label}: {counts[label]}")

print("\n原始审核表和语料文件均未修改。")

建议审核结果已保存： /content/xime_work/xime_macbert_colab_ready/data/processed/csc_quality_review/batches/review_batch_10_suggested.tsv

===== 第 10 批建议标签统计 =====
正确: 12
有歧义: 6
专有名词: 7
地区用语: 0

原始审核表和语料文件均未修改。


In [44]:
from pathlib import Path
import pandas as pd
import json

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
BATCH_DIR = ROOT / "data/processed/csc_quality_review/batches"
OUT_DIR = ROOT / "data/processed/csc_quality_review"

OUT_DIR.mkdir(parents=True, exist_ok=True)

# 允许使用的审核标签
ALLOWED_LABELS = {"正确", "有歧义", "专有名词", "地区用语"}

frames = []
batch_report = []

for i in range(1, 11):
    path = BATCH_DIR / f"review_batch_{i:02d}_suggested.tsv"

    if not path.exists():
        raise FileNotFoundError(f"缺少批次文件：{path}")

    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)

    required = {
        "分组", "来源", "原始划分", "来源ID",
        "差异数", "错误句_original", "目标句_correct",
        "判定", "备注"
    }
    missing_cols = required - set(df.columns)

    if missing_cols:
        raise ValueError(
            f"{path.name} 缺少列：{sorted(missing_cols)}"
        )

    if len(df) != 25:
        raise ValueError(
            f"{path.name} 应有 25 条，实际 {len(df)} 条"
        )

    labels = df["判定"].str.strip()
    invalid = sorted(set(labels) - ALLOWED_LABELS - {""})

    if invalid:
        raise ValueError(
            f"{path.name} 出现未知标签：{invalid}"
        )

    batch_report.append({
        "批次": i,
        "行数": len(df),
        "空标签数": int((labels == "").sum()),
        "正确": int((labels == "正确").sum()),
        "有歧义": int((labels == "有歧义").sum()),
        "专有名词": int((labels == "专有名词").sum()),
        "地区用语": int((labels == "地区用语").sum()),
    })

    # 记录批次来源，方便追踪
    df.insert(0, "审核批次", i)
    frames.append(df)

combined = pd.concat(frames, ignore_index=True)

# 检查关键文本字段
empty_text = (
    combined["错误句_original"].str.strip().eq("")
    | combined["目标句_correct"].str.strip().eq("")
)

# 检查重复文本对
pair_duplicates = combined.duplicated(
    subset=["错误句_original", "目标句_correct"],
    keep=False
)

# 汇总标签统计
label_counts = (
    combined["判定"]
    .str.strip()
    .value_counts()
    .reindex(sorted(ALLOWED_LABELS), fill_value=0)
)

# 输出新文件
combined_path = OUT_DIR / "review_250_suggested_combined.tsv"
report_path = OUT_DIR / "review_250_suggested_audit.json"

combined.to_csv(
    combined_path,
    sep="\t",
    index=False,
    encoding="utf-8"
)

audit = {
    "批次数": len(batch_report),
    "总行数": len(combined),
    "每批25条": all(x["行数"] == 25 for x in batch_report),
    "空标签数": int(combined["判定"].str.strip().eq("").sum()),
    "未知标签数": int(
        (~combined["判定"].str.strip().isin(ALLOWED_LABELS)
         & combined["判定"].str.strip().ne("")).sum()
    ),
    "空文本行数": int(empty_text.sum()),
    "重复文本对涉及行数": int(pair_duplicates.sum()),
    "标签统计": {
        str(k): int(v) for k, v in label_counts.items()
    },
    "各批统计": batch_report,
    "汇总文件": str(combined_path),
}

report_path.write_text(
    json.dumps(audit, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("===== 合并审计结果 =====")
print(json.dumps(audit, ensure_ascii=False, indent=2))
print("\n汇总表：", combined_path)
print("审计报告：", report_path)

# 基础完整性判断；重复文本对单独报告，不直接判为失败
passed = (
    len(combined) == 250
    and audit["每批25条"]
    and audit["空标签数"] == 0
    and audit["未知标签数"] == 0
    and audit["空文本行数"] == 0
)

print("\n基础完整性检查：", "通过" if passed else "未通过")

===== 合并审计结果 =====
{
  "批次数": 10,
  "总行数": 250,
  "每批25条": true,
  "空标签数": 0,
  "未知标签数": 0,
  "空文本行数": 0,
  "重复文本对涉及行数": 0,
  "标签统计": {
    "专有名词": 35,
    "地区用语": 3,
    "有歧义": 53,
    "正确": 159
  },
  "各批统计": [
    {
      "批次": 1,
      "行数": 25,
      "空标签数": 0,
      "正确": 13,
      "有歧义": 5,
      "专有名词": 5,
      "地区用语": 2
    },
    {
      "批次": 2,
      "行数": 25,
      "空标签数": 0,
      "正确": 17,
      "有歧义": 4,
      "专有名词": 3,
      "地区用语": 1
    },
    {
      "批次": 3,
      "行数": 25,
      "空标签数": 0,
      "正确": 17,
      "有歧义": 5,
      "专有名词": 3,
      "地区用语": 0
    },
    {
      "批次": 4,
      "行数": 25,
      "空标签数": 0,
      "正确": 19,
      "有歧义": 4,
      "专有名词": 2,
      "地区用语": 0
    },
    {
      "批次": 5,
      "行数": 25,
      "空标签数": 0,
      "正确": 17,
      "有歧义": 4,
      "专有名词": 4,
      "地区用语": 0
    },
    {
      "批次": 6,
      "行数": 25,
      "空标签数": 0,
      "正确": 18,
      "有歧义": 7,
      "专有名词": 0,
      "地区用语": 0
    },
    {
      "批次": 7,
      "行数"

In [45]:
from pathlib import Path
import json

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")

CANDIDATE_DIR = (
    ROOT / "data/processed/csc_candidates_simplified"
)

files = [
    CANDIDATE_DIR / "single_char.jsonl",
    CANDIDATE_DIR / "multi_2_3_char.jsonl",
    CANDIDATE_DIR / "review_4plus_char.jsonl",
]

for path in files:
    print("\n" + "=" * 80)
    print("文件：", path.name)

    if not path.exists():
        print("❌ 文件不存在：", path)
        continue

    total = 0
    first_record = None
    key_counts = {}

    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue

            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(
                    f"{path.name} 第 {line_no} 行 JSON 格式错误：{e}"
                )

            if not isinstance(obj, dict):
                raise ValueError(
                    f"{path.name} 第 {line_no} 行不是 JSON 对象"
                )

            total += 1

            if first_record is None:
                first_record = obj

            for key in obj:
                key_counts[key] = key_counts.get(key, 0) + 1

    print("有效记录数：", total)
    print("字段及字段出现次数：")
    for key, count in sorted(key_counts.items()):
        print(f"  {key}: {count}/{total}")

    if first_record is not None:
        print("\n第一条记录（仅展示，不修改）：")
        print(json.dumps(
            first_record,
            ensure_ascii=False,
            indent=2
        ))

print("\n只读检查完成：未写入、修改或删除任何候选文件。")


文件： single_char.jsonl
有效记录数： 167841
字段及字段出现次数：
  correct_text: 167841/167841
  diff_count: 167841/167841
  diff_positions: 167841/167841
  normalization: 167841/167841
  original_text: 167841/167841
  source: 167841/167841
  source_id: 167841/167841
  source_split: 167841/167841

第一条记录（仅展示，不修改）：
{
  "source": "shibing624/CSC",
  "source_split": "train",
  "source_id": "-",
  "original_text": "目前区次事件的细节还不清楚，伤亡人数也未确定。",
  "correct_text": "目前这次事件的细节还不清楚，伤亡人数也未确定。",
  "diff_count": 1,
  "diff_positions": [
    2
  ],
  "normalization": "OpenCC t2s"
}

文件： multi_2_3_char.jsonl
有效记录数： 75834
字段及字段出现次数：
  correct_text: 75834/75834
  diff_count: 75834/75834
  diff_positions: 75834/75834
  normalization: 75834/75834
  original_text: 75834/75834
  source: 75834/75834
  source_id: 75834/75834
  source_split: 75834/75834

第一条记录（仅展示，不修改）：
{
  "source": "shibing624/CSC",
  "source_split": "train",
  "source_id": "-",
  "original_text": "藉组毒虫所进行的工业间谍活动将肆虐相当一段时间",
  "correct_text": "借助毒虫所进行的工业间谍活动将肆虐相

In [46]:

from pathlib import Path
from collections import Counter
import json

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")

SOURCE_DIR = ROOT / "data/processed/csc_candidates_simplified"
OUT_DIR = ROOT / "data/processed/csc_structural_audit_v1"

# 保护已有输出，避免意外覆盖
if OUT_DIR.exists():
    raise FileExistsError(
        f"输出目录已存在，为保护已有结果，停止运行：{OUT_DIR}\n"
        "请先检查目录内容；确认需要重新运行时，再另选一个新版本目录。"
    )

INPUTS = [
    ("single_char.jsonl", "single"),
    ("multi_2_3_char.jsonl", "multi_2_3"),
    ("review_4plus_char.jsonl", "review_4plus"),
]

REQUIRED = {
    "source",
    "source_split",
    "source_id",
    "original_text",
    "correct_text",
    "diff_count",
    "diff_positions",
    "normalization",
}

def calculate_diff_positions(a, b):
    if len(a) != len(b):
        return None
    return [i for i, (x, y) in enumerate(zip(a, b)) if x != y]

def expected_bucket(diff_count):
    if diff_count == 1:
        return "single"
    if 2 <= diff_count <= 3:
        return "multi_2_3"
    if diff_count >= 4:
        return "review_4plus"
    return "invalid"

def write_jsonl(f, obj):
    f.write(json.dumps(obj, ensure_ascii=False) + "\n")

# 所有输出均为新文件
OUT_DIR.mkdir(parents=True, exist_ok=False)

valid_path = OUT_DIR / "structural_valid.jsonl"
quarantine_path = OUT_DIR / "structural_quarantine.jsonl"
report_path = OUT_DIR / "audit_report.json"

stats = {
    "input_rows": 0,
    "structural_valid_rows": 0,
    "quarantine_rows": 0,
    "error_types": Counter(),
    "by_file": {},
    "source_split_counts": Counter(),
}

# 跨文件追踪重复文本对
seen_pairs = set()

with (
    valid_path.open("w", encoding="utf-8") as vf,
    quarantine_path.open("w", encoding="utf-8") as qf,
):
    for filename, expected_group in INPUTS:
        path = SOURCE_DIR / filename

        if not path.exists():
            raise FileNotFoundError(f"找不到输入文件：{path}")

        file_stats = {
            "rows": 0,
            "valid": 0,
            "quarantined": 0,
            "error_types": Counter(),
        }

        with path.open("r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):
                if not line.strip():
                    continue

                stats["input_rows"] += 1
                file_stats["rows"] += 1
                errors = []

                try:
                    obj = json.loads(line)
                except json.JSONDecodeError:
                    obj = {
                        "_audit_source_file": filename,
                        "_audit_line": line_no,
                    }
                    errors.append("invalid_json")

                if not isinstance(obj, dict):
                    obj = {
                        "_audit_source_file": filename,
                        "_audit_line": line_no,
                    }
                    errors.append("not_json_object")

                if not errors:
                    missing = REQUIRED - set(obj.keys())
                    if missing:
                        errors.append("missing_fields")

                    original = obj.get("original_text")
                    correct = obj.get("correct_text")
                    diff_count = obj.get("diff_count")
                    diff_positions = obj.get("diff_positions")

                    if not isinstance(original, str) or not isinstance(correct, str):
                        errors.append("text_not_string")
                    elif not original.strip() or not correct.strip():
                        errors.append("empty_text")
                    elif len(original) != len(correct):
                        errors.append("length_mismatch")

                    if isinstance(original, str) and isinstance(correct, str):
                        calculated = calculate_diff_positions(original, correct)

                        if calculated is not None:
                            if not calculated:
                                errors.append("no_actual_difference")

                            if not isinstance(diff_count, int) or isinstance(diff_count, bool):
                                errors.append("invalid_diff_count")
                            elif diff_count != len(calculated):
                                errors.append("diff_count_mismatch")

                            if not isinstance(diff_positions, list):
                                errors.append("invalid_diff_positions")
                            elif diff_positions != calculated:
                                errors.append("diff_positions_mismatch")

                            if calculated:
                                actual_bucket = expected_bucket(len(calculated))
                                if actual_bucket != expected_group:
                                    errors.append("bucket_mismatch")

                    if obj.get("source") != "shibing624/CSC":
                        errors.append("unexpected_source")

                    source_split = obj.get("source_split")
                    if isinstance(source_split, str):
                        stats["source_split_counts"][source_split] += 1
                    else:
                        errors.append("invalid_source_split")

                    if not isinstance(obj.get("normalization"), str):
                        errors.append("invalid_normalization")

                    if isinstance(original, str) and isinstance(correct, str):
                        pair = (original, correct)
                        if pair in seen_pairs:
                            errors.append("duplicate_pair")
                        else:
                            seen_pairs.add(pair)

                if errors:
                    record = dict(obj)
                    record["_audit_source_file"] = filename
                    record["_audit_line"] = line_no
                    record["_audit_errors"] = sorted(set(errors))
                    write_jsonl(qf, record)

                    stats["quarantine_rows"] += 1
                    file_stats["quarantined"] += 1

                    for error in set(errors):
                        stats["error_types"][error] += 1
                        file_stats["error_types"][error] = (
                            file_stats["error_types"].get(error, 0) + 1
                        )
                else:
                    record = dict(obj)
                    record["_audit_source_file"] = filename
                    record["_audit_line"] = line_no
                    write_jsonl(vf, record)

                    stats["structural_valid_rows"] += 1
                    file_stats["valid"] += 1

        stats["by_file"][filename] = {
            **file_stats,
            "error_types": dict(file_stats["error_types"]),
        }

report = {
    "input_rows": stats["input_rows"],
    "structural_valid_rows": stats["structural_valid_rows"],
    "quarantine_rows": stats["quarantine_rows"],
    "accounting_matches": (
        stats["input_rows"]
        == stats["structural_valid_rows"] + stats["quarantine_rows"]
    ),
    "error_types": dict(stats["error_types"]),
    "source_split_counts": dict(stats["source_split_counts"]),
    "by_file": stats["by_file"],
    "outputs": {
        "structural_valid": str(valid_path),
        "quarantine": str(quarantine_path),
        "report": str(report_path),
    },
    "note": (
        "结构审计通过不代表语义正确；本报告不执行语义质量判定。"
    ),
}

report_path.write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("===== 全量结构审计 =====")
print(json.dumps(report, ensure_ascii=False, indent=2))
print("\n原始候选文件未修改。")

===== 全量结构审计 =====
{
  "input_rows": 245835,
  "structural_valid_rows": 245835,
  "quarantine_rows": 0,
  "accounting_matches": true,
  "error_types": {},
  "source_split_counts": {
    "train": 245835
  },
  "by_file": {
    "single_char.jsonl": {
      "rows": 167841,
      "valid": 167841,
      "quarantined": 0,
      "error_types": {}
    },
    "multi_2_3_char.jsonl": {
      "rows": 75834,
      "valid": 75834,
      "quarantined": 0,
      "error_types": {}
    },
    "review_4plus_char.jsonl": {
      "rows": 2160,
      "valid": 2160,
      "quarantined": 0,
      "error_types": {}
    }
  },
  "outputs": {
    "structural_valid": "/content/xime_work/xime_macbert_colab_ready/data/processed/csc_structural_audit_v1/structural_valid.jsonl",
    "quarantine": "/content/xime_work/xime_macbert_colab_ready/data/processed/csc_structural_audit_v1/structural_quarantine.jsonl",
    "report": "/content/xime_work/xime_macbert_colab_ready/data/processed/csc_structural_audit_v1/audit_report

In [47]:
from pathlib import Path
from collections import Counter
import json
import csv

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")

NEW_FILE = (
    ROOT / "data/processed/csc_structural_audit_v1"
    / "structural_valid.jsonl"
)

OLD_FILES = {
    "train": ROOT / "data/processed/train_v1.tsv",
    "valid": ROOT / "data/processed/valid_v1.tsv",
    "test": ROOT / "data/processed/test_v1.tsv",
}

# 防止覆盖旧审计结果
OUT_DIR = ROOT / "data/processed/csc_overlap_audit_v1"
if OUT_DIR.exists():
    raise FileExistsError(
        f"目录已存在，请检查后另选新目录：{OUT_DIR}"
    )
OUT_DIR.mkdir(parents=True)

def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def read_old_tsv(path):
    with path.open(encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        if not reader.fieldnames:
            raise ValueError(f"TSV 缺少表头：{path}")
        if not {"clean", "noisy"}.issubset(reader.fieldnames):
            raise ValueError(
                f"{path.name} 缺少 clean/noisy 字段："
                f"{reader.fieldnames}"
            )
        for row in reader:
            yield row["noisy"], row["clean"]

# 新数据统计
pair_counts = Counter()
noisy_counts = Counter()
clean_counts = Counter()
length_counts = Counter()
group_counts = Counter()

new_pairs = set()
new_noisy = set()
new_clean = set()

for obj in read_jsonl(NEW_FILE):
    noisy = obj["original_text"]
    clean = obj["correct_text"]

    pair_counts[(noisy, clean)] += 1
    noisy_counts[noisy] += 1
    clean_counts[clean] += 1

    new_pairs.add((noisy, clean))
    new_noisy.add(noisy)
    new_clean.add(clean)

    length_counts[len(noisy)] += 1
    group_counts[obj["_audit_source_file"]] += 1

# 旧数据集合
old_sets = {}
for split, path in OLD_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"找不到旧 split：{path}")

    pairs = set()
    noisies = set()
    cleans = set()

    for noisy, clean in read_old_tsv(path):
        pairs.add((noisy, clean))
        noisies.add(noisy)
        cleans.add(clean)

    old_sets[split] = {
        "pairs": pairs,
        "noisy": noisies,
        "clean": cleans,
    }

report = {
    "new_rows": sum(pair_counts.values()),
    "unique_pairs": len(new_pairs),
    "duplicate_pair_extra_occurrences": (
        sum(pair_counts.values()) - len(pair_counts)
    ),
    "unique_noisy_texts": len(new_noisy),
    "unique_clean_texts": len(new_clean),
    "noisy_texts_appearing_multiple_times": sum(
        n > 1 for n in noisy_counts.values()
    ),
    "clean_texts_appearing_multiple_times": sum(
        n > 1 for n in clean_counts.values()
    ),
    "source_file_counts": dict(group_counts),
    "length_distribution": {
        str(k): v for k, v in sorted(length_counts.items())
    },
    "overlap_with_old_splits": {},
}

for split, old in old_sets.items():
    report["overlap_with_old_splits"][split] = {
        "same_direction_pairs": len(new_pairs & old["pairs"]),
        "reverse_direction_pairs": len({
            (noisy, clean)
            for noisy, clean in new_pairs
            if (clean, noisy) in old["pairs"]
        }),
        "same_noisy_texts": len(new_noisy & old["noisy"]),
        "same_clean_texts": len(new_clean & old["clean"]),
        "new_noisy_equals_old_clean": len(new_noisy & old["clean"]),
        "new_clean_equals_old_noisy": len(new_clean & old["noisy"]),
    }

report_path = OUT_DIR / "overlap_report.json"
report_path.write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(json.dumps(report, ensure_ascii=False, indent=2))
print("\n审计报告：", report_path)
print("所有输入文件均只读，未修改。")

{
  "new_rows": 245835,
  "unique_pairs": 245835,
  "duplicate_pair_extra_occurrences": 0,
  "unique_noisy_texts": 245834,
  "unique_clean_texts": 240441,
  "noisy_texts_appearing_multiple_times": 1,
  "clean_texts_appearing_multiple_times": 5235,
  "source_file_counts": {
    "single_char.jsonl": 167841,
    "multi_2_3_char.jsonl": 75834,
    "review_4plus_char.jsonl": 2160
  },
  "length_distribution": {
    "3": 1,
    "4": 20,
    "5": 47,
    "6": 85,
    "7": 281,
    "8": 445,
    "9": 695,
    "10": 2535,
    "11": 2970,
    "12": 3232,
    "13": 3441,
    "14": 3674,
    "15": 3645,
    "16": 3760,
    "17": 3863,
    "18": 3971,
    "19": 3893,
    "20": 4150,
    "21": 4232,
    "22": 4098,
    "23": 3700,
    "24": 3485,
    "25": 3410,
    "26": 3128,
    "27": 3206,
    "28": 3326,
    "29": 3448,
    "30": 3627,
    "31": 3574,
    "32": 3743,
    "33": 3802,
    "34": 4030,
    "35": 4119,
    "36": 4055,
    "37": 4161,
    "38": 4255,
    "39": 4369,
    "40": 4293,
 

In [48]:

from pathlib import Path
from collections import Counter, defaultdict
import json

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
INPUT = (
    ROOT / "data/processed/csc_structural_audit_v1"
    / "structural_valid.jsonl"
)

records = []
with INPUT.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

noisy_targets = defaultdict(set)
noisy_records = defaultdict(list)
clean_sources = defaultdict(set)

for r in records:
    noisy = r["original_text"]
    clean = r["correct_text"]

    noisy_targets[noisy].add(clean)
    noisy_records[noisy].append(r)
    clean_sources[clean].add(noisy)

print("===== 1. 重复错误句检查 =====")

repeated_noisy = {
    noisy: targets
    for noisy, targets in noisy_targets.items()
    if len(targets) > 1
}

print("对应多个不同目标句的错误句数：", len(repeated_noisy))

for i, (noisy, targets) in enumerate(repeated_noisy.items(), 1):
    print(f"\n重复项 {i}")
    print("错误句：", noisy)
    print("目标句数量：", len(targets))

    for target in sorted(targets):
        print("  ->", target)

    for r in noisy_records[noisy]:
        print(
            "  来源文件:", r.get("_audit_source_file"),
            "| 差异数:", r.get("diff_count"),
            "| 来源划分:", r.get("source_split"),
            "| 来源ID:", r.get("source_id")
        )

print("\n===== 2. 长句分布 =====")

lengths = [
    (len(r["original_text"]), len(r["correct_text"]), r)
    for r in records
]

for threshold in [80, 96, 128, 160, 200]:
    count = sum(
        max(noisy_len, clean_len) > threshold
        for noisy_len, clean_len, _ in lengths
    )
    print(f"任一文本长度 > {threshold} 字符：{count}")

print("\n===== 3. 最长的 15 条样本 =====")

longest = sorted(
    lengths,
    key=lambda x: max(x[0], x[1]),
    reverse=True
)[:15]

for i, (noisy_len, clean_len, r) in enumerate(longest, 1):
    print(f"\n第 {i} 条 | noisy={noisy_len} | clean={clean_len}")
    print("分组文件：", r.get("_audit_source_file"))
    print("错误句：", r["original_text"][:350])
    print("目标句：", r["correct_text"][:350])

print("\n===== 4. 重复目标句统计 =====")

repeated_clean = {
    clean: sources
    for clean, sources in clean_sources.items()
    if len(sources) > 1
}

print("对应多个不同错误句的目标句数：", len(repeated_clean))

for i, (clean, sources) in enumerate(
    list(repeated_clean.items())[:10], 1
):
    print(f"\n示例 {i}")
    print("目标句：", clean)
    print("不同错误句数量：", len(sources))
    for noisy in list(sorted(sources))[:3]:
        print("  <-", noisy)

print("\n只读检查完成：没有修改任何文件。")

===== 1. 重复错误句检查 =====
对应多个不同目标句的错误句数： 1

重复项 1
错误句： 意味着未来平均每年要制定三步左右的法律。
目标句数量： 2
  -> 意味着未来平均每年要制定三部左右的法律。
  -> 意味着未来平均每年要制订三部左右的法律。
  来源文件: single_char.jsonl | 差异数: 1 | 来源划分: train | 来源ID: -
  来源文件: multi_2_3_char.jsonl | 差异数: 2 | 来源划分: train | 来源ID: -

===== 2. 长句分布 =====
任一文本长度 > 80 字符：8364
任一文本长度 > 96 字符：3125
任一文本长度 > 128 字符：457
任一文本长度 > 160 字符：4
任一文本长度 > 200 字符：1

===== 3. 最长的 15 条样本 =====

第 1 条 | noisy=258 | clean=258
分组文件： review_4plus_char.jsonl
错误句： ＡＭＹ你怎么样？我最近忙著念书，因为明天要考试，听说你的男朋友找到不错的工作，请你替我问他好。我真的很高兴下侗月你要来台湾，我已经安排好了，所以我要告诉你一下，新文说了下侗月不会下雨，但是热得不得了，我觉得这样的天气，适合我们去山上，我可以一边走路一边看风景，呼吸呼吸新鲜的空气。晚上我一定带你去夜市吃饭，因为在夜市有很多好吃得小吃，在台北有很多夜市，我在想到害我饿了。我记得你对太极有兴趣，所早上我们可以去公园看看，因为每侗早有很多人在那边打太极。来台湾的外国人一定参观博物馆，比方说茶博物，真得有意思，月侗月见。
目标句： ＡＭＹ你怎么样？我最近忙著念书，因为明天要考试，听说你的男朋友找到不错的工作，请你替我问他好。我真的很高兴下个月你要来台湾，我已经安排好了，所以我要告诉你一下，新闻说了下侗月不会下雨，但是热得不得了，我觉得这样的天气，适合我们去山上，我可以一边走路一边看风景，呼吸呼吸新鲜的空气。晚上我一定带你去夜市吃饭，因为在夜市有很多好吃的小吃，在台北有很多夜市，我在想到害我饿了。我记得你对太极有兴趣，所早上我们可以去公园看看，因为每个早有很多人在那边打太极。来台湾的外国人一定参观博物馆，比方说茶博物，真的有意思，月个月见。

第 2 条 | no

In [49]:

from pathlib import Path
import json

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")

print("===== 1. 搜索项目内的分词器文件 =====")

tokenizer_markers = {
    "tokenizer_config.json",
    "tokenizer.json",
    "vocab.txt",
    "special_tokens_map.json",
}

found_dirs = set()

for path in ROOT.rglob("*"):
    if path.is_file() and path.name in tokenizer_markers:
        found_dirs.add(str(path.parent))

if found_dirs:
    for d in sorted(found_dirs):
        print(d)
else:
    print("项目目录中没有找到常见分词器文件。")

print("\n===== 2. 检查已知模型目录 =====")

known_model_dirs = [
    ROOT / "models/macbert-correction-v1",
    ROOT / "models/macbert-smoke",
]

for model_dir in known_model_dirs:
    print(f"\n模型目录：{model_dir}")
    if not model_dir.exists():
        print("  不存在")
        continue

    for name in sorted(p.name for p in model_dir.iterdir()):
        print(" ", name)

print("\n===== 3. 精确检查目标冲突样本 =====")

input_file = (
    ROOT / "data/processed/csc_structural_audit_v1"
    / "structural_valid.jsonl"
)

targets_by_noisy = {}

with input_file.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        obj = json.loads(line)
        noisy = obj["original_text"]
        clean = obj["correct_text"]

        targets_by_noisy.setdefault(noisy, {})
        targets_by_noisy[noisy].setdefault(clean, [])
        targets_by_noisy[noisy][clean].append({
            "source_file": obj.get("_audit_source_file"),
            "diff_count": obj.get("diff_count"),
            "source_split": obj.get("source_split"),
            "source_id": obj.get("source_id"),
        })

conflicts = {
    noisy: targets
    for noisy, targets in targets_by_noisy.items()
    if len(targets) > 1
}

print("冲突错误句数：", len(conflicts))

for noisy, targets in conflicts.items():
    print("\n错误句：", noisy)
    for clean, records in targets.items():
        print("目标句：", clean)
        print("记录信息：", records)

print("\n只读检查完成，未修改任何文件。")

===== 1. 搜索项目内的分词器文件 =====
/content/xime_work/xime_macbert_colab_ready/models/macbert-correction-v1
/content/xime_work/xime_macbert_colab_ready/models/macbert-correction-v1/checkpoint-294
/content/xime_work/xime_macbert_colab_ready/models/macbert-correction-v1/checkpoint-882
/content/xime_work/xime_macbert_colab_ready/models/macbert-smoke
/content/xime_work/xime_macbert_colab_ready/models/macbert-smoke/checkpoint-16

===== 2. 检查已知模型目录 =====

模型目录：/content/xime_work/xime_macbert_colab_ready/models/macbert-correction-v1
  checkpoint-294
  checkpoint-882
  config.json
  eval_end_to_end_valid.json
  eval_test_v1.json
  model.safetensors
  tokenizer.json
  tokenizer_config.json
  training_args.bin

模型目录：/content/xime_work/xime_macbert_colab_ready/models/macbert-smoke
  checkpoint-16
  config.json
  eval_test_baseline.json
  model.safetensors
  tokenizer.json
  tokenizer_config.json
  training_args.bin

===== 3. 精确检查目标冲突样本 =====
冲突错误句数： 1

错误句： 意味着未来平均每年要制定三步左右的法律。
目标句： 意味着未来平均每年要制定三部左右的法律。


In [50]:

from pathlib import Path
from collections import Counter
import json
import re

from transformers import AutoTokenizer

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
MODEL_DIR = ROOT / "models/macbert-correction-v1"
INPUT_FILE = (
    ROOT / "data/processed/csc_structural_audit_v1"
    / "structural_valid.jsonl"
)

assert MODEL_DIR.exists(), f"模型目录不存在：{MODEL_DIR}"
assert (MODEL_DIR / "tokenizer.json").exists(), "找不到 tokenizer.json"
assert INPUT_FILE.exists(), f"语料文件不存在：{INPUT_FILE}"

print("===== 1. 加载旧模型自带的分词器 =====")

tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
)

config_path = MODEL_DIR / "config.json"
with config_path.open("r", encoding="utf-8") as f:
    model_config = json.load(f)

position_limit = model_config.get("max_position_embeddings")
tokenizer_limit = tokenizer.model_max_length

print("分词器类型：", type(tokenizer).__name__)
print("分词器 model_max_length：", tokenizer_limit)
print("模型 max_position_embeddings：", position_limit)
print("特殊 token：", tokenizer.num_special_tokens_to_add(pair=False))

# 如果训练脚本使用了其他长度上限，后面还要结合脚本判断。
print("\n===== 2. 查找训练脚本中的长度设置 =====")

scripts_dir = ROOT / "scripts"
patterns = re.compile(
    r"max_length|max_seq_length|MAX_LENGTH|"
    r"truncation\s*=|model_max_length",
    re.IGNORECASE,
)

matched_lines = 0

if scripts_dir.exists():
    for py_file in sorted(scripts_dir.glob("*.py")):
        try:
            lines = py_file.read_text(encoding="utf-8").splitlines()
        except Exception:
            continue

        hits = [
            (i, line.strip())
            for i, line in enumerate(lines, start=1)
            if patterns.search(line)
        ]

        if hits:
            print(f"\n文件：{py_file.relative_to(ROOT)}")
            for line_no, line in hits[:30]:
                print(f"  第 {line_no} 行：{line}")
                matched_lines += 1

if matched_lines == 0:
    print("scripts/ 顶层 Python 文件中没有找到匹配项。")

print("\n===== 3. 统计全部样本的实际 token 长度 =====")

# 分批编码，避免一次性将整个语料放入 tokenizer。
BATCH_SIZE = 128
thresholds = [64, 96, 128, 160, 256, 512]

stats = {
    "rows": 0,
    "noisy_chars": Counter(),
    "clean_chars": Counter(),
    "noisy_tokens": Counter(),
    "clean_tokens": Counter(),
    "either_tokens": Counter(),
    "noisy_over_limit": 0,
    "clean_over_limit": 0,
}

longest = []
batch = []

def process_batch(items):
    noisy_texts = [x["original_text"] for x in items]
    clean_texts = [x["correct_text"] for x in items]

    noisy_encoded = tokenizer(
        noisy_texts,
        add_special_tokens=True,
        padding=False,
        truncation=False,
    )
    clean_encoded = tokenizer(
        clean_texts,
        add_special_tokens=True,
        padding=False,
        truncation=False,
    )

    for i, obj in enumerate(items):
        noisy_len = len(noisy_encoded["input_ids"][i])
        clean_len = len(clean_encoded["input_ids"][i])

        noisy_chars = len(obj["original_text"])
        clean_chars = len(obj["correct_text"])
        either_len = max(noisy_len, clean_len)

        stats["rows"] += 1
        stats["noisy_chars"][noisy_chars] += 1
        stats["clean_chars"][clean_chars] += 1
        stats["noisy_tokens"][noisy_len] += 1
        stats["clean_tokens"][clean_len] += 1
        stats["either_tokens"][either_len] += 1

        for threshold in thresholds:
            if noisy_len > threshold:
                stats.setdefault(
                    f"noisy_gt_{threshold}", 0
                )
                stats[f"noisy_gt_{threshold}"] += 1

            if clean_len > threshold:
                stats.setdefault(
                    f"clean_gt_{threshold}", 0
                )
                stats[f"clean_gt_{threshold}"] += 1

        if position_limit and noisy_len > position_limit:
            stats["noisy_over_limit"] += 1

        if position_limit and clean_len > position_limit:
            stats["clean_over_limit"] += 1

        longest.append({
            "max_tokens": either_len,
            "noisy_tokens": noisy_len,
            "clean_tokens": clean_len,
            "noisy_chars": noisy_chars,
            "clean_chars": clean_chars,
            "source_file": obj.get("_audit_source_file", "-"),
            "source_id": obj.get("source_id", "-"),
            "noisy": obj["original_text"],
            "clean": obj["correct_text"],
        })

with INPUT_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        obj = json.loads(line)
        batch.append(obj)

        if len(batch) >= BATCH_SIZE:
            process_batch(batch)
            batch.clear()

if batch:
    process_batch(batch)

print("样本总数：", stats["rows"])

print("\n--- 超过不同 token 阈值的样本数 ---")
for threshold in thresholds:
    print(
        f"> {threshold} tokens："
        f"noisy={stats.get(f'noisy_gt_{threshold}', 0)}，"
        f"clean={stats.get(f'clean_gt_{threshold}', 0)}"
    )

print("\n--- 超过模型位置长度的样本数 ---")
print("noisy：", stats["noisy_over_limit"])
print("clean：", stats["clean_over_limit"])

print("\n===== 4. 最长的 15 条样本（按 token 数） =====")

longest.sort(key=lambda x: x["max_tokens"], reverse=True)

for rank, item in enumerate(longest[:15], start=1):
    print(
        f"\n第 {rank} 条："
        f"max_tokens={item['max_tokens']}，"
        f"noisy_tokens={item['noisy_tokens']}，"
        f"clean_tokens={item['clean_tokens']}，"
        f"字符数={item['noisy_chars']}/{item['clean_chars']}"
    )
    print("来源：", item["source_file"])
    print("错误句：", item["noisy"][:300])
    print("目标句：", item["clean"][:300])

print("\n只读审计完成：没有修改任何文件。")

===== 1. 加载旧模型自带的分词器 =====
分词器类型： BertTokenizer
分词器 model_max_length： 1000000000000000019884624838656
模型 max_position_embeddings： 512
特殊 token： 2

===== 2. 查找训练脚本中的长度设置 =====

文件：scripts/correction_labels.py
  第 102 行：def collate(examples: list[dict], pad_id: int, max_length: int | None = None) -> dict:
  第 107 行：if max_length:
  第 108 行：width = min(width, max_length)

文件：scripts/eval_correction.py
  第 141 行：if len(ids) > args.max_length - 2:

文件：scripts/eval_dual.py
  第 133 行：def eval_correction(model_dir: Path, test_file: Path, max_samples: int, max_length: int,
  第 218 行：if len(ids) > max_length - 2:
  第 302 行：args.max_length, args.batch_size, args.mask_correct_ratio,

文件：scripts/eval_end_to_end.py
  第 86 行：if len(char_ids) + 2 > args.max_length:

文件：scripts/evaluate.py
  第 14 行：ds=ds.map(lambda b:tok(b["noisy"], truncation=True, max_length=128), batched=True, remove_columns=ds.column_names)

文件：scripts/train.py
  第 26 行：return tok(batch["noisy"], truncation=True, max_length=args.ma

In [51]:

from pathlib import Path
import re
import json

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
scripts = [
    ROOT / "scripts/train_correction.py",
    ROOT / "scripts/train.py",
]

print("===== 1. 检查训练参数和长句处理逻辑 =====")

patterns = re.compile(
    r"max_length|preflight|add_argument|"
    r"truncation|mask_typo_prob|skip|filtered|"
    r"max_token_len|parse_args",
    re.IGNORECASE,
)

for path in scripts:
    print(f"\n{'=' * 24}\n文件：{path.name}\n{'=' * 24}")

    if not path.exists():
        print("文件不存在")
        continue

    lines = path.read_text(encoding="utf-8").splitlines()
    shown = set()

    for i, line in enumerate(lines):
        if patterns.search(line):
            # 展示命中行上下文，避免只看到孤立语句。
            start = max(0, i - 2)
            end = min(len(lines), i + 4)

            if any(j in shown for j in range(start, end)):
                continue

            for j in range(start, end):
                print(f"{j + 1:4d}: {lines[j]}")
                shown.add(j)

print("\n===== 2. 检查旧模型记录的训练参数 =====")

try:
    import torch

    args_file = ROOT / "models/macbert-correction-v1/training_args.bin"

    if args_file.exists():
        try:
            # 仅加载本项目已有的训练参数文件。
            args = torch.load(
                args_file,
                map_location="cpu",
                weights_only=False,
            )

            if hasattr(args, "to_dict"):
                args = args.to_dict()
            elif hasattr(args, "__dict__"):
                args = vars(args)

            if isinstance(args, dict):
                for key, value in sorted(args.items()):
                    if any(
                        k in key.lower()
                        for k in [
                            "length", "batch", "epoch",
                            "learning_rate", "gradient",
                            "fp16", "bf16", "seed",
                        ]
                    ):
                        print(f"{key}: {value}")
            else:
                print("参数文件已读取，但对象格式不是常见字典。")
        except Exception as e:
            print("无法读取训练参数：", repr(e))
    else:
        print("没有找到 training_args.bin")
except Exception as e:
    print("检查失败：", repr(e))

print("\n只读检查完成，没有修改文件。")

===== 1. 检查训练参数和长句处理逻辑 =====

文件：train_correction.py
  63: 
  64: 
  65: def preflight(tok, rows: list[tuple[str, str]], max_length: int, mask_typo_prob: float,
  66:               mask_correct_prob: float, seed: int, limit: int = 0):
  67:     """逐条对齐检查并构造样本；返回 (examples, stats)。"""
  68:     special_ids = set(tok.all_special_ids)
  74:     examples = []
  75:     stats = {
  76:         "rows": len(rows), "used": 0, "skipped_length_mismatch": 0,
  77:         "skipped_empty_typo": 0, "truncated": 0, "max_token_len": 0,
  78:         "total_typo_positions": 0, "total_masked_typo": 0,
  79:         "total_masked_correct": 0, "special_token_leaks": 0,
  82:         if limit and stats["used"] >= limit:
  83:             break
  84:         c = tok(clean, add_special_tokens=True, truncation=True, max_length=max_length)["input_ids"]
  85:         n = tok(noisy, add_special_tokens=True, truncation=True, max_length=max_length)["input_ids"]
  86:         if len(c) >= max_length or len(n) >= m

In [52]:

from pathlib import Path
from collections import Counter
import json

from transformers import AutoTokenizer

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
MODEL_DIR = ROOT / "models/macbert-correction-v1"
INPUT_FILE = (
    ROOT / "data/processed/csc_structural_audit_v1"
    / "structural_valid.jsonl"
)

tok = AutoTokenizer.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
)

limits = [128, 160]
stats = {
    limit: Counter()
    for limit in limits
}

examples = {
    limit: []
    for limit in limits
}

def char_diff_positions(a, b):
    if len(a) != len(b):
        return None
    return [i for i, (x, y) in enumerate(zip(a, b)) if x != y]

with INPUT_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        obj = json.loads(line)
        noisy = obj["original_text"]
        clean = obj["correct_text"]

        diffs = char_diff_positions(noisy, clean)
        if diffs is None:
            # 理论上结构审计已排除此情况，仍保留防御性检查。
            for limit in limits:
                stats[limit]["unequal_char_length"] += 1
            continue

        # 不加特殊 token，便于分析正文的 token 位置。
        noisy_ids = tok(
            noisy,
            add_special_tokens=False,
            truncation=False,
        )["input_ids"]

        clean_ids = tok(
            clean,
            add_special_tokens=False,
            truncation=False,
        )["input_ids"]

        for limit in limits:
            # 两个特殊 token：[CLS]、[SEP]
            body_capacity = limit - tok.num_special_tokens_to_add(pair=False)

            s = stats[limit]
            s["rows"] += 1

            if len(noisy_ids) > body_capacity or len(clean_ids) > body_capacity:
                s["would_truncate"] += 1

            # 检查原始差异字符是否位于保留区域之后。
            # 这里按字符位置检查，避免把 token 下标误当成字符下标。
            # 对标准中文 BERT 的逐字分词，通常是一字一 token；
            # 拉丁字母、数字等情况可能不完全如此。
            truncated_chars = max(0, len(noisy) - body_capacity)

            if any(pos >= truncated_chars for pos in diffs):
                s["diff_in_retained_prefix"] += 1
            else:
                s["all_diffs_beyond_prefix"] += 1

            # 实际截断后的 token 序列是否仍然长度一致
            n_kept = noisy_ids[:body_capacity]
            c_kept = clean_ids[:body_capacity]

            if len(n_kept) != len(c_kept):
                s["token_length_mismatch_after_truncation"] += 1

            if len(examples[limit]) < 5 and (
                len(noisy_ids) > body_capacity
                and any(pos >= truncated_chars for pos in diffs)
            ):
                examples[limit].append({
                    "chars": len(noisy),
                    "noisy_tokens": len(noisy_ids),
                    "clean_tokens": len(clean_ids),
                    "diff_positions": diffs,
                    "noisy": noisy[:220],
                    "clean": clean[:220],
                })

print("===== 截断风险比较 =====")

for limit in limits:
    s = stats[limit]
    print(f"\n--- max_length={limit} ---")
    print("样本总数：", s["rows"])
    print("会发生 token 截断：", s["would_truncate"])
    print("差异字符仍在保留前缀内：", s["diff_in_retained_prefix"])
    print("所有差异字符均在截断区域：", s["all_diffs_beyond_prefix"])
    print("截断后 token 长度不一致：",
          s["token_length_mismatch_after_truncation"])

    for i, item in enumerate(examples[limit], start=1):
        print(f"\n风险样本 {i}")
        print("字符数：", item["chars"])
        print("token 数：", item["noisy_tokens"], "/", item["clean_tokens"])
        print("字符差异位置（从 0 开始）：", item["diff_positions"])
        print("错误句：", item["noisy"])
        print("目标句：", item["clean"])

print("\n只读检查完成：没有修改任何文件。")

===== 截断风险比较 =====

--- max_length=128 ---
样本总数： 245835
会发生 token 截断： 505
差异字符仍在保留前缀内： 245829
所有差异字符均在截断区域： 6
截断后 token 长度不一致： 0

风险样本 1
字符数： 137
token 数： 137 / 137
字符差异位置（从 0 开始）： [49]
错误句： 所罗门美邦公司的分析师狄克里门特表示，联准会在几周前才突如其来降息之后，此时再降息半个百分点虽有点习怪，但是目前的情况确实很有必要继续采取积极行动，而该公司也预期联准会将这么做他指出，在消费者信心衰退，又无严重通货膨胀疑虑的情况下，联准会的决策官员有很大的空间可以放宽利率。
目标句： 所罗门美邦公司的分析师狄克里门特表示，联准会在几周前才突如其来降息之后，此时再降息半个百分点虽有点奇怪，但是目前的情况确实很有必要继续采取积极行动，而该公司也预期联准会将这么做他指出，在消费者信心衰退，又无严重通货膨胀疑虑的情况下，联准会的决策官员有很大的空间可以放宽利率。

风险样本 2
字符数： 140
token 数： 140 / 140
字符差异位置（从 0 开始）： [92]
错误句： 事情正在进展，中共方面告诉我们，他们希望在年底以前解决这些问题。拉米在和欧洲联盟轮值主席法国总统席哈克会晤以后表示，我们和中共朝这个方面正运作得很顺利。拉米的发言人说，最近一项重大的发产是有关配销的问题，这对欧洲企业在中国大陆这个广大市场发展大型卖场、百货公司以及仓储是很重要的。
目标句： 事情正在进展，中共方面告诉我们，他们希望在年底以前解决这些问题。拉米在和欧洲联盟轮值主席法国总统席哈克会晤以后表示，我们和中共朝这个方面正运作得很顺利。拉米的发言人说，最近一项重大的发展是有关配销的问题，这对欧洲企业在中国大陆这个广大市场发展大型卖场、百货公司以及仓储是很重要的。

风险样本 3
字符数： 139
token 数： 139 / 139
字符差异位置（从 0 开始）： [53]
错误句： 小泉告诉竞争对手，借由将规模庞大的日本邮政事业民营化，将可改变日本。小泉说：有人指称我是一个怪胎，甚至更难停说法，但我认为我是拥有常识的言行举止温和者。执政党和反对党一直对邮政事业员工的要求言听计从，令我深感愤怒。我一直想要改变这种架

In [53]:
from pathlib import Path
from collections import Counter
from transformers import AutoTokenizer
import json
import itertools

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
DATA = ROOT / "data/processed/csc_structural_audit_v1/structural_valid.jsonl"
MODEL = ROOT / "models/macbert-correction-v1"

tok = AutoTokenizer.from_pretrained(
    str(MODEL),
    use_fast=True,
    local_files_only=True,
)

print("Tokenizer:", type(tok).__name__)
print("支持 offset_mapping:", tok.is_fast)

if not tok.is_fast:
    raise RuntimeError("当前 tokenizer 不支持 offset_mapping，请先检查 tokenizer 文件。")

if not DATA.exists():
    raise FileNotFoundError(DATA)

def char_diff_positions(a, b):
    if len(a) != len(b):
        return None
    return [i for i, (x, y) in enumerate(zip(a, b)) if x != y]

def token_index_for_char(offsets, pos):
    # 返回覆盖字符位置 pos 的 token 下标；未找到则返回 None
    for i, (start, end) in enumerate(offsets):
        if start <= pos < end:
            return i
    return None

limits = (128, 160)
stats = {
    limit: Counter() for limit in limits
}
examples = {
    limit: [] for limit in limits
}

# 分批处理，避免一次性把全部 token 和 offset 放入内存
batch_size = 256

with DATA.open("r", encoding="utf-8") as f:
    while True:
        batch = list(itertools.islice(f, batch_size))
        if not batch:
            break

        rows = [json.loads(line) for line in batch]
        texts = []

        for row in rows:
            noisy = row.get("noisy", row.get("original_text"))
            clean = row.get("clean", row.get("correct_text"))

            if not isinstance(noisy, str) or not isinstance(clean, str):
                raise ValueError(
                    f"找不到 noisy/clean 文本字段，实际字段：{list(row.keys())}"
                )
            if len(noisy) != len(clean):
                raise ValueError("发现长度不一致样本，请检查结构审计输入。")

            texts.extend([noisy, clean])

        enc = tok(
            texts,
            add_special_tokens=False,
            truncation=False,
            return_offsets_mapping=True,
        )

        for j, row in enumerate(rows):
            noisy = row.get("noisy", row.get("original_text"))
            clean = row.get("clean", row.get("correct_text"))
            diffs = char_diff_positions(noisy, clean)

            n_idx = j * 2
            c_idx = n_idx + 1
            n_offsets = enc["offset_mapping"][n_idx]
            c_offsets = enc["offset_mapping"][c_idx]
            n_len = len(enc["input_ids"][n_idx])
            c_len = len(enc["input_ids"][c_idx])

            for limit in limits:
                s = stats[limit]
                capacity = limit - tok.num_special_tokens_to_add(pair=False)

                s["rows"] += 1
                if n_len > capacity or c_len > capacity:
                    s["would_truncate"] += 1

                if n_len != c_len:
                    s["raw_token_length_mismatch"] += 1

                kept_n = min(n_len, capacity)
                kept_c = min(c_len, capacity)
                if kept_n != kept_c:
                    s["truncated_token_length_mismatch"] += 1

                # 判断每个差异字符是否在 noisy 和 clean 中都被保留
                retained_flags = []
                for pos in diffs:
                    ni = token_index_for_char(n_offsets, pos)
                    ci = token_index_for_char(c_offsets, pos)

                    retained = (
                        ni is not None and ci is not None
                        and ni < capacity and ci < capacity
                    )
                    retained_flags.append(retained)

                has_retained_diff = any(retained_flags)
                has_cut_diff = not all(retained_flags)

                if has_retained_diff:
                    s["has_diff_retained_in_both"] += 1

                if n_len > capacity or c_len > capacity:
                    if not has_retained_diff:
                        s["all_diffs_outside_retained_prefix"] += 1
                    elif has_cut_diff:
                        s["mixed_retained_and_cut_diffs"] += 1

                    if has_cut_diff and len(examples[limit]) < 5:
                        examples[limit].append({
                            "noisy": noisy,
                            "clean": clean,
                            "diff_positions": diffs,
                            "token_lengths": (n_len, c_len),
                            "capacity": capacity,
                            "retained_flags": retained_flags,
                        })

for limit in limits:
    print(f"\n===== max_length={limit}：偏移映射审计 =====")
    for key, value in stats[limit].items():
        print(f"{key}: {value}")

    print("\n截断且存在差异字符未被共同保留的样本示例：")
    for i, ex in enumerate(examples[limit], 1):
        print(f"\n示例 {i}")
        print("token 长度：", ex["token_lengths"])
        print("保留 token 容量：", ex["capacity"])
        print("差异字符位置：", ex["diff_positions"])
        print("各差异是否在两侧都保留：", ex["retained_flags"])
        print("错误句：", ex["noisy"])
        print("目标句：", ex["clean"])

Tokenizer: BertTokenizer
支持 offset_mapping: True

===== max_length=128：偏移映射审计 =====
rows: 245835
has_diff_retained_in_both: 245823
would_truncate: 505
all_diffs_outside_retained_prefix: 12
mixed_retained_and_cut_diffs: 42

截断且存在差异字符未被共同保留的样本示例：

示例 1
token 长度： (136, 136)
保留 token 容量： 126
差异字符位置： [134]
各差异是否在两侧都保留： [False]
错误句： 公安迅速采取镇压行动逮捕冒出的成群法轮功学员，许多法轮功学员被按倒在地上，公安以脚踢其头脸，然后把他们拖走。至少有一名妇女头上鲜血直流，但公安并未伸出援手，清洁人员迅速前来将地上血迹清洗干净。数名紧紧抱住母亲的幼童亦一齐被押走，广场上到处散置脱落的鞋子，甚至还有一副眼睛。
目标句： 公安迅速采取镇压行动逮捕冒出的成群法轮功学员，许多法轮功学员被按倒在地上，公安以脚踢其头脸，然后把他们拖走。至少有一名妇女头上鲜血直流，但公安并未伸出援手，清洁人员迅速前来将地上血迹清洗干净。数名紧紧抱住母亲的幼童亦一齐被押走，广场上到处散置脱落的鞋子，甚至还有一副眼镜。

示例 2
token 长度： (136, 136)
保留 token 容量： 126
差异字符位置： [133]
各差异是否在两侧都保留： [False]
错误句： 空中巴士执行副总裁莱立表示，我们知道几年后，机舱内连线一定会成为不可或缺的服务，对我们的航空公司客户是如此，对他们最重视的乘客---来往频繁的商务旅客也是如此。我们看出电子邮件是机舱内连线最重要的一个因素，由于的开发，现在在机舱内收发电子邮件将会和在家里或办公室一样旁便。
目标句： 空中巴士执行副总裁莱立表示，我们知道几年后，机舱内连线一定会成为不可或缺的服务，对我们的航空公司客户是如此，对他们最重视的乘客---来往频繁的商务旅客也是如此。我们看出电子邮件是机舱内连线最重要的一个因素，由于的开发，现在在机舱内收发电子邮件将会和在家里或办公室一样方便。

示例 3
token 长度： (133, 133)
保留

In [54]:
from pathlib import Path
import ast

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
TARGETS = [
    ROOT / "scripts/correction_labels.py",
    ROOT / "scripts/train_correction.py",
]

wanted = {
    "align_pair",
    "build_example",
    "collate",
    "preflight",
}

for path in TARGETS:
    print("\n" + "=" * 80)
    print("文件：", path)

    if not path.exists():
        print("文件不存在")
        continue

    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    found = False
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            if node.name in wanted:
                found = True
                start = node.lineno
                end = node.end_lineno
                lines = source.splitlines()

                print(f"\n--- {node.name}（第 {start}–{end} 行）---")
                print("\n".join(lines[start - 1:end]))

    if not found:
        print("没有找到目标函数，请检查文件中的实际函数名称。")


文件： /content/xime_work/xime_macbert_colab_ready/scripts/correction_labels.py

--- align_pair（第 29–35 行）---
def align_pair(clean_ids: list[int], noisy_ids: list[int]) -> dict:
    """检查两个序列能否逐位对齐；不能时报出原因。"""
    if len(clean_ids) != len(noisy_ids):
        return {"ok": False, "reason": f"长度不一致：clean={len(clean_ids)} noisy={len(noisy_ids)}",
                "typo_positions": []}
    typo = [i for i, (c, n) in enumerate(zip(clean_ids, noisy_ids)) if c != n]
    return {"ok": True, "reason": "", "typo_positions": typo}

--- build_example（第 38–99 行）---
def build_example(clean_ids: list[int], noisy_ids: list[int], mask_id: int,
                  rng: random.Random, mask_typo_prob: float = 0.8,
                  mask_correct_prob: float = 0.15,
                  special_ids: set[int] | None = None) -> dict | None:
    """构造一条训练样本。

    参数
    ----
    clean_ids / noisy_ids : 同一句的两种 token id（长度必须一致）
    mask_id               : [MASK] 的 id
    rng                   : random.Random 实例（保证可复现）
 

In [56]:
from pathlib import Path
from transformers import AutoTokenizer
from collections import Counter
import itertools
import json

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")
DATA = ROOT / "data/processed/csc_structural_audit_v1/structural_valid.jsonl"
MODEL = ROOT / "models/macbert-correction-v1"

tok = AutoTokenizer.from_pretrained(
    str(MODEL),
    use_fast=True,
    local_files_only=True,
)

assert tok.is_fast, "需要支持 offset_mapping 的 fast tokenizer"

MAX_LENGTHS = (128, 160)
BATCH_PAIRS = 128

stats = Counter()
examples = []

def get_text(row, primary, fallback):
    value = row.get(primary, row.get(fallback))
    if not isinstance(value, str):
        raise ValueError(f"缺少文本字段：{primary}/{fallback}")
    return value

with DATA.open("r", encoding="utf-8") as f:
    while True:
        raw_batch = list(itertools.islice(f, BATCH_PAIRS))
        if not raw_batch:
            break

        rows = [json.loads(line) for line in raw_batch]
        noisy_list = [
            get_text(r, "noisy", "original_text") for r in rows
        ]
        clean_list = [
            get_text(r, "clean", "correct_text") for r in rows
        ]

        if any(len(n) != len(c) for n, c in zip(noisy_list, clean_list)):
            raise ValueError("发现字符长度不一致的文本对")

        texts = []
        for noisy, clean in zip(noisy_list, clean_list):
            texts.extend([noisy, clean])

        enc = tok(
            texts,
            add_special_tokens=True,
            truncation=False,
            return_offsets_mapping=True,
        )

        for i, (noisy, clean) in enumerate(zip(noisy_list, clean_list)):
            n_idx, c_idx = 2 * i, 2 * i + 1

            n_ids = enc["input_ids"][n_idx]
            c_ids = enc["input_ids"][c_idx]
            n_offsets = [tuple(x) for x in enc["offset_mapping"][n_idx]]
            c_offsets = [tuple(x) for x in enc["offset_mapping"][c_idx]]

            stats["rows"] += 1

            same_length = len(n_ids) == len(c_ids)
            same_offsets = n_offsets == c_offsets

            if not same_length:
                stats["full_token_length_mismatch"] += 1

            if same_offsets:
                stats["same_token_boundaries"] += 1
            else:
                stats["different_token_boundaries"] += 1

            # 逐 token 下标比较是否安全，至少要求两侧 token 字符范围相同
            if same_length and same_offsets:
                stats["same_length_and_boundaries"] += 1

            # 统计每种 max_length 下的截断情况
            for limit in MAX_LENGTHS:
                capacity = limit - tok.num_special_tokens_to_add(pair=False)
                key = f"max{limit}_"

                if len(n_ids) > limit or len(c_ids) > limit:
                    stats[key + "would_truncate"] += 1

                # 只有边界一致时，才统计截断后仍被保留的差异 token
                if same_length and same_offsets:
                    diffs = [
                        p for p, (a, b) in enumerate(zip(n_ids, c_ids))
                        if a != b
                    ]
                    # 只统计正文 token；特殊 token 不作为错字
                    usable_diffs = [
                        p for p in diffs
                        if n_ids[p] not in set(tok.all_special_ids)
                        and c_ids[p] not in set(tok.all_special_ids)
                    ]

                    cut_diffs = [p for p in usable_diffs if p >= capacity]
                    kept_diffs = [p for p in usable_diffs if p < capacity]

                    if cut_diffs:
                        stats[key + "rows_with_cut_diff_tokens"] += 1
                    if usable_diffs and not kept_diffs:
                        stats[key + "rows_all_diff_tokens_cut"] += 1

            if not (same_length and same_offsets) and len(examples) < 10:
                examples.append({
                    "noisy": noisy,
                    "clean": clean,
                    "noisy_token_count": len(n_ids),
                    "clean_token_count": len(c_ids),
                    "noisy_offsets": n_offsets[:20],
                    "clean_offsets": c_offsets[:20],
                })

print("===== 全量 token 边界审计 =====")
for key, value in stats.items():
    print(f"{key}: {value}")

print("\n===== 最多 10 条边界不一致示例 =====")
for i, ex in enumerate(examples, 1):
    print(f"\n示例 {i}")
    print("token 数：", ex["noisy_token_count"], "/", ex["clean_token_count"])
    print("错误句：", ex["noisy"])
    print("目标句：", ex["clean"])
    print("错误句 token offsets 前 20 个：", ex["noisy_offsets"])
    print("目标句 token offsets 前 20 个：", ex["clean_offsets"])

===== 全量 token 边界审计 =====
rows: 245835
same_token_boundaries: 245835
same_length_and_boundaries: 245835
max128_would_truncate: 505
max128_rows_with_cut_diff_tokens: 64
max128_rows_all_diff_tokens_cut: 16
max160_would_truncate: 4
max160_rows_with_cut_diff_tokens: 2

===== 最多 10 条边界不一致示例 =====


In [57]:
%cd /content/xime_work/xime_macbert_colab_ready

!python scripts/train_correction.py --help

/content/xime_work/xime_macbert_colab_ready
usage: train_correction.py [-h] [--base BASE] --train-file TRAIN_FILE
                           --valid-file VALID_FILE [--output-dir OUTPUT_DIR]
                           [--epochs EPOCHS] [--batch-size BATCH_SIZE]
                           [--grad-accum GRAD_ACCUM] [--lr LR]
                           [--warmup-ratio WARMUP_RATIO]
                           [--max-length MAX_LENGTH]
                           [--max-train-samples MAX_TRAIN_SAMPLES]
                           [--mask-typo-prob MASK_TYPO_PROB]
                           [--mask-correct-prob MASK_CORRECT_PROB]
                           [--seed SEED] [--fp16] [--preflight-only]

中文纠错训练（clean 作为标签，固定 MacBERT MLM 头）

options:
  -h, --help            show this help message and exit
  --base BASE
  --train-file TRAIN_FILE
  --valid-file VALID_FILE
  --output-dir OUTPUT_DIR
  --epochs EPOCHS
  --batch-size BATCH_SIZE
  --grad-accum GRAD_ACCUM
  --lr LR
  --warmup-ratio WARMUP_RA

In [58]:
from pathlib import Path

ROOT = Path("/content/xime_work/xime_macbert_colab_ready")

for name in ("train_v1.tsv", "valid_v1.tsv", "test_v1.tsv"):
    matches = list(ROOT.rglob(name))
    print(f"\n{name}:")
    if matches:
        for p in matches:
            print(" ", p)
    else:
        print("  未找到")


train_v1.tsv:
  /content/xime_work/xime_macbert_colab_ready/data/processed/train_v1.tsv

valid_v1.tsv:
  /content/xime_work/xime_macbert_colab_ready/data/processed/valid_v1.tsv

test_v1.tsv:
  /content/xime_work/xime_macbert_colab_ready/data/processed/test_v1.tsv


In [59]:
%cd /content/xime_work/xime_macbert_colab_ready

!python scripts/train_correction.py \
  --base models/macbert-correction-v1 \
  --train-file data/processed/train_v1.tsv \
  --valid-file data/processed/valid_v1.tsv \
  --output-dir reports/preflight_old_v1_len160 \
  --max-length 160 \
  --preflight-only

/content/xime_work/xime_macbert_colab_ready
中文纠错训练（对齐标签监督）：input=noisy，label=clean
这不是普通 MLM：损失的标签来自 clean 列，而非输入自身。
训练文件 train_v1.tsv: 4691 条；验证文件 valid_v1.tsv: 583 条
特殊 token：{'[UNK]': 100, '[SEP]': 102, '[PAD]': 0, '[CLS]': 101, '[MASK]': 103}
[MASK] id = 103；词表大小 = 21128
[train] 读取 4691，可用 4691，长度不一致跳过 0，无错字 482，截断 0
[train] 错字位置 4209 个（平均 0.90/句），其中被 mask 3418；抽查正确位置 3733；最长 token 6
[valid] 读取 583，可用 583，长度不一致跳过 0，无错字 67，截断 0
[valid] 错字位置 516 个（平均 0.89/句），其中被 mask 409；抽查正确位置 474；最长 token 6
[检查] 标签中特殊 token 泄漏：0（应恒为 0）
[样例] 输入: [CLS] [UNK] [MASK] [SEP]
       标签: - - 人 -
[PASS] 预检通过：tokenizer 对齐、特殊 token、截断与标签口径均正常。
--preflight-only 已指定：不加载模型、不训练，退出。


In [60]:
%cd /content/xime_work/xime_macbert_colab_ready

import json
from pathlib import Path

path = Path("data/processed/csc_structural_audit_v1/structural_valid.jsonl")

print("文件存在：", path.exists())
print("文件大小：", f"{path.stat().st_size:,} bytes" if path.exists() else "无")

with path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        item = json.loads(line)
        print(f"\n===== 样本 {i + 1} =====")
        print("字段：", list(item.keys()))
        print(json.dumps(item, ensure_ascii=False, indent=2))

/content/xime_work/xime_macbert_colab_ready
文件存在： True
文件大小： 124,981,477 bytes

===== 样本 1 =====
字段： ['source', 'source_split', 'source_id', 'original_text', 'correct_text', 'diff_count', 'diff_positions', 'normalization', '_audit_source_file', '_audit_line']
{
  "source": "shibing624/CSC",
  "source_split": "train",
  "source_id": "-",
  "original_text": "目前区次事件的细节还不清楚，伤亡人数也未确定。",
  "correct_text": "目前这次事件的细节还不清楚，伤亡人数也未确定。",
  "diff_count": 1,
  "diff_positions": [
    2
  ],
  "normalization": "OpenCC t2s",
  "_audit_source_file": "single_char.jsonl",
  "_audit_line": 1
}

===== 样本 2 =====
字段： ['source', 'source_split', 'source_id', 'original_text', 'correct_text', 'diff_count', 'diff_positions', 'normalization', '_audit_source_file', '_audit_line']
{
  "source": "shibing624/CSC",
  "source_split": "train",
  "source_id": "-",
  "original_text": "报导中并未说明出口量，但据引述药厂主管的话指出，每一种药物最大的庄度出口量都达到十二吨之谱。",
  "correct_text": "报导中并未说明出口量，但据引述药厂主管的话指出，每一种药物最大的年度出口量都达到十二吨之谱。",
  "diff_count": 1,
  "

In [61]:

%cd /content/xime_work/xime_macbert_colab_ready

import csv
import json
import hashlib
from collections import Counter, defaultdict
from pathlib import Path

# ---------- 1. 输入与输出 ----------
src = Path("data/processed/csc_structural_audit_v1/structural_valid.jsonl")
out_dir = Path("data/processed/csc_split_v1")

if not src.exists():
    raise FileNotFoundError(f"找不到输入文件：{src}")

# 防止重复运行时覆盖已有结果
if out_dir.exists():
    raise FileExistsError(
        f"输出目录已存在：{out_dir}\n"
        "为保护已有结果，本次没有覆盖任何文件。"
        "请先检查该目录，确认后再决定是否使用新版本目录。"
    )

out_dir.mkdir(parents=True, exist_ok=False)

# 之前审计发现的同一错误句对应两个不同目标句
ambiguous_noisy = "意味着未来平均每年要制定三步左右的法律。"

# ---------- 2. 读取并筛选 ----------
selected = []
excluded_review = 0
excluded_ambiguous = 0
excluded_other = 0

with src.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        row = json.loads(line)

        noisy = row["original_text"]
        clean = row["correct_text"]
        diff_count = row["diff_count"]

        # 隔离已知目标冲突的错误句
        if noisy == ambiguous_noisy:
            excluded_ambiguous += 1
            continue

        # 只纳入 1–3 字差异
        if diff_count not in (1, 2, 3):
            if diff_count >= 4:
                excluded_review += 1
            else:
                excluded_other += 1
            continue

        # 再做基础一致性检查
        if not noisy or not clean or len(noisy) != len(clean):
            raise ValueError(f"第 {line_no} 行文本为空或长度不一致")

        actual_positions = [
            i for i, (a, b) in enumerate(zip(noisy, clean))
            if a != b
        ]

        if len(actual_positions) != diff_count:
            raise ValueError(
                f"第 {line_no} 行 diff_count 不一致："
                f"记录={diff_count}，实际={len(actual_positions)}"
            )

        if actual_positions != row["diff_positions"]:
            raise ValueError(
                f"第 {line_no} 行 diff_positions 与文本不一致"
            )

        selected.append({
            "noisy": noisy,
            "clean": clean,
            "diff_count": diff_count,
            "bucket": row["_audit_source_file"],
        })

# ---------- 3. 防止重复文本对和错误句冲突 ----------
pair_to_row = {}
noisy_to_cleans = defaultdict(set)

for row in selected:
    pair = (row["noisy"], row["clean"])

    if pair in pair_to_row:
        raise ValueError(f"发现重复文本对：{pair}")

    pair_to_row[pair] = row
    noisy_to_cleans[row["noisy"]].add(row["clean"])

conflicts = {
    noisy: cleans
    for noisy, cleans in noisy_to_cleans.items()
    if len(cleans) > 1
}

if conflicts:
    raise ValueError(
        "筛选后仍有同一错误句对应多个目标句，停止划分："
        + repr(list(conflicts.items())[:5])
    )

# ---------- 4. 按 clean 分组，确定性划分 ----------
# 相同 clean 文本始终进入同一个集合。
# 使用固定哈希，不依赖运行时随机状态。
train_rows = []
valid_rows = []

for row in selected:
    clean = row["clean"]
    digest = hashlib.sha256(
        ("xime-csc-split-v1|" + clean).encode("utf-8")
    ).hexdigest()

    bucket = int(digest[:8], 16) % 100

    if bucket < 5:
        valid_rows.append(row)
    else:
        train_rows.append(row)

# ---------- 5. 写出 TSV：列名兼容训练脚本 ----------
def write_tsv(path, rows):
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["clean", "noisy"],
            delimiter="\t",
            lineterminator="\n",
        )
        writer.writeheader()
        for row in rows:
            writer.writerow({
                "clean": row["clean"],
                "noisy": row["noisy"],
            })

train_path = out_dir / "train.tsv"
valid_path = out_dir / "valid.tsv"

write_tsv(train_path, train_rows)
write_tsv(valid_path, valid_rows)

# ---------- 6. 回读验证输出 ----------
def read_tsv(path):
    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        if reader.fieldnames != ["clean", "noisy"]:
            raise ValueError(f"TSV 列名不正确：{reader.fieldnames}")
        rows = list(reader)
    return rows

train_check = read_tsv(train_path)
valid_check = read_tsv(valid_path)

train_clean = {r["clean"] for r in train_check}
valid_clean = {r["clean"] for r in valid_check}
train_noisy = {r["noisy"] for r in train_check}
valid_noisy = {r["noisy"] for r in valid_check}

pair_overlap = (
    {(r["noisy"], r["clean"]) for r in train_check}
    & {(r["noisy"], r["clean"]) for r in valid_check}
)

print("=" * 60)
print("CSC 新数据集划分报告")
print("=" * 60)
print("输入总行数：", sum(1 for _ in src.open("r", encoding="utf-8")))
print("筛选后总行数：", len(selected))
print("排除 4 字及以上差异：", excluded_review)
print("排除已知歧义错误句：", excluded_ambiguous)
print("排除其他异常：", excluded_other)
print()
print("训练集：", len(train_check))
print("验证集：", len(valid_check))
print("实际验证集比例：", round(
    len(valid_check) / max(len(selected), 1) * 100, 2
), "%")
print()
print("训练集 clean 数：", len(train_clean))
print("验证集 clean 数：", len(valid_clean))
print("跨集合 clean 重叠：", len(train_clean & valid_clean))
print("跨集合 noisy 重叠：", len(train_noisy & valid_noisy))
print("跨集合重复文本对：", len(pair_overlap))
print("训练集差异数分布：", dict(Counter(r["diff_count"] for r in train_rows)))
print("验证集差异数分布：", dict(Counter(r["diff_count"] for r in valid_rows)))
print("输出文件：", train_path, valid_path)

assert train_check and valid_check, "训练集或验证集为空"
assert not (train_clean & valid_clean), "发现 clean 跨集合泄漏"
assert not (train_noisy & valid_noisy), "发现 noisy 跨集合重叠"
assert not pair_overlap, "发现重复文本对跨集合重叠"

print("\nPASS：输出可回读，集合间文本检查通过。")

/content/xime_work/xime_macbert_colab_ready
CSC 新数据集划分报告
输入总行数： 245835
筛选后总行数： 243673
排除 4 字及以上差异： 2160
排除已知歧义错误句： 2
排除其他异常： 0

训练集： 231371
验证集： 12302
实际验证集比例： 5.05 %

训练集 clean 数： 226374
验证集 clean 数： 12030
跨集合 clean 重叠： 0
跨集合 noisy 重叠： 0
跨集合重复文本对： 0
训练集差异数分布： {1: 159282, 2: 54984, 3: 17105}
验证集差异数分布： {1: 8558, 3: 890, 2: 2854}
输出文件： data/processed/csc_split_v1/train.tsv data/processed/csc_split_v1/valid.tsv

PASS：输出可回读，集合间文本检查通过。


In [62]:

%cd /content/xime_work/xime_macbert_colab_ready

!python scripts/train_correction.py \
  --base models/macbert-correction-v1 \
  --train-file data/processed/csc_split_v1/train.tsv \
  --valid-file data/processed/csc_split_v1/valid.tsv \
  --output-dir reports/preflight_csc_split_v1_len160 \
  --max-length 160 \
  --preflight-only

/content/xime_work/xime_macbert_colab_ready
中文纠错训练（对齐标签监督）：input=noisy，label=clean
这不是普通 MLM：损失的标签来自 clean 列，而非输入自身。
训练文件 train.tsv: 231371 条；验证文件 valid.tsv: 12302 条
特殊 token：{'[UNK]': 100, '[SEP]': 102, '[PAD]': 0, '[CLS]': 101, '[MASK]': 103}
[MASK] id = 103；词表大小 = 21128
[train] 读取 231371，可用 231371，长度不一致跳过 0，无错字 3676，截断 2
[train] 错字位置 313228 个（平均 1.35/句），其中被 mask 250762；抽查正确位置 1143571；最长 token 160
[valid] 读取 12302，可用 12302，长度不一致跳过 0，无错字 178，截断 0
[valid] 错字位置 16557 个（平均 1.35/句），其中被 mask 13202；抽查正确位置 60687；最长 token 141
[WARN] 存在达到 160 上限的样本，长句会被截断（错字可能落在截断区）
[检查] 标签中特殊 token 泄漏：0（应恒为 0）
[样例] 输入: [CLS] 报 [MASK] 中 并 未 说 [MASK] 出 [MASK] 量 ， 但 据 引 [MASK] [MASK] 厂 [MASK] 管 的 话 指 出 ， 每 一 种 药 物 最 大 的 [MASK] 度 出 [MASK] 量 都 达 到 十 二 吨 之 谱 。 [SEP]
       标签: - - 导 - - - - 明 - 口 - - - - - 述 药 - 主 - - - - - - - - - - - - - - 年 - - 口 - - - - - - - - - - -
[PASS] 预检通过：tokenizer 对齐、特殊 token、截断与标签口径均正常。
--preflight-only 已指定：不加载模型、不训练，退出。


In [63]:

%cd /content/xime_work/xime_macbert_colab_ready

import csv
from pathlib import Path
from transformers import AutoTokenizer

base = "models/macbert-correction-v1"
path = Path("data/processed/csc_split_v1/train.tsv")
max_length = 160

tok = AutoTokenizer.from_pretrained(
    base,
    use_fast=True,
    local_files_only=True,
)

if not tok.is_fast:
    raise RuntimeError("当前 tokenizer 不支持 offset_mapping，停止检查。")

with path.open("r", encoding="utf-8", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

found = 0
for row_idx, row in enumerate(rows, start=1):
    noisy = row["noisy"]
    clean = row["clean"]

    noisy_enc = tok(
        noisy,
        add_special_tokens=True,
        truncation=False,
        return_offsets_mapping=True,
    )
    clean_enc = tok(
        clean,
        add_special_tokens=True,
        truncation=False,
        return_offsets_mapping=True,
    )

    n_len = len(noisy_enc["input_ids"])
    c_len = len(clean_enc["input_ids"])

    if n_len < max_length and c_len < max_length:
        continue

    found += 1
    print("\n" + "=" * 70)
    print("TSV 数据行（不含表头）：", row_idx)
    print("noisy token 长度：", n_len)
    print("clean token 长度：", c_len)
    print("错误句：", noisy)
    print("目标句：", clean)

    char_diffs = [
        i for i, (a, b) in enumerate(zip(noisy, clean)) if a != b
    ]
    print("字符差异位置（从 0 开始）：", char_diffs)

    for pos in char_diffs:
        print(f"\n字符位置 {pos}: {noisy[pos]} -> {clean[pos]}")

        for name, enc in (("noisy", noisy_enc), ("clean", clean_enc)):
            matches = [
                i for i, (start, end) in enumerate(enc["offset_mapping"])
                if start <= pos < end and end > start
            ]
            if not matches:
                print(f"  {name}: 未找到对应 token")
                continue

            ti = matches[0]
            retained = ti < max_length - 1
            token = tok.convert_ids_to_tokens(enc["input_ids"][ti])
            print(
                f"  {name}: token_index={ti}, token={token!r}, "
                f"offset={enc['offset_mapping'][ti]}, "
                f"{'保留范围内' if retained else '截断后丢失'}"
            )

if found == 0:
    print("没有找到达到或超过 max_length 的训练样本。")
else:
    print(f"\n检查完成：找到 {found} 条达到或超过长度上限的样本。")

/content/xime_work/xime_macbert_colab_ready

TSV 数据行（不含表头）： 129196
noisy token 长度： 160
clean token 长度： 160
错误句： 我在高中的时候在中巴文化中心中山学校念书，２０１０年来台湾交换。在那段时间我决定了我要来台湾念书，台湾的城府给中巴学校三个奖学金，为了得到台湾的奖学金，我好好儿的念书。虽然我从小时候很用功，可是我开始有得到奖学金的目的时候我更用功。那一天是我最快乐的一天，不但是因为是我得到了奖学金的一天，而且是我成功地做到目的的一天。
目标句： 我在高中的时候在中巴文化中心中山学校念书，２０１０年来台湾交换。在那段时间我决定了我要来台湾念书，台湾的政府给中巴学校三个奖学金，为了得到台湾的奖学金，我好好儿的念书。虽然我从小时候很用功，可是我开始有得到奖学金的目的时候我更用功。那一天是我最快乐的一天，不但是因为是我得到了奖学金的一天，而且是我成功地做到目的的一天。
字符差异位置（从 0 开始）： [52]

字符位置 52: 城 -> 政
  noisy: token_index=51, token='城', offset=(52, 53), 保留范围内
  clean: token_index=51, token='政', offset=(52, 53), 保留范围内

TSV 数据行（不含表头）： 184535
noisy token 长度： 180
clean token 长度： 180
错误句： 四月我生日的那天我还睡觉的时候有人敲我的门，我以为我作梦，累地我把门大开，我看到我的家人拿著蛋糕也叫我去花园看一下生日礼物，我慢慢也不说话地去楼下，我吓一跳看到一个在我们花园站著的超大的大象，妈妈爸爸来了然后帮我爬大象上去，我整天抱著我新朋友也跟他说我的困难，从那天起我说狠多话，去动物园也过快乐的生活。这是为甚么那天是我最快乐的一天因为那天让我再爱动物。
目标句： 四月我生日的那天我还睡觉的时候有人敲我的门，我以为我作梦，累地我把门打开，我看到我的家人拿著蛋糕也叫我去花园看一下生日礼物，我慢慢也不说话地去楼下，我吓一跳看到一个在我们花园站著的超大的大象，妈妈爸爸来了然后帮我爬大象上去，我整天抱著我新朋友也跟他说我的困难，从那天起我说很多话，去动物园也过快乐的生活。这

In [64]:

!nvidia-smi

Sat Oct 17 18:07:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [65]:

import torch

print("PyTorch 版本：", torch.__version__)
print("CUDA 可用：", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 型号：", torch.cuda.get_device_name(0))
    print("显存总量：", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GiB")
    print("当前已分配显存：", round(
        torch.cuda.memory_allocated(0) / 1024**3, 2
    ), "GiB")
else:
    print("当前没有可用 CUDA GPU，先不要启动正式训练。")

PyTorch 版本： 2.11.0+cu130
CUDA 可用： True
GPU 型号： Tesla T4
显存总量： 14.56 GiB
当前已分配显存： 0.0 GiB


In [66]:
%cd /content/xime_work/xime_macbert_colab_ready

!python scripts/train_correction.py --help

/content/xime_work/xime_macbert_colab_ready
usage: train_correction.py [-h] [--base BASE] --train-file TRAIN_FILE
                           --valid-file VALID_FILE [--output-dir OUTPUT_DIR]
                           [--epochs EPOCHS] [--batch-size BATCH_SIZE]
                           [--grad-accum GRAD_ACCUM] [--lr LR]
                           [--warmup-ratio WARMUP_RATIO]
                           [--max-length MAX_LENGTH]
                           [--max-train-samples MAX_TRAIN_SAMPLES]
                           [--mask-typo-prob MASK_TYPO_PROB]
                           [--mask-correct-prob MASK_CORRECT_PROB]
                           [--seed SEED] [--fp16] [--preflight-only]

中文纠错训练（clean 作为标签，固定 MacBERT MLM 头）

options:
  -h, --help            show this help message and exit
  --base BASE
  --train-file TRAIN_FILE
  --valid-file VALID_FILE
  --output-dir OUTPUT_DIR
  --epochs EPOCHS
  --batch-size BATCH_SIZE
  --grad-accum GRAD_ACCUM
  --lr LR
  --warmup-ratio WARMUP_RA

In [67]:
%cd /content/xime_work/xime_macbert_colab_ready

from pathlib import Path
import csv
import random

project = Path("/content/xime_work/xime_macbert_colab_ready")
source_train = project / "data/processed/csc_split_v1/train.tsv"
source_valid = project / "data/processed/csc_split_v1/valid.tsv"

smoke_dir = project / "data/processed/csc_smoke_v1"

if not source_train.is_file() or not source_valid.is_file():
    raise FileNotFoundError("正式训练集或验证集不存在，请先检查路径。")

if smoke_dir.exists():
    raise FileExistsError(
        f"测试目录已存在，为防止覆盖，停止执行：{smoke_dir}"
    )

smoke_dir.mkdir(parents=True)

def read_tsv(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        if not {"clean", "noisy"}.issubset(reader.fieldnames or []):
            raise ValueError(f"TSV 字段不正确：{path}")
        rows = [
            {"clean": row["clean"], "noisy": row["noisy"]}
            for row in reader
        ]
    return rows

train_rows = read_tsv(source_train)
valid_rows = read_tsv(source_valid)

rng = random.Random(20261018)
smoke_train = rng.sample(train_rows, min(1000, len(train_rows)))
smoke_valid = rng.sample(valid_rows, min(200, len(valid_rows)))

def write_tsv(path, rows):
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["clean", "noisy"],
            delimiter="\t",
            lineterminator="\n",
        )
        writer.writeheader()
        writer.writerows(rows)

write_tsv(smoke_dir / "train.tsv", smoke_train)
write_tsv(smoke_dir / "valid.tsv", smoke_valid)

print("冒烟测试集创建完成")
print("训练样本：", len(smoke_train))
print("验证样本：", len(smoke_valid))
print("输出目录：", smoke_dir)
print("正式数据集未修改。")

/content/xime_work/xime_macbert_colab_ready
冒烟测试集创建完成
训练样本： 1000
验证样本： 200
输出目录： /content/xime_work/xime_macbert_colab_ready/data/processed/csc_smoke_v1
正式数据集未修改。


In [68]:
%cd /content/xime_work/xime_macbert_colab_ready

!python scripts/train_correction.py \
  --base models/macbert-correction-v1 \
  --train-file data/processed/csc_smoke_v1/train.tsv \
  --valid-file data/processed/csc_smoke_v1/valid.tsv \
  --output-dir models/macbert-csc-smoke-v1 \
  --epochs 1 \
  --batch-size 8 \
  --grad-accum 2 \
  --lr 5e-5 \
  --warmup-ratio 0.05 \
  --max-length 160 \
  --seed 20261018 \
  --fp16

/content/xime_work/xime_macbert_colab_ready
中文纠错训练（对齐标签监督）：input=noisy，label=clean
这不是普通 MLM：损失的标签来自 clean 列，而非输入自身。
训练文件 train.tsv: 1000 条；验证文件 valid.tsv: 200 条
特殊 token：{'[UNK]': 100, '[SEP]': 102, '[PAD]': 0, '[CLS]': 101, '[MASK]': 103}
[MASK] id = 103；词表大小 = 21128
[train] 读取 1000，可用 1000，长度不一致跳过 0，无错字 7，截断 0
[train] 错字位置 1389 个（平均 1.39/句），其中被 mask 1101；抽查正确位置 4973；最长 token 138
[valid] 读取 200，可用 200，长度不一致跳过 0，无错字 4，截断 0
[valid] 错字位置 261 个（平均 1.30/句），其中被 mask 223；抽查正确位置 972；最长 token 105
[检查] 标签中特殊 token 泄漏：0（应恒为 0）
[样例] 输入: [CLS] 简 直 忙 过 与 [MASK] 。 [SEP]
       标签: - 简 - - - 育 儿 - -
[PASS] 预检通过：tokenizer 对齐、特殊 token、截断与标签口径均正常。
Loading weights: 100% 202/202 [00:00<00:00, 6439.92it/s]
[训练参数] 预计总更新步数=63，预热步数=3
{'loss': '3.907', 'grad_norm': '29.73', 'learning_rate': '1.167e-05', 'epoch': '0.8'}
 98% 62/63 [00:08<00:00,  9.43it/s]
  0% 0/25 [00:00<?, ?it/s]
 32% 8/25 [00:00<00:00, 75.83it/s]
 64% 16/25 [00:00<00:00, 68.15it/s]
                                   
{'eval_loss': '1.459', 

In [69]:
%cd /content/xime_work/xime_macbert_colab_ready

!python scripts/eval_dual.py --help

/content/xime_work/xime_macbert_colab_ready
usage: eval_dual.py [-h] --mode {shuangpin,correction} [--errors ERRORS]
                    [--model MODEL] [--test-file TEST_FILE]
                    [--max-samples MAX_SAMPLES] [--max-length MAX_LENGTH]
                    [--batch-size BATCH_SIZE]
                    [--mask-correct-ratio MASK_CORRECT_RATIO]
                    [--mask-correct-prob MASK_CORRECT_PROB] [--seed SEED]
                    [--out OUT]

双层评估：双拼键码恢复 + 中文纠错

options:
  -h, --help            show this help message and exit
  --mode {shuangpin,correction}
  --errors ERRORS       shuangpin 模式：errors.jsonl
  --model MODEL         correction 模式：模型目录
  --test-file TEST_FILE
                        correction 模式：test.tsv
  --max-samples MAX_SAMPLES
  --max-length MAX_LENGTH
  --batch-size BATCH_SIZE
  --mask-correct-ratio MASK_CORRECT_RATIO
                        抽查多少比例的'本就正确'位置一起 mask，用于测误纠正率
  --mask-correct-prob MASK_CORRECT_PROB
  --seed SEED
  --out OUT


In [70]:
%cd /content/xime_work/xime_macbert_colab_ready

from pathlib import Path

project = Path("/content/xime_work/xime_macbert_colab_ready")

checks = {
    "测试集": project / "data/processed/test_v1.tsv",
    "原始模型": project / "models/macbert-correction-v1",
    "冒烟模型": project / "models/macbert-csc-smoke-v1",
}

for name, path in checks.items():
    print(f"{name}: {'存在' if path.exists() else '不存在'} — {path}")
    if not path.exists():
        raise FileNotFoundError(f"{name}不存在，停止评估。")

report_dir = project / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

for name in [
    "eval_base_test_v1.json",
    "eval_csc_smoke_test_v1.json",
]:
    path = report_dir / name
    if path.exists():
        raise FileExistsError(f"报告已存在，为避免覆盖，请更换文件名：{path}")

print("\n检查通过，可以开始评估。")

/content/xime_work/xime_macbert_colab_ready
测试集: 存在 — /content/xime_work/xime_macbert_colab_ready/data/processed/test_v1.tsv
原始模型: 存在 — /content/xime_work/xime_macbert_colab_ready/models/macbert-correction-v1
冒烟模型: 存在 — /content/xime_work/xime_macbert_colab_ready/models/macbert-csc-smoke-v1

检查通过，可以开始评估。


In [71]:
%cd /content/xime_work/xime_macbert_colab_ready

!python scripts/eval_dual.py \
  --mode correction \
  --model models/macbert-correction-v1 \
  --test-file data/processed/test_v1.tsv \
  --max-length 160 \
  --batch-size 8 \
  --seed 20261018 \
  --out reports/eval_base_test_v1.json

/content/xime_work/xime_macbert_colab_ready
Loading weights: 100% 202/202 [00:00<00:00, 14434.29it/s]
设备：cuda；测试句：650；模型：/content/xime_work/xime_macbert_colab_ready/models/macbert-correction-v1
{
  "mode": "correction",
  "model": "/content/xime_work/xime_macbert_colab_ready/models/macbert-correction-v1",
  "test_file": "/content/xime_work/xime_macbert_colab_ready/data/processed/test_v1.tsv",
  "device": "cuda",
  "samples_skipped": 0,
  "typo_positions": 650,
  "correction_accuracy_top1": 0.4169,
  "correction_accuracy_top3": 0.5554,
  "kept_positions_sampled": 6,
  "preservation_rate": 0.6667,
  "false_correction_rate": 0.3333,
  "position_detection": {
    "precision": 0.9969,
    "recall": 1.0,
    "f1": 0.9985,
    "predicted_changes": 652,
    "true_changes": 650,
    "note": "把'模型改动了的位置'当作检测结果；基于本次 mask 位置集合，属代理指标。"
  },
  "mask_correct_ratio": 0.2,
  "metric_note": "错字位置在评估时被显式 mask（oracle 定位），不是端到端准确率。"
}
已写出：/content/xime_work/xime_macbert_colab_ready/reports/eval_base_test_v

In [72]:
%cd /content/xime_work/xime_macbert_colab_ready

!python scripts/eval_dual.py \
  --mode correction \
  --model models/macbert-csc-smoke-v1 \
  --test-file data/processed/test_v1.tsv \
  --max-length 160 \
  --batch-size 8 \
  --seed 20261018 \
  --out reports/eval_csc_smoke_test_v1.json

/content/xime_work/xime_macbert_colab_ready
Loading weights: 100% 202/202 [00:00<00:00, 14245.95it/s]
设备：cuda；测试句：650；模型：/content/xime_work/xime_macbert_colab_ready/models/macbert-csc-smoke-v1
{
  "mode": "correction",
  "model": "/content/xime_work/xime_macbert_colab_ready/models/macbert-csc-smoke-v1",
  "test_file": "/content/xime_work/xime_macbert_colab_ready/data/processed/test_v1.tsv",
  "device": "cuda",
  "samples_skipped": 0,
  "typo_positions": 650,
  "correction_accuracy_top1": 0.4231,
  "correction_accuracy_top3": 0.6431,
  "kept_positions_sampled": 6,
  "preservation_rate": 0.6667,
  "false_correction_rate": 0.3333,
  "position_detection": {
    "precision": 0.9969,
    "recall": 1.0,
    "f1": 0.9985,
    "predicted_changes": 652,
    "true_changes": 650,
    "note": "把'模型改动了的位置'当作检测结果；基于本次 mask 位置集合，属代理指标。"
  },
  "mask_correct_ratio": 0.2,
  "metric_note": "错字位置在评估时被显式 mask（oracle 定位），不是端到端准确率。"
}
已写出：/content/xime_work/xime_macbert_colab_ready/reports/eval_csc_smoke_tes

In [73]:
%cd /content/xime_work/xime_macbert_colab_ready

from pathlib import Path

root = Path("/content/xime_work/xime_macbert_colab_ready")
valid = root / "data/processed/csc_split_v1/valid.tsv"

reports = [
    root / "reports/eval_base_csc_valid_v1.json",
    root / "reports/eval_smoke_csc_valid_v1.json",
]

assert valid.is_file(), f"找不到 CSC 验证集：{valid}"

for path in reports:
    if path.exists():
        raise FileExistsError(f"报告已存在，请换一个新文件名：{path}")

print("CSC 验证集：", valid)
print("验证样本数：12,302")
print("两个报告路径均可使用。")

/content/xime_work/xime_macbert_colab_ready
CSC 验证集： /content/xime_work/xime_macbert_colab_ready/data/processed/csc_split_v1/valid.tsv
验证样本数：12,302
两个报告路径均可使用。


In [74]:
%cd /content/xime_work/xime_macbert_colab_ready

!python scripts/eval_dual.py \
  --mode correction \
  --model models/macbert-correction-v1 \
  --test-file data/processed/csc_split_v1/valid.tsv \
  --max-length 160 \
  --batch-size 8 \
  --seed 20261018 \
  --out reports/eval_base_csc_valid_v1.json

/content/xime_work/xime_macbert_colab_ready
Loading weights: 100% 202/202 [00:00<00:00, 12718.22it/s]
设备：cuda；测试句：2000；模型：/content/xime_work/xime_macbert_colab_ready/models/macbert-correction-v1
{
  "mode": "correction",
  "model": "/content/xime_work/xime_macbert_colab_ready/models/macbert-correction-v1",
  "test_file": "/content/xime_work/xime_macbert_colab_ready/data/processed/csc_split_v1/valid.tsv",
  "device": "cuda",
  "samples_skipped": 0,
  "typo_positions": 2000,
  "correction_accuracy_top1": 0.3195,
  "correction_accuracy_top3": 0.4935,
  "kept_positions_sampled": 16112,
  "preservation_rate": 0.4297,
  "false_correction_rate": 0.5703,
  "position_detection": {
    "precision": 0.1778,
    "recall": 0.9935,
    "f1": 0.3016,
    "predicted_changes": 11176,
    "true_changes": 2000,
    "note": "把'模型改动了的位置'当作检测结果；基于本次 mask 位置集合，属代理指标。"
  },
  "mask_correct_ratio": 0.2,
  "metric_note": "错字位置在评估时被显式 mask（oracle 定位），不是端到端准确率。"
}
已写出：/content/xime_work/xime_macbert_colab_ready/r

In [75]:
%cd /content/xime_work/xime_macbert_colab_ready

!python scripts/eval_dual.py \
  --mode correction \
  --model models/macbert-csc-smoke-v1 \
  --test-file data/processed/csc_split_v1/valid.tsv \
  --max-length 160 \
  --batch-size 8 \
  --seed 20261018 \
  --out reports/eval_smoke_csc_valid_v1.json

/content/xime_work/xime_macbert_colab_ready
Loading weights: 100% 202/202 [00:00<00:00, 12484.34it/s]
设备：cuda；测试句：2000；模型：/content/xime_work/xime_macbert_colab_ready/models/macbert-csc-smoke-v1
{
  "mode": "correction",
  "model": "/content/xime_work/xime_macbert_colab_ready/models/macbert-csc-smoke-v1",
  "test_file": "/content/xime_work/xime_macbert_colab_ready/data/processed/csc_split_v1/valid.tsv",
  "device": "cuda",
  "samples_skipped": 0,
  "typo_positions": 2000,
  "correction_accuracy_top1": 0.5325,
  "correction_accuracy_top3": 0.6925,
  "kept_positions_sampled": 16112,
  "preservation_rate": 0.5929,
  "false_correction_rate": 0.4071,
  "position_detection": {
    "precision": 0.2323,
    "recall": 0.9925,
    "f1": 0.3765,
    "predicted_changes": 8544,
    "true_changes": 2000,
    "note": "把'模型改动了的位置'当作检测结果；基于本次 mask 位置集合，属代理指标。"
  },
  "mask_correct_ratio": 0.2,
  "metric_note": "错字位置在评估时被显式 mask（oracle 定位），不是端到端准确率。"
}
已写出：/content/xime_work/xime_macbert_colab_ready/repo

In [76]:
%cd /content/xime_work/xime_macbert_colab_ready

from pathlib import Path

p = Path("scripts/eval_dual.py")
lines = p.read_text(encoding="utf-8").splitlines()

terms = (
    "max-samples",
    "max_samples",
    "kept_positions_sampled",
    "mask_correct_ratio",
    "correction_accuracy_top1",
)

for i, line in enumerate(lines):
    if any(term in line for term in terms):
        start = max(0, i - 3)
        end = min(len(lines), i + 5)
        print(f"\n===== 第 {start+1}–{end} 行 =====")
        for j in range(start, end):
            print(f"{j+1:4}: {lines[j]}")

/content/xime_work/xime_macbert_colab_ready

===== 第 130–137 行 =====
 130:     return rows
 131: 
 132: 
 133: def eval_correction(model_dir: Path, test_file: Path, max_samples: int, max_length: int,
 134:                     batch_size: int, mask_correct_ratio: float, mask_correct_prob: float,
 135:                     seed: int, out_path: Path | None) -> int:
 136:     try:
 137:         import random

===== 第 131–138 行 =====
 131: 
 132: 
 133: def eval_correction(model_dir: Path, test_file: Path, max_samples: int, max_length: int,
 134:                     batch_size: int, mask_correct_ratio: float, mask_correct_prob: float,
 135:                     seed: int, out_path: Path | None) -> int:
 136:     try:
 137:         import random
 138: 

===== 第 159–166 行 =====
 159:         return 2
 160: 
 161:     rng = random.Random(seed)
 162:     pairs = load_pairs(test_file, max_samples)
 163:     print(f"设备：{device}；测试句：{len(pairs)}；模型：{model_dir.resolve()}")
 164: 
 165:     typo_total

In [77]:
%cd /content/xime_work/xime_macbert_colab_ready

from pathlib import Path

lines = Path("scripts/eval_dual.py").read_text(
    encoding="utf-8"
).splitlines()

for i in range(75, 132):
    print(f"{i+1:4}: {lines[i]}")

/content/xime_work/xime_macbert_colab_ready
  76:     result = {
  77:         "mode": "shuangpin",
  78:         "file": str(errors_path.resolve()),
  79:         "records": n,
  80:         "distinct_wrong_codes": len({r["wrong_code"] for r in rows}),
  81:         "ime_decodable": decodable,
  82:         "ime_decodable_rate": round(decodable / n, 4),
  83:         "ime_ambiguous": ime_ambiguous,
  84:         "ime_ambiguous_rate": round(ime_ambiguous / n, 4),
  85:         "recovery_unique": unique_recovery,
  86:         "recovery_unique_rate": round(unique_recovery / n, 4),
  87:         "recovery_candidates_mean": round(sum(rec_sizes) / n, 3),
  88:         "recovery_candidates_max": max(rec_sizes),
  89:         "baseline_trust_typed_code": {
  90:             "unique_typed": typed_unique,
  91:             "correct": typed_correct,
  92:             "accuracy": round(typed_correct / n, 4),
  93:             "note": "直接相信键码所对应的唯一音节；因为错误键码通常根本不是目标音节，准确率接近 0。",
  94:         },
 

In [78]:
%cd /content/xime_work/xime_macbert_colab_ready

from pathlib import Path

root = Path("/content/xime_work/xime_macbert_colab_ready")
valid = root / "data/processed/csc_split_v1/valid.tsv"

outputs = [
    root / "reports/eval_base_csc_valid_full_v1.json",
    root / "reports/eval_smoke_csc_valid_full_v1.json",
]

assert valid.is_file(), f"验证集不存在：{valid}"

with valid.open("r", encoding="utf-8-sig") as f:
    row_count = sum(1 for _ in f) - 1

assert row_count == 12302, f"样本数与预期不符：{row_count}"

for path in outputs:
    assert not path.exists(), f"输出文件已存在，请换新文件名：{path}"

print("验证集样本数：", row_count)
print("两个报告路径均未占用，可以开始完整评估。")

/content/xime_work/xime_macbert_colab_ready
验证集样本数： 12302
两个报告路径均未占用，可以开始完整评估。


In [79]:
%cd /content/xime_work/xime_macbert_colab_ready

!python scripts/eval_dual.py \
  --mode correction \
  --model models/macbert-correction-v1 \
  --test-file data/processed/csc_split_v1/valid.tsv \
  --max-samples 0 \
  --max-length 160 \
  --batch-size 8 \
  --seed 20261018 \
  --out reports/eval_base_csc_valid_full_v1.json

/content/xime_work/xime_macbert_colab_ready
Loading weights: 100% 202/202 [00:00<00:00, 12655.52it/s]
设备：cuda；测试句：12302；模型：/content/xime_work/xime_macbert_colab_ready/models/macbert-correction-v1
{
  "mode": "correction",
  "model": "/content/xime_work/xime_macbert_colab_ready/models/macbert-correction-v1",
  "test_file": "/content/xime_work/xime_macbert_colab_ready/data/processed/csc_split_v1/valid.tsv",
  "device": "cuda",
  "samples_skipped": 0,
  "typo_positions": 16936,
  "correction_accuracy_top1": 0.2807,
  "correction_accuracy_top3": 0.4233,
  "kept_positions_sampled": 101321,
  "preservation_rate": 0.4172,
  "false_correction_rate": 0.5828,
  "position_detection": {
    "precision": 0.2214,
    "recall": 0.9914,
    "f1": 0.362,
    "predicted_changes": 75837,
    "true_changes": 16936,
    "note": "把'模型改动了的位置'当作检测结果；基于本次 mask 位置集合，属代理指标。"
  },
  "mask_correct_ratio": 0.2,
  "metric_note": "错字位置在评估时被显式 mask（oracle 定位），不是端到端准确率。"
}
已写出：/content/xime_work/xime_macbert_colab_read

In [80]:
%cd /content/xime_work/xime_macbert_colab_ready

!python scripts/eval_dual.py \
  --mode correction \
  --model models/macbert-csc-smoke-v1 \
  --test-file data/processed/csc_split_v1/valid.tsv \
  --max-samples 0 \
  --max-length 160 \
  --batch-size 8 \
  --seed 20261018 \
  --out reports/eval_smoke_csc_valid_full_v1.json

/content/xime_work/xime_macbert_colab_ready
Loading weights: 100% 202/202 [00:00<00:00, 12801.04it/s]
设备：cuda；测试句：12302；模型：/content/xime_work/xime_macbert_colab_ready/models/macbert-csc-smoke-v1
{
  "mode": "correction",
  "model": "/content/xime_work/xime_macbert_colab_ready/models/macbert-csc-smoke-v1",
  "test_file": "/content/xime_work/xime_macbert_colab_ready/data/processed/csc_split_v1/valid.tsv",
  "device": "cuda",
  "samples_skipped": 0,
  "typo_positions": 16936,
  "correction_accuracy_top1": 0.4711,
  "correction_accuracy_top3": 0.6204,
  "kept_positions_sampled": 101321,
  "preservation_rate": 0.5817,
  "false_correction_rate": 0.4183,
  "position_detection": {
    "precision": 0.2839,
    "recall": 0.9923,
    "f1": 0.4415,
    "predicted_changes": 59183,
    "true_changes": 16936,
    "note": "把'模型改动了的位置'当作检测结果；基于本次 mask 位置集合，属代理指标。"
  },
  "mask_correct_ratio": 0.2,
  "metric_note": "错字位置在评估时被显式 mask（oracle 定位），不是端到端准确率。"
}
已写出：/content/xime_work/xime_macbert_colab_ready

In [81]:
%cd /content/xime_work/xime_macbert_colab_ready

!python scripts/train_correction.py \
  --base models/macbert-correction-v1 \
  --train-file data/processed/csc_split_v1/train.tsv \
  --valid-file data/processed/csc_split_v1/valid.tsv \
  --output-dir models/macbert-csc-full-v1-epoch1 \
  --epochs 1 \
  --batch-size 8 \
  --grad-accum 2 \
  --lr 5e-5 \
  --warmup-ratio 0.05 \
  --max-length 160 \
  --seed 20261018 \
  --fp16 \
  --preflight-only

/content/xime_work/xime_macbert_colab_ready
中文纠错训练（对齐标签监督）：input=noisy，label=clean
这不是普通 MLM：损失的标签来自 clean 列，而非输入自身。
训练文件 train.tsv: 231371 条；验证文件 valid.tsv: 12302 条
特殊 token：{'[UNK]': 100, '[SEP]': 102, '[PAD]': 0, '[CLS]': 101, '[MASK]': 103}
[MASK] id = 103；词表大小 = 21128
[train] 读取 231371，可用 231371，长度不一致跳过 0，无错字 3676，截断 2
[train] 错字位置 313228 个（平均 1.35/句），其中被 mask 250513；抽查正确位置 1142554；最长 token 160
[valid] 读取 12302，可用 12302，长度不一致跳过 0，无错字 178，截断 0
[valid] 错字位置 16557 个（平均 1.35/句），其中被 mask 13254；抽查正确位置 60609；最长 token 141
[WARN] 存在达到 160 上限的样本，长句会被截断（错字可能落在截断区）
[检查] 标签中特殊 token 泄漏：0（应恒为 0）
[样例] 输入: [CLS] 报 导 中 并 [MASK] 说 明 [MASK] 口 量 ， 但 据 引 述 药 厂 主 管 的 话 指 出 ， 每 一 种 药 物 [MASK] 大 的 庄 度 出 口 量 都 达 到 十 二 吨 之 谱 。 [SEP]
       标签: - - - - - 未 - - 出 - - - - 据 - - - - - - - - 指 - - - - - - - 最 - - 年 度 - 口 - - - - - - - - - - -
[PASS] 预检通过：tokenizer 对齐、特殊 token、截断与标签口径均正常。
--preflight-only 已指定：不加载模型、不训练，退出。


In [82]:
from pathlib import Path

out = Path(
    "/content/xime_work/xime_macbert_colab_ready/"
    "models/macbert-csc-full-v1-epoch1"
)

assert not out.exists(), f"输出目录已存在，避免覆盖：{out}"
print("输出目录未占用，可以开始正式训练。")

输出目录未占用，可以开始正式训练。


In [83]:
%cd /content/xime_work/xime_macbert_colab_ready

!python scripts/train_correction.py \
  --base models/macbert-correction-v1 \
  --train-file data/processed/csc_split_v1/train.tsv \
  --valid-file data/processed/csc_split_v1/valid.tsv \
  --output-dir models/macbert-csc-full-v1-epoch1 \
  --epochs 1 \
  --batch-size 8 \
  --grad-accum 2 \
  --lr 5e-5 \
  --warmup-ratio 0.05 \
  --max-length 160 \
  --seed 20261018 \
  --fp16

/content/xime_work/xime_macbert_colab_ready
中文纠错训练（对齐标签监督）：input=noisy，label=clean
这不是普通 MLM：损失的标签来自 clean 列，而非输入自身。
训练文件 train.tsv: 231371 条；验证文件 valid.tsv: 12302 条
特殊 token：{'[UNK]': 100, '[SEP]': 102, '[PAD]': 0, '[CLS]': 101, '[MASK]': 103}
[MASK] id = 103；词表大小 = 21128
[train] 读取 231371，可用 231371，长度不一致跳过 0，无错字 3676，截断 2
[train] 错字位置 313228 个（平均 1.35/句），其中被 mask 250513；抽查正确位置 1142554；最长 token 160
[valid] 读取 12302，可用 12302，长度不一致跳过 0，无错字 178，截断 0
[valid] 错字位置 16557 个（平均 1.35/句），其中被 mask 13254；抽查正确位置 60609；最长 token 141
[WARN] 存在达到 160 上限的样本，长句会被截断（错字可能落在截断区）
[检查] 标签中特殊 token 泄漏：0（应恒为 0）
[样例] 输入: [CLS] 报 导 中 并 [MASK] 说 明 [MASK] 口 量 ， 但 据 引 述 药 厂 主 管 的 话 指 出 ， 每 一 种 药 物 [MASK] 大 的 庄 度 出 口 量 都 达 到 十 二 吨 之 谱 。 [SEP]
       标签: - - - - - 未 - - 出 - - - - 据 - - - - - - - - 指 - - - - - - - 最 - - 年 度 - 口 - - - - - - - - - - -
[PASS] 预检通过：tokenizer 对齐、特殊 token、截断与标签口径均正常。
Loading weights: 100% 202/202 [00:00<00:00, 5675.12it/s]
[训练参数] 预计总更新步数=14461，预热步数=723
{'loss': '5.925', 'grad_norm': '30.97'